In [ ]:
# PHASE 0 - Cell 1: Install the core dependencies required by Vaultify

%pip install -q \
    qdrant-client \
    groq \
    docling \
    "rapidocr<3.8" \
    onnxruntime \
    "mcp[cli]" \
    flask \
    pyngrok \
    python-dotenv \
    sentence-transformers

print("✅ Vaultify core dependencies installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.4/458.4 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.8/261.8 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.5 MB/s eta 0:00:00


In [ ]:
# PHASE 0 - Cell 2: Load and validate API credentials from Colab Secrets

from google.colab import userdata

REQUIRED_SECRETS = [
    "QDRANT_URL",
    "QDRANT_API_KEY",
    "GROQ_API_KEY",
    "NGROK_TOKEN",
]

loaded_secrets = {}
missing_secrets = []

for secret_name in REQUIRED_SECRETS:
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None

    if secret_value:
        loaded_secrets[secret_name] = secret_value
        print(f"✅ {secret_name}: Found")
    else:
        missing_secrets.append(secret_name)
        print(f"❌ {secret_name}: Missing or notebook access is disabled")

if missing_secrets:
    raise RuntimeError(
        "Missing required Colab Secrets: " + ", ".join(missing_secrets)
    )

QDRANT_URL = loaded_secrets["QDRANT_URL"]
QDRANT_API_KEY = loaded_secrets["QDRANT_API_KEY"]
GROQ_API_KEY = loaded_secrets["GROQ_API_KEY"]
NGROK_TOKEN = loaded_secrets["NGROK_TOKEN"]

print("\n✅ All required Vaultify credentials were loaded securely.")

✅ QDRANT_URL: Found
✅ QDRANT_API_KEY: Found
✅ GROQ_API_KEY: Found
✅ NGROK_TOKEN: Found

✅ All required Vaultify credentials were loaded securely.


In [ ]:
# PHASE 0 - Cell 3: Initialize and test the Qdrant Cloud connection

from qdrant_client import QdrantClient

COLLECTION_NAME = "vaultify_v3_documents"

try:
    qdrant = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        timeout=30,
    )

    collection_response = qdrant.get_collections()
    existing_collections = [
        collection.name for collection in collection_response.collections
    ]

    print("✅ Qdrant Cloud connection successful.")
    print(f"📦 Existing collections: {existing_collections or 'None'}")

    if COLLECTION_NAME in existing_collections:
        print(f"ℹ️ '{COLLECTION_NAME}' already exists.")
        print("No data was modified.")
    else:
        print(f"✅ New collection name is available: '{COLLECTION_NAME}'")
        print("The collection will be created later during the ingestion phase.")

except Exception as error:
    raise RuntimeError(f"Qdrant connection failed: {error}") from error

✅ Qdrant Cloud connection successful.
📦 Existing collections: ['vaultify_documents']
✅ New collection name is available: 'vaultify_v3_documents'
The collection will be created later during the ingestion phase.


In [ ]:
# PHASE 0 - Cell 4: Initialize Groq and verify the preferred LLM model

from groq import Groq

LLM_MODEL = "llama-3.3-70b-versatile"

try:
    groq_client = Groq(api_key=GROQ_API_KEY)

    model_response = groq_client.models.list()
    available_model_ids = {
        model.id for model in model_response.data
    }

    if LLM_MODEL not in available_model_ids:
        raise RuntimeError(
            f"The preferred model '{LLM_MODEL}' is not available for this account."
        )

    test_response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": "Reply with exactly one word: READY",
            }
        ],
        temperature=0,
        max_tokens=10,
    )

    test_message = test_response.choices[0].message.content.strip()

    print("✅ Groq Cloud connection successful.")
    print(f"✅ Preferred model is available: {LLM_MODEL}")
    print(f"🤖 Test response: {test_message}")

except Exception as error:
    raise RuntimeError(f"Groq connection test failed: {error}") from error

✅ Groq Cloud connection successful.
✅ Preferred model is available: llama-3.3-70b-versatile
🤖 Test response: READY


In [ ]:
# PHASE 0 - Cell 5: Verify the Colab runtime and GPU availability

import platform
import sys
import torch

print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"🖥️ Platform: {platform.platform()}")
print(f"⚡ CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. In Colab, select Runtime > Change runtime type > T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"🎮 GPU: {gpu_name}")
print(f"🧠 GPU memory: {gpu_memory_gb:.1f} GB")
print("✅ The runtime is ready for document parsing and embeddings.")

🐍 Python version: 3.12.13
🖥️ Platform: Linux-6.6.122+-x86_64-with-glibc2.35
⚡ CUDA available: True
🎮 GPU: Tesla T4
🧠 GPU memory: 14.6 GB
✅ The runtime is ready for document parsing and embeddings.


In [ ]:
# PHASE 0 - Cell 6: Define the central Vaultify configuration

from pathlib import Path

PROJECT_NAME = "Vaultify"
PROJECT_VERSION = "3.0.0"

COLLECTION_NAME = "vaultify_v3_documents"

TENANT_ID_FIELD = "tenant_id"
DOCUMENT_HASH_FIELD = "document_hash"
FILENAME_FIELD = "filename"
CHUNK_INDEX_FIELD = "chunk_index"
CHUNK_TYPE_FIELD = "chunk_type"
TEXT_FIELD = "text"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_SIZE = 384

TEXT_CHUNK_SIZE = 800
TEXT_CHUNK_OVERLAP = 120

RETRIEVAL_CANDIDATE_LIMIT = 12
FINAL_CONTEXT_LIMIT = 6
MAX_ANSWER_TOKENS = 400

PROJECT_ROOT = Path("/content/vaultify_v3")
UPLOADS_DIR = PROJECT_ROOT / "uploads"
CACHE_DIR = PROJECT_ROOT / "cache"

for directory in [PROJECT_ROOT, UPLOADS_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"✅ Project: {PROJECT_NAME} v{PROJECT_VERSION}")
print(f"✅ Collection: {COLLECTION_NAME}")
print(f"✅ Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"✅ Vector size: {VECTOR_SIZE}")
print(f"✅ Text chunking: {TEXT_CHUNK_SIZE} characters with {TEXT_CHUNK_OVERLAP} overlap")
print(f"✅ Runtime workspace: {PROJECT_ROOT}")

✅ Project: Vaultify v3.0.0
✅ Collection: vaultify_v3_documents
✅ Embedding model: sentence-transformers/all-MiniLM-L6-v2
✅ Vector size: 384
✅ Text chunking: 800 characters with 120 overlap
✅ Runtime workspace: /content/vaultify_v3


In [ ]:
# PHASE 0 - Cell 7: Load and validate the embedding model on the GPU

from sentence_transformers import SentenceTransformer

try:
    embedding_model = SentenceTransformer(
        EMBEDDING_MODEL_NAME,
        device="cuda",
    )

    validation_texts = [
        "Vaultify helps companies search their private documents.",
        "Financial reports can contain tables and numerical data.",
    ]

    validation_embeddings = embedding_model.encode(
        validation_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    actual_vector_size = validation_embeddings.shape[1]

    if actual_vector_size != VECTOR_SIZE:
        raise RuntimeError(
            f"Expected {VECTOR_SIZE} dimensions, but received {actual_vector_size}."
        )

    print("✅ Embedding model loaded successfully.")
    print(f"🎮 Model device: {embedding_model.device}")
    print(f"📐 Embedding shape: {validation_embeddings.shape}")
    print(f"✅ Vector size validated: {actual_vector_size}")

except Exception as error:
    raise RuntimeError(f"Embedding model validation failed: {error}") from error

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded successfully.
🎮 Model device: cuda:0
📐 Embedding shape: (2, 384)
✅ Vector size validated: 384


In [ ]:
# PHASE 0 - Cell 8: Create and validate the Vaultify v3 Qdrant collection

from qdrant_client.models import Distance, PayloadSchemaType, VectorParams

try:
    existing_collections = {
        collection.name for collection in qdrant.get_collections().collections
    }

    if COLLECTION_NAME not in existing_collections:
        qdrant.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(
                size=VECTOR_SIZE,
                distance=Distance.COSINE,
            ),
        )
        print(f"✅ Created collection: {COLLECTION_NAME}")
    else:
        print(f"ℹ️ Collection already exists: {COLLECTION_NAME}")

    collection_info = qdrant.get_collection(COLLECTION_NAME)
    vector_config = collection_info.config.params.vectors

    if vector_config.size != VECTOR_SIZE:
        raise RuntimeError(
            f"Collection vector size is {vector_config.size}, expected {VECTOR_SIZE}."
        )

    if vector_config.distance != Distance.COSINE:
        raise RuntimeError(
            f"Collection distance is {vector_config.distance}, expected COSINE."
        )

    existing_indexes = collection_info.payload_schema or {}

    for field_name in [TENANT_ID_FIELD, DOCUMENT_HASH_FIELD]:
        if field_name not in existing_indexes:
            qdrant.create_payload_index(
                collection_name=COLLECTION_NAME,
                field_name=field_name,
                field_schema=PayloadSchemaType.KEYWORD,
                wait=True,
            )
            print(f"✅ Created keyword index for '{field_name}'.")
        else:
            print(f"ℹ️ Keyword index already exists for '{field_name}'.")

    point_count = qdrant.count(
        collection_name=COLLECTION_NAME,
        exact=True,
    ).count

    print(f"📊 Current point count: {point_count}")
    print("✅ Qdrant collection is ready for safe multi-tenant ingestion.")

except Exception as error:
    raise RuntimeError(f"Qdrant collection setup failed: {error}") from error

✅ Created collection: vaultify_v3_documents
✅ Created keyword index for 'tenant_id'.
✅ Created keyword index for 'document_hash'.
📊 Current point count: 0
✅ Qdrant collection is ready for safe multi-tenant ingestion.


In [ ]:
# PHASE 1 - Cell 1: Define document identity and duplicate prevention helpers

import hashlib
import uuid

from qdrant_client.models import FieldCondition, Filter, MatchValue


def calculate_document_hash(file_path, block_size=1024 * 1024):
    """
    Calculate a SHA-256 fingerprint from the binary content of a document.
    """
    path = Path(file_path)

    if not path.is_file():
        raise FileNotFoundError(f"Document file was not found: {path}")

    hasher = hashlib.sha256()

    with path.open("rb") as document_file:
        while block := document_file.read(block_size):
            hasher.update(block)

    return hasher.hexdigest()


def create_chunk_id(tenant_id, document_hash, chunk_index):
    """
    Create a stable and tenant-safe Qdrant point ID for one document chunk.
    """
    raw_identity = (
        f"{PROJECT_NAME}:{PROJECT_VERSION}:"
        f"{tenant_id}:{document_hash}:{chunk_index}"
    )

    return str(uuid.uuid5(uuid.NAMESPACE_URL, raw_identity))


def get_indexed_chunk_count(tenant_id, document_hash):
    """
    Count the chunks already indexed for one exact document and one tenant.
    """
    document_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(value=tenant_id),
            ),
            FieldCondition(
                key=DOCUMENT_HASH_FIELD,
                match=MatchValue(value=document_hash),
            ),
        ]
    )

    return qdrant.count(
        collection_name=COLLECTION_NAME,
        count_filter=document_filter,
        exact=True,
    ).count


print("✅ Document identity helpers are ready.")
print("✅ Chunk IDs will be stable, duplicate-safe, and tenant-safe.")

✅ Document identity helpers are ready.
✅ Chunk IDs will be stable, duplicate-safe, and tenant-safe.


In [ ]:
# PHASE 1 - Cell 2: Register the demo tenants and their seed documents

SEED_DOCUMENTS_DIR = UPLOADS_DIR / "seed_documents"
SEED_DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)

DEMO_TENANTS = {
    "company_a": {
        "tenant_id": "demo_apple_tenant",
        "display_name": "Apple Financial Demo",
    },
    "company_b": {
        "tenant_id": "demo_tesla_tenant",
        "display_name": "Tesla Financial Demo",
    },
}

SEED_DOCUMENTS = [
    {
        "tenant_key": "company_a",
        "filename": "apple_fy2025_10k.pdf",
        "source_url": (
            "https://s2.q4cdn.com/470004039/files/"
            "doc_financials/2025/ar/_10-K-2025-As-Filed.pdf"
        ),
    },
    {
        "tenant_key": "company_b",
        "filename": "tesla_q4_2025_update.pdf",
        "source_url": (
            "https://assets-ir.tesla.com/tesla-contents/"
            "IR/TSLA-Q4-2025-Update.pdf"
        ),
    },
]

print("✅ Demo tenants registered:")
for tenant_key, tenant in DEMO_TENANTS.items():
    print(f"   - {tenant_key}: {tenant['tenant_id']}")

print("\n✅ Seed documents registered:")
for document in SEED_DOCUMENTS:
    print(f"   - {document['filename']} → {document['tenant_key']}")

✅ Demo tenants registered:
   - company_a: demo_apple_tenant
   - company_b: demo_tesla_tenant

✅ Seed documents registered:
   - apple_fy2025_10k.pdf → company_a
   - tesla_q4_2025_update.pdf → company_b


In [ ]:
# PHASE 1 - Cell 3: Download the seed documents safely and idempotently

import shutil
from urllib.request import Request, urlopen


def download_seed_document(document):
    """
    Download one seed document only when a valid local copy is not available.
    """
    target_path = SEED_DOCUMENTS_DIR / document["filename"]

    if target_path.is_file():
        with target_path.open("rb") as existing_file:
            is_pdf = existing_file.read(5) == b"%PDF-"

        if is_pdf:
            file_size_mb = target_path.stat().st_size / (1024 * 1024)
            print(f"ℹ️ Already available: {target_path.name} ({file_size_mb:.2f} MB)")
            return target_path

        target_path.unlink()
        print(f"⚠️ Removed an invalid local file: {target_path.name}")

    temporary_path = target_path.with_suffix(target_path.suffix + ".part")
    request = Request(
        document["source_url"],
        headers={"User-Agent": "Mozilla/5.0 (Vaultify v3 seed downloader)"},
    )

    print(f"📥 Downloading: {target_path.name}")

    try:
        with urlopen(request, timeout=60) as response:
            with temporary_path.open("wb") as temporary_file:
                shutil.copyfileobj(response, temporary_file)

        with temporary_path.open("rb") as downloaded_file:
            is_pdf = downloaded_file.read(5) == b"%PDF-"

        if not is_pdf:
            raise RuntimeError("The downloaded file is not a valid PDF.")

        temporary_path.replace(target_path)

    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    file_size_mb = target_path.stat().st_size / (1024 * 1024)
    print(f"✅ Downloaded: {target_path.name} ({file_size_mb:.2f} MB)")

    return target_path


seed_document_paths = {}

for seed_document in SEED_DOCUMENTS:
    seed_document_paths[seed_document["filename"]] = download_seed_document(
        seed_document
    )

print("\n✅ All seed documents are ready for ingestion.")

📥 Downloading: apple_fy2025_10k.pdf
✅ Downloaded: apple_fy2025_10k.pdf (1.77 MB)
📥 Downloading: tesla_q4_2025_update.pdf
✅ Downloaded: tesla_q4_2025_update.pdf (5.39 MB)

✅ All seed documents are ready for ingestion.


In [ ]:
# PHASE 1 - Cell 4: Parse the seed PDFs with Docling

import time
import torch

from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    PdfPipelineOptions,
)

pipeline_options = PdfPipelineOptions()

if torch.cuda.is_available():
    pipeline_options.accelerator_options = AcceleratorOptions(
        num_threads=4,
        device=AcceleratorDevice.CUDA,
    )

document_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)


def parse_pdf_to_markdown(pdf_path):
    """
    Convert one PDF document into Markdown while preserving document structure.
    """
    pdf_path = Path(pdf_path)

    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF file was not found: {pdf_path}")

    conversion_result = document_converter.convert(str(pdf_path))
    return conversion_result.document.export_to_markdown()


parsed_seed_documents = {}

for seed_document in SEED_DOCUMENTS:
    filename = seed_document["filename"]
    pdf_path = seed_document_paths[filename]

    print(f"\n🚀 Parsing: {filename}")
    start_time = time.time()

    markdown_content = parse_pdf_to_markdown(pdf_path)
    parsed_seed_documents[filename] = markdown_content

    elapsed_seconds = time.time() - start_time

    print(f"✅ Parsed in {elapsed_seconds:.1f} seconds")
    print(f"📝 Extracted characters: {len(markdown_content):,}")

print("\n✅ All seed PDFs were parsed successfully.")


🚀 Parsing: apple_fy2025_10k.pdf


[INFO] 2026-07-10 14:16:52,880 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-10 14:16:52,952 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-07-10 14:16:52,957 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-07-10 14:16:53,390 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-10 14:16:53,398 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-07-10 14:16:53,401 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-07-10 14:16:53,514 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-10 14:16:53,587 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ Parsed in 91.5 seconds
📝 Extracted characters: 342,871

🚀 Parsing: tesla_q4_2025_update.pdf


[WARNING] 2026-07-10 14:18:49,405 [RapidOCR] main.py:125: The text detection result is empty


✅ Parsed in 97.5 seconds
📝 Extracted characters: 71,743

✅ All seed PDFs were parsed successfully.


In [ ]:
# PHASE 1 - Cell 5: Chunk Markdown with table preservation and strict token limits

import re

MAX_CHUNK_TOKENS = embedding_model.max_seq_length - 16
SPECIAL_TOKEN_BUFFER = 2
TOKEN_PAYLOAD_LIMIT = MAX_CHUNK_TOKENS - SPECIAL_TOKEN_BUFFER

# Keep table prefixes small enough to leave room for actual table rows.
MAX_TABLE_PREFIX_TOKENS = min(
    96,
    TOKEN_PAYLOAD_LIMIT // 2,
)


def get_raw_token_ids(text):
    """
    Tokenize text without adding model-specific special tokens.
    """
    return embedding_model.tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]


def count_raw_tokens(text):
    """
    Count tokenizer tokens without special tokens.
    """
    return len(get_raw_token_ids(text))


def count_embedding_tokens(text):
    """
    Count the final number of tokens passed to the embedding model.
    """
    return len(
        embedding_model.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )


def decode_token_slice(token_ids):
    """
    Decode tokenizer IDs back into normalized text.
    """
    return embedding_model.tokenizer.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()


def truncate_text_to_token_limit(text, token_limit):
    """
    Truncate text to an exact raw-token limit.
    """
    text = text.strip()

    if not text or token_limit <= 0:
        return ""

    token_ids = get_raw_token_ids(text)

    if len(token_ids) <= token_limit:
        return text

    return decode_token_slice(token_ids[:token_limit])


def split_to_token_safe_text(
    text,
    token_limit=TOKEN_PAYLOAD_LIMIT,
):
    """
    Split text using exact tokenizer IDs so every piece fits the token budget.
    """
    text = text.strip()

    if not text:
        return []

    raw_token_ids = get_raw_token_ids(text)

    if len(raw_token_ids) <= token_limit:
        return [text]

    safe_chunks = []

    for start in range(
        0,
        len(raw_token_ids),
        token_limit,
    ):
        token_slice = raw_token_ids[
            start:start + token_limit
        ]

        decoded_chunk = decode_token_slice(token_slice)

        if decoded_chunk:
            safe_chunks.append(decoded_chunk)

    return safe_chunks


def is_markdown_separator_row(line):
    """
    Return True when a Markdown table row only contains separators.

    Example:
    |-----|:------:|------:|
    """
    stripped_line = line.strip()

    if not stripped_line.startswith("|"):
        return False

    cells = [
        cell.strip()
        for cell in stripped_line.strip("|").split("|")
    ]

    if not cells:
        return False

    return all(
        bool(re.fullmatch(r":?-{3,}:?", cell))
        for cell in cells
    )


def build_safe_table_prefix(section, header_line):
    """
    Build a compact table prefix while preserving section and column context.

    Markdown separator rows are intentionally excluded because they add
    token cost without adding semantic information.
    """
    section = section.strip() or "Unknown section"
    header_line = header_line.strip()

    section_label = f"Section: {section}\nTable columns:\n"

    section_token_count = count_raw_tokens(section_label)

    if section_token_count >= MAX_TABLE_PREFIX_TOKENS:
        return truncate_text_to_token_limit(
            section_label,
            MAX_TABLE_PREFIX_TOKENS,
        )

    available_header_tokens = (
        MAX_TABLE_PREFIX_TOKENS
        - section_token_count
    )

    compact_header = truncate_text_to_token_limit(
        header_line,
        available_header_tokens,
    )

    prefix = section_label + compact_header

    return prefix.strip()


def split_text_into_chunks(text, section):
    """
    Split normal text by character size, then enforce the exact token limit.
    """
    text = text.strip()

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = min(
            start + TEXT_CHUNK_SIZE,
            len(text),
        )

        character_chunk = text[start:end].strip()

        for token_safe_chunk in split_to_token_safe_text(
            character_chunk
        ):
            chunks.append(
                {
                    "text": token_safe_chunk,
                    "chunk_type": "text",
                    "section": section,
                }
            )

        if end >= len(text):
            break

        next_start = end - TEXT_CHUNK_OVERLAP

        # Prevent an invalid overlap configuration from causing an infinite loop.
        start = max(
            next_start,
            start + 1,
        )

    return chunks


def split_oversized_table_row(
    row,
    table_prefix,
    section,
):
    """
    Split one oversized table row using exact tokenizer IDs.

    The compact table prefix is repeated for every resulting piece.
    """
    prefix_with_newline = table_prefix.rstrip() + "\n"

    prefix_token_count = count_raw_tokens(
        prefix_with_newline
    )

    available_row_tokens = (
        TOKEN_PAYLOAD_LIMIT
        - prefix_token_count
    )

    if available_row_tokens <= 0:
        raise RuntimeError(
            "The compact table prefix still exceeds the token budget. "
            f"Section: {section}. "
            f"Prefix tokens: {prefix_token_count}. "
            f"Payload limit: {TOKEN_PAYLOAD_LIMIT}."
        )

    row_token_ids = get_raw_token_ids(row)
    row_chunks = []

    for start in range(
        0,
        len(row_token_ids),
        available_row_tokens,
    ):
        token_slice = row_token_ids[
            start:start + available_row_tokens
        ]

        decoded_row_part = decode_token_slice(
            token_slice
        )

        if not decoded_row_part:
            continue

        chunk_text = (
            prefix_with_newline
            + decoded_row_part
        )

        final_token_count = count_embedding_tokens(
            chunk_text
        )

        if final_token_count > MAX_CHUNK_TOKENS:
            raise RuntimeError(
                "An oversized table-row piece was generated unexpectedly. "
                f"Section: {section}. "
                f"Generated tokens: {final_token_count}."
            )

        row_chunks.append(
            {
                "text": chunk_text,
                "chunk_type": "table",
                "section": section,
            }
        )

    return row_chunks


def split_table_into_chunks(
    table_lines,
    section,
):
    """
    Split a Markdown table by rows while repeating compact column context.

    Markdown separator rows are removed because they contain formatting,
    not useful semantic content.
    """
    if not table_lines:
        return []

    cleaned_lines = [
        line.strip()
        for line in table_lines
        if line.strip()
    ]

    if not cleaned_lines:
        return []

    header_line = cleaned_lines[0]

    if (
        len(cleaned_lines) > 1
        and is_markdown_separator_row(cleaned_lines[1])
    ):
        data_lines = cleaned_lines[2:]
    else:
        data_lines = cleaned_lines[1:]

    table_prefix = build_safe_table_prefix(
        section=section,
        header_line=header_line,
    )

    chunks = []
    current_rows = []

    def build_table_text(rows):
        """
        Combine the compact prefix with one or more table rows.
        """
        return (
            table_prefix.rstrip()
            + "\n"
            + "\n".join(rows)
        ).strip()

    def append_current_rows():
        """
        Append a table chunk only after confirming its token length.
        """
        nonlocal current_rows

        if not current_rows:
            return

        table_text = build_table_text(
            current_rows
        )

        token_count = count_embedding_tokens(
            table_text
        )

        if token_count > MAX_CHUNK_TOKENS:
            raise RuntimeError(
                "An internally generated table chunk exceeded "
                "the project token limit. "
                f"Section: {section}. "
                f"Generated tokens: {token_count}."
            )

        chunks.append(
            {
                "text": table_text,
                "chunk_type": "table",
                "section": section,
            }
        )

        current_rows = []

    if not data_lines:
        header_only_text = table_prefix.strip()

        for text_piece in split_to_token_safe_text(
            header_only_text
        ):
            chunks.append(
                {
                    "text": text_piece,
                    "chunk_type": "table",
                    "section": section,
                }
            )

        return chunks

    for row in data_lines:
        candidate_rows = current_rows + [row]

        candidate_text = build_table_text(
            candidate_rows
        )

        if (
            count_embedding_tokens(candidate_text)
            <= MAX_CHUNK_TOKENS
        ):
            current_rows = candidate_rows
            continue

        # Save rows that were already confirmed to fit.
        append_current_rows()

        single_row_text = build_table_text(
            [row]
        )

        if (
            count_embedding_tokens(single_row_text)
            <= MAX_CHUNK_TOKENS
        ):
            current_rows = [row]
        else:
            # A single row is too large and must be split exactly by tokens.
            chunks.extend(
                split_oversized_table_row(
                    row=row,
                    table_prefix=table_prefix,
                    section=section,
                )
            )

    append_current_rows()

    return chunks


def smart_chunk_markdown(markdown_text):
    """
    Split Markdown into token-safe text and table chunks with section context.
    """
    chunks = []
    lines = markdown_text.splitlines()

    current_section = "Document introduction"
    text_buffer = []

    def flush_text_buffer():
        """
        Convert accumulated normal text into token-safe chunks.
        """
        nonlocal text_buffer

        buffered_text = "\n".join(
            text_buffer
        ).strip()

        if buffered_text:
            chunks.extend(
                split_text_into_chunks(
                    text=buffered_text,
                    section=current_section,
                )
            )

        text_buffer = []

    line_index = 0

    while line_index < len(lines):
        line = lines[line_index]
        stripped_line = line.strip()

        heading_match = re.match(
            r"^(#{1,6})\s+(.+)$",
            stripped_line,
        )

        if heading_match:
            flush_text_buffer()

            current_section = (
                heading_match.group(2).strip()
            )

            text_buffer.append(
                stripped_line
            )

            line_index += 1
            continue

        if stripped_line.startswith("|"):
            flush_text_buffer()

            table_lines = []

            while (
                line_index < len(lines)
                and lines[line_index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[line_index]
                )

                line_index += 1

            chunks.extend(
                split_table_into_chunks(
                    table_lines=table_lines,
                    section=current_section,
                )
            )

            continue

        text_buffer.append(line)
        line_index += 1

    flush_text_buffer()

    for chunk_index, chunk in enumerate(
        chunks
    ):
        chunk["chunk_index"] = chunk_index

    return chunks


seed_chunks = {}
total_seed_chunks = 0

for filename, markdown_content in (
    parsed_seed_documents.items()
):
    document_chunks = smart_chunk_markdown(
        markdown_content
    )

    seed_chunks[filename] = document_chunks

    text_chunk_count = sum(
        chunk["chunk_type"] == "text"
        for chunk in document_chunks
    )

    table_chunk_count = sum(
        chunk["chunk_type"] == "table"
        for chunk in document_chunks
    )

    total_seed_chunks += len(
        document_chunks
    )

    print(f"\n📄 {filename}")
    print(
        f"   Total chunks: "
        f"{len(document_chunks)}"
    )
    print(
        f"   Text chunks: "
        f"{text_chunk_count}"
    )
    print(
        f"   Table chunks: "
        f"{table_chunk_count}"
    )

print(
    f"\n📊 Total chunks across all seed documents: "
    f"{total_seed_chunks}"
)
print(
    f"✅ Chunking completed with a "
    f"{MAX_CHUNK_TOKENS}-token safety target."
)
print(
    "ℹ️ Final token safety will be confirmed "
    "by the next validation cell."
)


📄 apple_fy2025_10k.pdf
   Total chunks: 609
   Text chunks: 522
   Table chunks: 87

📄 tesla_q4_2025_update.pdf
   Total chunks: 140
   Text chunks: 70
   Table chunks: 70

📊 Total chunks across all seed documents: 749
✅ Chunking completed with a 240-token safety target.
ℹ️ Final token safety will be confirmed by the next validation cell.


In [ ]:
# PHASE 1 - Cell 6: Prepare chunk records and validate embedding token limits

seed_document_lookup = {
    document["filename"]: document
    for document in SEED_DOCUMENTS
}

prepared_documents = {}
chunks_pending_indexing = []
all_chunk_records = []


if not seed_chunks:
    raise RuntimeError(
        "No chunks were generated. "
        "Run Phase 1 - Cell 5 successfully before validation."
    )


for filename, document_chunks in seed_chunks.items():
    if filename not in seed_document_lookup:
        raise KeyError(
            f"Seed document configuration was not found for: {filename}"
        )

    seed_document = seed_document_lookup[
        filename
    ]

    tenant_key = seed_document[
        "tenant_key"
    ]

    if tenant_key not in DEMO_TENANTS:
        raise KeyError(
            f"Tenant configuration was not found for: {tenant_key}"
        )

    tenant = DEMO_TENANTS[
        tenant_key
    ]

    tenant_id = tenant[
        "tenant_id"
    ]

    document_path = seed_document_paths[
        filename
    ]

    document_hash = calculate_document_hash(
        document_path
    )

    expected_chunk_count = len(
        document_chunks
    )

    indexed_chunk_count = get_indexed_chunk_count(
        tenant_id,
        document_hash,
    )

    prepared_documents[filename] = {
        "tenant_id": tenant_id,
        "document_hash": document_hash,
        "expected_chunk_count": expected_chunk_count,
        "indexed_chunk_count": indexed_chunk_count,
    }

    print(f"\n📄 {filename}")
    print(f"   Tenant: {tenant_id}")
    print(
        f"   Document hash: "
        f"{document_hash[:16]}..."
    )
    print(
        f"   Expected chunks: "
        f"{expected_chunk_count}"
    )
    print(
        f"   Already indexed: "
        f"{indexed_chunk_count}"
    )

    for chunk in document_chunks:
        required_chunk_fields = {
            "chunk_index",
            "chunk_type",
            "section",
            "text",
        }

        missing_chunk_fields = (
            required_chunk_fields
            - set(chunk.keys())
        )

        if missing_chunk_fields:
            raise KeyError(
                f"Chunk from {filename} is missing fields: "
                f"{sorted(missing_chunk_fields)}"
            )

        chunk_text = chunk[
            "text"
        ].strip()

        if not chunk_text:
            raise ValueError(
                f"An empty chunk was generated for {filename}. "
                f"Chunk index: {chunk['chunk_index']}"
            )

        chunk_record = {
            TENANT_ID_FIELD: tenant_id,
            DOCUMENT_HASH_FIELD: document_hash,
            FILENAME_FIELD: filename,
            CHUNK_INDEX_FIELD: chunk[
                "chunk_index"
            ],
            CHUNK_TYPE_FIELD: chunk[
                "chunk_type"
            ],
            "section": chunk[
                "section"
            ],
            TEXT_FIELD: chunk_text,
        }

        all_chunk_records.append(
            chunk_record
        )

        # Deterministic IDs will make re-indexing safe.
        # If the complete document is not present, prepare all chunks.
        if (
            indexed_chunk_count
            != expected_chunk_count
        ):
            chunks_pending_indexing.append(
                chunk_record
            )


total_chunk_count = len(
    all_chunk_records
)

pending_chunk_count = len(
    chunks_pending_indexing
)


if total_chunk_count == 0:
    raise RuntimeError(
        "Chunk validation cannot continue because no chunk records exist."
    )


all_chunk_texts = [
    record[TEXT_FIELD]
    for record in all_chunk_records
]


tokenized_chunks = embedding_model.tokenizer(
    all_chunk_texts,
    add_special_tokens=True,
    truncation=False,
)


token_lengths = [
    len(token_ids)
    for token_ids in tokenized_chunks[
        "input_ids"
    ]
]


model_token_limit = (
    embedding_model.max_seq_length
)

project_token_limit = (
    MAX_CHUNK_TOKENS
)


oversized_chunks = [
    (record, token_count)
    for record, token_count in zip(
        all_chunk_records,
        token_lengths,
    )
    if token_count > project_token_limit
]


hard_limit_violations = [
    (record, token_count)
    for record, token_count in zip(
        all_chunk_records,
        token_lengths,
    )
    if token_count > model_token_limit
]


longest_chunk_token_count = max(
    token_lengths
)

shortest_chunk_token_count = min(
    token_lengths
)

average_chunk_token_count = (
    sum(token_lengths)
    / len(token_lengths)
)


text_chunk_count = sum(
    record[CHUNK_TYPE_FIELD] == "text"
    for record in all_chunk_records
)

table_chunk_count = sum(
    record[CHUNK_TYPE_FIELD] == "table"
    for record in all_chunk_records
)


print("\n📊 Chunk validation summary")
print(
    f"   Total chunks across all documents: "
    f"{total_chunk_count}"
)
print(
    f"   Text chunks: "
    f"{text_chunk_count}"
)
print(
    f"   Table chunks: "
    f"{table_chunk_count}"
)
print(
    f"   Chunks pending indexing: "
    f"{pending_chunk_count}"
)
print(
    f"   Embedding model hard limit: "
    f"{model_token_limit}"
)
print(
    f"   Project safety limit: "
    f"{project_token_limit}"
)
print(
    f"   Shortest chunk: "
    f"{shortest_chunk_token_count} tokens"
)
print(
    f"   Average chunk: "
    f"{average_chunk_token_count:.1f} tokens"
)
print(
    f"   Longest chunk: "
    f"{longest_chunk_token_count} tokens"
)
print(
    f"   Project limit violations: "
    f"{len(oversized_chunks)}"
)
print(
    f"   Hard model limit violations: "
    f"{len(hard_limit_violations)}"
)


if oversized_chunks:
    example_record, example_token_count = (
        oversized_chunks[0]
    )

    raise RuntimeError(
        "Some chunks exceed the project token safety limit. "
        f"Example file: "
        f"{example_record[FILENAME_FIELD]}. "
        f"Section: "
        f"{example_record['section']}. "
        f"Chunk index: "
        f"{example_record[CHUNK_INDEX_FIELD]}. "
        f"Token count: "
        f"{example_token_count}. "
        f"Project limit: "
        f"{project_token_limit}. "
        "Chunking must be corrected before embeddings are generated."
    )


if hard_limit_violations:
    example_record, example_token_count = (
        hard_limit_violations[0]
    )

    raise RuntimeError(
        "Some chunks exceed the embedding model hard limit. "
        f"Example file: "
        f"{example_record[FILENAME_FIELD]}. "
        f"Section: "
        f"{example_record['section']}. "
        f"Token count: "
        f"{example_token_count}. "
        f"Model limit: "
        f"{model_token_limit}."
    )


print(
    "✅ All chunks fit within the project token safety limit."
)
print(
    "✅ No chunk exceeds the embedding model hard limit."
)
print(
    "✅ Chunk records are ready for embedding and safe indexing."
)


📄 apple_fy2025_10k.pdf
   Tenant: demo_apple_tenant
   Document hash: 3eb270b22acb7d8d...
   Expected chunks: 609
   Already indexed: 0

📄 tesla_q4_2025_update.pdf
   Tenant: demo_tesla_tenant
   Document hash: 05a5eb4fff82710f...
   Expected chunks: 140
   Already indexed: 0

📊 Chunk validation summary
   Total chunks across all documents: 749
   Text chunks: 592
   Table chunks: 157
   Chunks pending indexing: 749
   Embedding model hard limit: 256
   Project safety limit: 240
   Shortest chunk: 5 tokens
   Average chunk: 118.8 tokens
   Longest chunk: 240 tokens
   Project limit violations: 0
   Hard model limit violations: 0
✅ All chunks fit within the project token safety limit.
✅ No chunk exceeds the embedding model hard limit.
✅ Chunk records are ready for embedding and safe indexing.


In [ ]:
# PHASE 1 - Cell 7: Generate normalized embeddings for pending chunks

import time
import numpy as np

pending_chunk_texts = [
    record[TEXT_FIELD]
    for record in chunks_pending_indexing
]

pending_embedding_count = len(
    pending_chunk_texts
)

print(
    f"📊 Chunks pending embedding: "
    f"{pending_embedding_count}"
)


if pending_embedding_count == 0:
    pending_embeddings = np.empty(
        shape=(0, VECTOR_SIZE),
        dtype=np.float32,
    )

    print(
        "ℹ️ No embeddings were generated because "
        "all document chunks are already indexed."
    )

else:
    embedding_start_time = time.time()

    pending_embeddings = embedding_model.encode(
        pending_chunk_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    embedding_elapsed_time = (
        time.time()
        - embedding_start_time
    )

    if pending_embeddings.ndim != 2:
        raise RuntimeError(
            "The embedding model returned an unexpected array shape. "
            f"Received shape: {pending_embeddings.shape}"
        )

    generated_embedding_count = (
        pending_embeddings.shape[0]
    )

    generated_vector_size = (
        pending_embeddings.shape[1]
    )

    if (
        generated_embedding_count
        != pending_embedding_count
    ):
        raise RuntimeError(
            "The number of generated embeddings does not match "
            "the number of pending chunks. "
            f"Chunks: {pending_embedding_count}, "
            f"embeddings: {generated_embedding_count}."
        )

    if generated_vector_size != VECTOR_SIZE:
        raise RuntimeError(
            "The generated embedding size does not match "
            "the configured Qdrant vector size. "
            f"Expected: {VECTOR_SIZE}, "
            f"received: {generated_vector_size}."
        )

    if not np.isfinite(
        pending_embeddings
    ).all():
        raise RuntimeError(
            "The generated embeddings contain NaN "
            "or infinite values."
        )

    vector_norms = np.linalg.norm(
        pending_embeddings,
        axis=1,
    )

    average_vector_norm = float(
        vector_norms.mean()
    )

    print("\n📊 Embedding generation summary")
    print(
        f"   Generated embeddings: "
        f"{generated_embedding_count}"
    )
    print(
        f"   Vector dimensions: "
        f"{generated_vector_size}"
    )
    print(
        f"   Array shape: "
        f"{pending_embeddings.shape}"
    )
    print(
        f"   Data type: "
        f"{pending_embeddings.dtype}"
    )
    print(
        f"   Average vector norm: "
        f"{average_vector_norm:.4f}"
    )
    print(
        f"   Generation time: "
        f"{embedding_elapsed_time:.1f} seconds"
    )

    print(
        "✅ Embeddings were generated and normalized successfully."
    )
    print(
        "✅ Embeddings are ready for deterministic Qdrant indexing."
    )

📊 Chunks pending embedding: 749


Batches:   0%|          | 0/12 [00:00<?, ?it/s]


📊 Embedding generation summary
   Generated embeddings: 749
   Vector dimensions: 384
   Array shape: (749, 384)
   Data type: float32
   Average vector norm: 1.0000
   Generation time: 1.9 seconds
✅ Embeddings were generated and normalized successfully.
✅ Embeddings are ready for deterministic Qdrant indexing.


In [ ]:
# PHASE 1 - Cell 8: Build deterministic Qdrant points and index pending chunks

import time
import uuid

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
    PointStruct,
)


QDRANT_UPLOAD_BATCH_SIZE = 100


def build_deterministic_point_id(
    tenant_id,
    document_hash,
    chunk_index,
):
    """
    Build a stable UUID from tenant, document, and chunk identity.

    Re-indexing the same document generates the same point IDs,
    preventing duplicate Qdrant records.
    """
    identity_string = (
        f"{tenant_id}:"
        f"{document_hash}:"
        f"{chunk_index}"
    )

    return str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            identity_string,
        )
    )


if len(chunks_pending_indexing) != len(pending_embeddings):
    raise RuntimeError(
        "Pending chunk and embedding counts do not match. "
        f"Chunks: {len(chunks_pending_indexing)}, "
        f"embeddings: {len(pending_embeddings)}."
    )


qdrant_points = []


for chunk_record, embedding_vector in zip(
    chunks_pending_indexing,
    pending_embeddings,
):
    tenant_id = chunk_record[
        TENANT_ID_FIELD
    ]

    document_hash = chunk_record[
        DOCUMENT_HASH_FIELD
    ]

    chunk_index = chunk_record[
        CHUNK_INDEX_FIELD
    ]

    point_id = build_deterministic_point_id(
        tenant_id=tenant_id,
        document_hash=document_hash,
        chunk_index=chunk_index,
    )

    point_payload = {
        TENANT_ID_FIELD: tenant_id,
        DOCUMENT_HASH_FIELD: document_hash,
        FILENAME_FIELD: chunk_record[
            FILENAME_FIELD
        ],
        CHUNK_INDEX_FIELD: chunk_index,
        CHUNK_TYPE_FIELD: chunk_record[
            CHUNK_TYPE_FIELD
        ],
        "section": chunk_record[
            "section"
        ],
        TEXT_FIELD: chunk_record[
            TEXT_FIELD
        ],
    }

    qdrant_points.append(
        PointStruct(
            id=point_id,
            vector=embedding_vector.tolist(),
            payload=point_payload,
        )
    )


prepared_point_count = len(
    qdrant_points
)


print(
    f"📦 Prepared Qdrant points: "
    f"{prepared_point_count}"
)


if prepared_point_count == 0:
    print(
        "ℹ️ No Qdrant upload was needed because "
        "all document chunks are already indexed."
    )

else:
    upload_start_time = time.time()
    uploaded_point_count = 0

    total_batches = (
        prepared_point_count
        + QDRANT_UPLOAD_BATCH_SIZE
        - 1
    ) // QDRANT_UPLOAD_BATCH_SIZE

    for batch_number, start in enumerate(
        range(
            0,
            prepared_point_count,
            QDRANT_UPLOAD_BATCH_SIZE,
        ),
        start=1,
    ):
        point_batch = qdrant_points[
            start:start + QDRANT_UPLOAD_BATCH_SIZE
        ]

        qdrant.upsert(
            collection_name=COLLECTION_NAME,
            points=point_batch,
            wait=True,
        )

        uploaded_point_count += len(
            point_batch
        )

        print(
            f"   Uploaded batch "
            f"{batch_number}/{total_batches} "
            f"({len(point_batch)} points)"
        )

    upload_elapsed_time = (
        time.time()
        - upload_start_time
    )

    if uploaded_point_count != prepared_point_count:
        raise RuntimeError(
            "The uploaded point count does not match "
            "the prepared point count. "
            f"Prepared: {prepared_point_count}, "
            f"uploaded: {uploaded_point_count}."
        )

    print("\n📊 Qdrant indexing summary")
    print(
        f"   Uploaded points: "
        f"{uploaded_point_count}"
    )
    print(
        f"   Upload batches: "
        f"{total_batches}"
    )
    print(
        f"   Batch size: "
        f"{QDRANT_UPLOAD_BATCH_SIZE}"
    )
    print(
        f"   Upload time: "
        f"{upload_elapsed_time:.1f} seconds"
    )


print("\n🔍 Verifying indexed documents")


verification_failures = []


for filename, document_info in prepared_documents.items():
    tenant_id = document_info[
        "tenant_id"
    ]

    document_hash = document_info[
        "document_hash"
    ]

    expected_chunk_count = document_info[
        "expected_chunk_count"
    ]

    document_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(
                    value=tenant_id
                ),
            ),
            FieldCondition(
                key=DOCUMENT_HASH_FIELD,
                match=MatchValue(
                    value=document_hash
                ),
            ),
        ]
    )

    indexed_chunk_count = qdrant.count(
        collection_name=COLLECTION_NAME,
        count_filter=document_filter,
        exact=True,
    ).count

    verification_status = (
        "✅"
        if indexed_chunk_count
        == expected_chunk_count
        else "❌"
    )

    print(f"\n{verification_status} {filename}")
    print(
        f"   Tenant: "
        f"{tenant_id}"
    )
    print(
        f"   Expected chunks: "
        f"{expected_chunk_count}"
    )
    print(
        f"   Indexed chunks: "
        f"{indexed_chunk_count}"
    )

    if indexed_chunk_count != expected_chunk_count:
        verification_failures.append(
            {
                "filename": filename,
                "expected": expected_chunk_count,
                "indexed": indexed_chunk_count,
            }
        )


collection_info = qdrant.get_collection(
    collection_name=COLLECTION_NAME
)


print("\n📊 Collection summary")
print(
    f"   Collection: "
    f"{COLLECTION_NAME}"
)
print(
    f"   Total collection points: "
    f"{collection_info.points_count}"
)


if verification_failures:
    first_failure = verification_failures[0]

    raise RuntimeError(
        "Qdrant indexing verification failed. "
        f"Document: {first_failure['filename']}. "
        f"Expected: {first_failure['expected']}. "
        f"Indexed: {first_failure['indexed']}."
    )


print(
    "✅ All pending chunks were indexed successfully."
)
print(
    "✅ Document-level Qdrant counts match the expected chunk counts."
)
print(
    "✅ Deterministic IDs make repeated indexing duplicate-safe."
)

📦 Prepared Qdrant points: 749
   Uploaded batch 1/8 (100 points)
   Uploaded batch 2/8 (100 points)
   Uploaded batch 3/8 (100 points)
   Uploaded batch 4/8 (100 points)
   Uploaded batch 5/8 (100 points)
   Uploaded batch 6/8 (100 points)
   Uploaded batch 7/8 (100 points)
   Uploaded batch 8/8 (49 points)

📊 Qdrant indexing summary
   Uploaded points: 749
   Upload batches: 8
   Batch size: 100
   Upload time: 3.2 seconds

🔍 Verifying indexed documents

✅ apple_fy2025_10k.pdf
   Tenant: demo_apple_tenant
   Expected chunks: 609
   Indexed chunks: 609

✅ tesla_q4_2025_update.pdf
   Tenant: demo_tesla_tenant
   Expected chunks: 140
   Indexed chunks: 140

📊 Collection summary
   Collection: vaultify_v3_documents
   Total collection points: 749
✅ All pending chunks were indexed successfully.
✅ Document-level Qdrant counts match the expected chunk counts.
✅ Deterministic IDs make repeated indexing duplicate-safe.


In [ ]:
# PHASE 1 - Cell 9: Define tenant-filtered semantic search and run positive retrieval tests

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)


DEFAULT_RETRIEVAL_LIMIT = 5


def search_tenant_documents(
    query,
    tenant_id,
    limit=DEFAULT_RETRIEVAL_LIMIT,
):
    """
    Search document chunks belonging only to the specified tenant.

    The query is embedded with the same normalized embedding model used
    during ingestion. Qdrant applies the tenant filter before returning
    semantically similar chunks.
    """
    query = query.strip()

    if not query:
        raise ValueError(
            "The search query cannot be empty."
        )

    if not tenant_id:
        raise ValueError(
            "A tenant ID is required for document search."
        )

    if limit <= 0:
        raise ValueError(
            "The retrieval limit must be greater than zero."
        )

    query_vector = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    if query_vector.shape[0] != VECTOR_SIZE:
        raise RuntimeError(
            "The query embedding size does not match "
            "the configured Qdrant vector size. "
            f"Expected: {VECTOR_SIZE}, "
            f"received: {query_vector.shape[0]}."
        )

    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(
                    value=tenant_id
                ),
            )
        ]
    )

    search_response = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        query_filter=tenant_filter,
        limit=limit,
        with_payload=True,
    )

    return search_response.points


def print_search_results(
    query,
    tenant_id,
    results,
):
    """
    Print a readable summary of retrieved Qdrant chunks.
    """
    print("\n" + "=" * 90)
    print(f"🔎 Query: {query}")
    print(f"🏢 Tenant: {tenant_id}")
    print(f"📦 Retrieved results: {len(results)}")
    print("=" * 90)

    if not results:
        print(
            "⚠️ No document chunks were retrieved."
        )
        return

    for result_index, result in enumerate(
        results,
        start=1,
    ):
        payload = result.payload or {}

        filename = payload.get(
            FILENAME_FIELD,
            "Unknown file",
        )

        section = payload.get(
            "section",
            "Unknown section",
        )

        chunk_type = payload.get(
            CHUNK_TYPE_FIELD,
            "unknown",
        )

        chunk_index = payload.get(
            CHUNK_INDEX_FIELD,
            "unknown",
        )

        chunk_text = payload.get(
            TEXT_FIELD,
            "",
        ).strip()

        preview = (
            chunk_text[:700]
            + ("..." if len(chunk_text) > 700 else "")
        )

        print(f"\nResult {result_index}")
        print(f"   Score: {result.score:.4f}")
        print(f"   File: {filename}")
        print(f"   Section: {section}")
        print(f"   Chunk type: {chunk_type}")
        print(f"   Chunk index: {chunk_index}")
        print("   Content preview:")
        print(preview)


positive_retrieval_tests = [
    {
        "name": "Apple revenue retrieval",
        "tenant_id": DEMO_TENANTS[
            "company_a"
        ]["tenant_id"],
        "query": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
    },
    {
        "name": "Tesla revenue retrieval",
        "tenant_id": DEMO_TENANTS[
            "company_b"
        ]["tenant_id"],
        "query": (
            "What was Tesla's total revenue "
            "in the fourth quarter of 2025?"
        ),
    },
]


retrieval_test_results = {}


for retrieval_test in positive_retrieval_tests:
    test_name = retrieval_test[
        "name"
    ]

    tenant_id = retrieval_test[
        "tenant_id"
    ]

    query = retrieval_test[
        "query"
    ]

    results = search_tenant_documents(
        query=query,
        tenant_id=tenant_id,
        limit=DEFAULT_RETRIEVAL_LIMIT,
    )

    retrieval_test_results[
        test_name
    ] = results

    print_search_results(
        query=query,
        tenant_id=tenant_id,
        results=results,
    )


print("\n📊 Positive retrieval test summary")

for test_name, results in retrieval_test_results.items():
    status = "✅" if results else "❌"

    print(
        f"   {status} {test_name}: "
        f"{len(results)} results"
    )


if any(
    not results
    for results in retrieval_test_results.values()
):
    raise RuntimeError(
        "At least one positive retrieval test returned no results."
    )


print(
    "✅ Both tenants returned document chunks from their own indexed data."
)
print(
    "ℹ️ Review the result scores and content previews before "
    "continuing to answer generation."
)


🔎 Query: What were Apple's total net sales in fiscal year 2025?
🏢 Tenant: demo_apple_tenant
📦 Retrieved results: 5

Result 1
   Score: 0.6956
   File: apple_fy2025_10k.pdf
   Section: Mac
   Chunk type: text
   Chunk index: 220
   Content preview:
## Mac

Mac net sales increased during 2025 compared to 2024 primarily due to higher net sales of laptops and desktops.

Result 2
   Score: 0.6613
   File: apple_fy2025_10k.pdf
   Section: iPhone
   Chunk type: text
   Chunk index: 219
   Content preview:
## iPhone

iPhone net sales increased during 2025 compared to 2024 due to higher net sales of Pro models.

Result 3
   Score: 0.6574
   File: apple_fy2025_10k.pdf
   Section: Note 2 - Revenue
   Chunk type: table
   Chunk index: 318
   Content preview:
Section: Note 2 - Revenue
Table columns:
|                                                                                                    | 2025      | 2024      | 2023      |
| iPhone                                                      

In [ ]:
# PHASE 1 - Cell 10: Generate grounded answers from retrieved document chunks

DEFAULT_CONTEXT_RESULT_LIMIT = 5
GROQ_MODEL_NAME = "llama-3.3-70b-versatile"


def build_retrieval_context(results):
    """
    Convert retrieved Qdrant points into a structured context for the LLM.
    """
    if not results:
        return ""

    context_sections = []

    for result_index, result in enumerate(
        results,
        start=1,
    ):
        payload = result.payload or {}

        filename = payload.get(
            FILENAME_FIELD,
            "Unknown file",
        )

        section = payload.get(
            "section",
            "Unknown section",
        )

        chunk_type = payload.get(
            CHUNK_TYPE_FIELD,
            "unknown",
        )

        chunk_text = payload.get(
            TEXT_FIELD,
            "",
        ).strip()

        if not chunk_text:
            continue

        context_sections.append(
            "\n".join(
                [
                    f"[Context {result_index}]",
                    f"File: {filename}",
                    f"Section: {section}",
                    f"Chunk type: {chunk_type}",
                    f"Similarity score: {result.score:.4f}",
                    "Content:",
                    chunk_text,
                ]
            )
        )

    return "\n\n---\n\n".join(
        context_sections
    )


def answer_tenant_question(
    question,
    tenant_id,
    retrieval_limit=DEFAULT_CONTEXT_RESULT_LIMIT,
):
    """
    Retrieve tenant-isolated document chunks and generate a grounded answer.

    The language model is instructed to use only retrieved document context
    and to avoid answering when the required information is unavailable.
    """
    question = question.strip()

    if not question:
        raise ValueError(
            "The question cannot be empty."
        )

    if not tenant_id:
        raise ValueError(
            "A tenant ID is required."
        )

    retrieved_results = search_tenant_documents(
        query=question,
        tenant_id=tenant_id,
        limit=retrieval_limit,
    )

    retrieval_context = build_retrieval_context(
        retrieved_results
    )

    if not retrieval_context:
        return {
            "answer": (
                "The requested information was not found "
                "in the tenant's indexed documents."
            ),
            "results": retrieved_results,
        }

    system_prompt = """
You are Vaultify, a document-grounded business assistant.

Answer the user's question using only the supplied document context.

Rules:
1. Do not use outside knowledge.
2. Ignore context passages that are unrelated to the question.
3. Carefully distinguish quarterly values from annual values.
4. Carefully distinguish totals from category-level values.
5. Preserve the units used in the document, such as millions or billions.
6. If multiple years or quarters appear, select only the period requested.
7. If the supplied context does not contain enough information, clearly say so.
8. Never invent, estimate, or infer a missing number.
9. Give a direct answer first, followed by a brief explanation.
10. Mention the source file and section used for the answer.
""".strip()

    user_prompt = f"""
Question:
{question}

Retrieved document context:
{retrieval_context}
""".strip()

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=0.0,
        max_tokens=500,
    )

    answer = (
        response.choices[0]
        .message.content
        .strip()
    )

    return {
        "answer": answer,
        "results": retrieved_results,
    }


grounded_answer_tests = [
    {
        "name": "Apple total net sales",
        "tenant_id": DEMO_TENANTS[
            "company_a"
        ]["tenant_id"],
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
    },
    {
        "name": "Tesla Q4 total revenue",
        "tenant_id": DEMO_TENANTS[
            "company_b"
        ]["tenant_id"],
        "question": (
            "What was Tesla's total revenue "
            "in the fourth quarter of 2025?"
        ),
    },
]


grounded_answer_results = {}


for answer_test in grounded_answer_tests:
    print("\n" + "=" * 90)
    print(
        f"🧪 Test: "
        f"{answer_test['name']}"
    )
    print(
        f"🏢 Tenant: "
        f"{answer_test['tenant_id']}"
    )
    print(
        f"❓ Question: "
        f"{answer_test['question']}"
    )
    print("=" * 90)

    test_result = answer_tenant_question(
        question=answer_test["question"],
        tenant_id=answer_test["tenant_id"],
    )

    grounded_answer_results[
        answer_test["name"]
    ] = test_result

    print("\n🤖 Grounded answer:")
    print(
        test_result["answer"]
    )

    print(
        f"\n📦 Context chunks supplied to Groq: "
        f"{len(test_result['results'])}"
    )


print(
    "\n✅ Grounded answer generation tests completed."
)
print(
    "ℹ️ Verify that each answer uses the correct company, "
    "period, total, and document source."
)


🧪 Test: Apple total net sales
🏢 Tenant: demo_apple_tenant
❓ Question: What were Apple's total net sales in fiscal year 2025?

🤖 Grounded answer:
$416,161 million
The answer is found in the table in Section: Note 2 - Revenue of the file apple_fy2025_10k.pdf, which provides the total net sales for fiscal year 2025.

📦 Context chunks supplied to Groq: 5

🧪 Test: Tesla Q4 total revenue
🏢 Tenant: demo_tesla_tenant
❓ Question: What was Tesla's total revenue in the fourth quarter of 2025?

🤖 Grounded answer:
$24,901 million
The answer is found in the file "tesla_q4_2025_update.pdf", Section: (Unaudited), in the table under the column "Q4-2025" for the row "Total revenues". This value represents Tesla's total revenue in the fourth quarter of 2025.

📦 Context chunks supplied to Groq: 5

✅ Grounded answer generation tests completed.
ℹ️ Verify that each answer uses the correct company, period, total, and document source.


In [ ]:
# PHASE 1 - Cell 11: Run adversarial tenant-isolation tests

negative_tenant_tests = [
    {
        "name": "Apple tenant must not answer Tesla revenue",
        "tenant_id": DEMO_TENANTS[
            "company_a"
        ]["tenant_id"],
        "question": (
            "What was Tesla's total revenue "
            "in the fourth quarter of 2025?"
        ),
        "forbidden_filename": (
            "tesla_q4_2025_update.pdf"
        ),
    },
    {
        "name": "Tesla tenant must not answer Apple net sales",
        "tenant_id": DEMO_TENANTS[
            "company_b"
        ]["tenant_id"],
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "forbidden_filename": (
            "apple_fy2025_10k.pdf"
        ),
    },
]


negative_test_results = {}


for negative_test in negative_tenant_tests:
    print("\n" + "=" * 90)
    print(
        f"🔐 Test: "
        f"{negative_test['name']}"
    )
    print(
        f"🏢 Active tenant: "
        f"{negative_test['tenant_id']}"
    )
    print(
        f"❓ Question: "
        f"{negative_test['question']}"
    )
    print("=" * 90)

    test_result = answer_tenant_question(
        question=negative_test[
            "question"
        ],
        tenant_id=negative_test[
            "tenant_id"
        ],
    )

    negative_test_results[
        negative_test["name"]
    ] = test_result

    retrieved_filenames = {
        result.payload.get(
            FILENAME_FIELD
        )
        for result in test_result[
            "results"
        ]
        if result.payload
    }

    forbidden_filename = negative_test[
        "forbidden_filename"
    ]

    leaked_forbidden_document = (
        forbidden_filename
        in retrieved_filenames
    )

    print("\n🤖 Grounded answer:")
    print(
        test_result["answer"]
    )

    print(
        "\n📄 Retrieved files:"
    )

    for filename in sorted(
        filename
        for filename in retrieved_filenames
        if filename
    ):
        print(
            f"   - {filename}"
        )

    print(
        f"\n🚨 Forbidden document retrieved: "
        f"{leaked_forbidden_document}"
    )

    if leaked_forbidden_document:
        raise RuntimeError(
            "Critical tenant isolation failure. "
            f"The forbidden document "
            f"{forbidden_filename} "
            "was retrieved for another tenant."
        )


print("\n📊 Tenant isolation test summary")

for test_name, test_result in (
    negative_test_results.items()
):
    retrieved_filenames = {
        result.payload.get(
            FILENAME_FIELD
        )
        for result in test_result[
            "results"
        ]
        if result.payload
    }

    print(
        f"   ✅ {test_name}"
    )

    print(
        f"      Retrieved files: "
        f"{sorted(retrieved_filenames)}"
    )


print(
    "✅ No forbidden tenant documents were retrieved."
)
print(
    "✅ Qdrant tenant filtering prevented cross-tenant data access."
)
print(
    "ℹ️ Review the generated answers and confirm that "
    "the model did not invent information absent from the active tenant."
)


🔐 Test: Apple tenant must not answer Tesla revenue
🏢 Active tenant: demo_apple_tenant
❓ Question: What was Tesla's total revenue in the fourth quarter of 2025?

🤖 Grounded answer:
The supplied context does not contain enough information to answer the question about Tesla's total revenue in the fourth quarter of 2025. 

The provided document context appears to be related to Apple's financial reports, not Tesla's. Therefore, it is not possible to determine Tesla's total revenue in the fourth quarter of 2025 using the given context.

Source file: apple_fy2025_10k.pdf, various sections.

📄 Retrieved files:
   - apple_fy2025_10k.pdf

🚨 Forbidden document retrieved: False

🔐 Test: Tesla tenant must not answer Apple net sales
🏢 Active tenant: demo_tesla_tenant
❓ Question: What were Apple's total net sales in fiscal year 2025?

🤖 Grounded answer:
No, the total net sales for Apple in fiscal year 2025 are not available in the provided document context. 

The provided document context appears to

In [ ]:
# PHASE 2 - Cell 1: Define the tenant-bound Vaultify MCP server and document tool

from importlib.metadata import version

from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.exceptions import ToolError


MCP_SERVER_NAME = "Vaultify"
MCP_RETRIEVAL_LIMIT = 5

# The tenant is selected on the trusted server side.
# Never expose tenant_id as a user-controlled MCP tool argument.
ACTIVE_DEMO_TENANT_KEY = "company_a"

ACTIVE_DEMO_TENANT_ID = DEMO_TENANTS[
    ACTIVE_DEMO_TENANT_KEY
]["tenant_id"]


vaultify_mcp = FastMCP(
    MCP_SERVER_NAME
)


def serialize_retrieval_sources(results):
    """
    Convert retrieved Qdrant points into MCP-safe source metadata.
    """
    serialized_sources = []

    for result in results:
        payload = result.payload or {}

        serialized_sources.append(
            {
                "filename": payload.get(
                    FILENAME_FIELD,
                    "Unknown file",
                ),
                "section": payload.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": payload.get(
                    CHUNK_TYPE_FIELD,
                    "unknown",
                ),
                "chunk_index": payload.get(
                    CHUNK_INDEX_FIELD,
                ),
                "similarity_score": round(
                    float(result.score),
                    4,
                ),
            }
        )

    return serialized_sources


@vaultify_mcp.tool()
def ask_documents(question: str) -> dict:
    """
    Answer a question using only documents belonging to the authenticated tenant.

    The tool performs tenant-filtered semantic retrieval and grounded answer
    generation. It never accepts a tenant ID from the MCP client.

    Args:
        question: A question about the active tenant's indexed documents.

    Returns:
        A grounded answer together with non-sensitive source metadata.
    """
    cleaned_question = question.strip()

    if not cleaned_question:
        raise ToolError(
            "The question cannot be empty."
        )

    try:
        result = answer_tenant_question(
            question=cleaned_question,
            tenant_id=ACTIVE_DEMO_TENANT_ID,
            retrieval_limit=MCP_RETRIEVAL_LIMIT,
        )

    except Exception as error:
        raise ToolError(
            "Vaultify could not process the document question. "
            f"Reason: {error}"
        ) from error

    return {
        "answer": result["answer"],
        "tenant_id": ACTIVE_DEMO_TENANT_ID,
        "sources": serialize_retrieval_sources(
            result["results"]
        ),
    }


print("✅ Vaultify MCP server was defined successfully.")
print(f"📦 MCP SDK version: {version('mcp')}")
print(f"🛠️ Registered tool: ask_documents")
print(f"🏢 Active demo tenant key: {ACTIVE_DEMO_TENANT_KEY}")
print(f"🔐 Active demo tenant ID: {ACTIVE_DEMO_TENANT_ID}")
print(
    "✅ tenant_id is controlled by the server "
    "and is not exposed as a tool argument."
)
print(
    "ℹ️ The MCP server has only been defined; "
    "no network port has been opened yet."
)

✅ Vaultify MCP server was defined successfully.
📦 MCP SDK version: 1.28.1
🛠️ Registered tool: ask_documents
🏢 Active demo tenant key: company_a
🔐 Active demo tenant ID: demo_apple_tenant
✅ tenant_id is controlled by the server and is not exposed as a tool argument.
ℹ️ The MCP server has only been defined; no network port has been opened yet.


In [ ]:
# PHASE 2 - Cell 2: Test the MCP tool through an in-memory client session

import json

from mcp.shared.memory import (
    create_connected_server_and_client_session,
)


MCP_TEST_QUESTION = (
    "What were Apple's total net sales "
    "in fiscal year 2025?"
)


async with create_connected_server_and_client_session(
    vaultify_mcp,
    raise_exceptions=True,
) as mcp_client_session:

    available_tools = (
        await mcp_client_session.list_tools()
    )

    registered_tool_names = [
        tool.name
        for tool in available_tools.tools
    ]

    print("📋 Registered MCP tools:")

    for tool_name in registered_tool_names:
        print(f"   - {tool_name}")

    if "ask_documents" not in registered_tool_names:
        raise RuntimeError(
            "The ask_documents tool was not exposed "
            "through the MCP protocol."
        )

    print("\n🧪 Calling MCP tool")
    print(
        f"   Question: {MCP_TEST_QUESTION}"
    )

    tool_result = await mcp_client_session.call_tool(
        "ask_documents",
        {
            "question": MCP_TEST_QUESTION,
        },
    )


structured_result = (
    tool_result.structuredContent
    if tool_result.structuredContent
    else None
)


if structured_result is None:
    text_content_blocks = [
        content.text
        for content in tool_result.content
        if hasattr(content, "text")
    ]

    if not text_content_blocks:
        raise RuntimeError(
            "The MCP tool returned no readable content."
        )

    raw_text_result = "\n".join(
        text_content_blocks
    )

    try:
        structured_result = json.loads(
            raw_text_result
        )

    except json.JSONDecodeError as error:
        raise RuntimeError(
            "The MCP tool response could not be parsed "
            "as structured JSON."
        ) from error


if tool_result.isError:
    raise RuntimeError(
        "The MCP tool returned an error response."
    )


returned_answer = structured_result.get(
    "answer",
    "",
).strip()

returned_tenant_id = structured_result.get(
    "tenant_id"
)

returned_sources = structured_result.get(
    "sources",
    [],
)


if not returned_answer:
    raise RuntimeError(
        "The MCP tool returned an empty answer."
    )


if returned_tenant_id != ACTIVE_DEMO_TENANT_ID:
    raise RuntimeError(
        "The MCP tool returned an unexpected tenant ID. "
        f"Expected: {ACTIVE_DEMO_TENANT_ID}, "
        f"received: {returned_tenant_id}."
    )


if not returned_sources:
    raise RuntimeError(
        "The MCP tool returned no source metadata."
    )


unexpected_source_files = {
    source.get("filename")
    for source in returned_sources
    if source.get("filename")
    != "apple_fy2025_10k.pdf"
}


print("\n🤖 MCP answer:")
print(returned_answer)

print("\n📚 MCP sources:")

for source_index, source in enumerate(
    returned_sources,
    start=1,
):
    print(
        f"   {source_index}. "
        f"{source.get('filename')} | "
        f"{source.get('section')} | "
        f"score={source.get('similarity_score')}"
    )

print(
    f"\n🔐 Returned tenant ID: "
    f"{returned_tenant_id}"
)

print(
    f"🚨 Unexpected source files: "
    f"{sorted(unexpected_source_files)}"
)


if unexpected_source_files:
    raise RuntimeError(
        "The MCP tool returned a document belonging "
        "to an unexpected tenant."
    )


print(
    "\n✅ The MCP server exposed the expected tool."
)
print(
    "✅ The tool was called through the MCP protocol successfully."
)
print(
    "✅ The answer was grounded in Apple tenant documents."
)
print(
    "✅ No unexpected tenant document appeared in the MCP response."
)
print(
    "ℹ️ No network port or external tunnel was required for this test."
)

📋 Registered MCP tools:
   - ask_documents

🧪 Calling MCP tool
   Question: What were Apple's total net sales in fiscal year 2025?

🤖 MCP answer:
$416,161 million
The answer is found in the table in Section "Note 2 - Revenue" of the file apple_fy2025_10k.pdf, which provides the total net sales for fiscal year 2025.

📚 MCP sources:
   1. apple_fy2025_10k.pdf | Mac | score=0.6956
   2. apple_fy2025_10k.pdf | iPhone | score=0.6613
   3. apple_fy2025_10k.pdf | Note 2 - Revenue | score=0.6574
   4. apple_fy2025_10k.pdf | 18 U.S.C. SECTION 1350, AS ADOPTED PURSUANT TO SECTION 906 OF THE SARBANES-OXLEY ACT OF 2002 | score=0.6401
   5. apple_fy2025_10k.pdf | Products and Services Performance | score=0.635

🔐 Returned tenant ID: demo_apple_tenant
🚨 Unexpected source files: []

✅ The MCP server exposed the expected tool.
✅ The tool was called through the MCP protocol successfully.
✅ The answer was grounded in Apple tenant documents.
✅ No unexpected tenant document appeared in the MCP response.
ℹ

In [ ]:
# PHASE 2 - Cell 3: Start one remote-compatible MCP server and test it locally

import json
import socket
import threading
import time

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.exceptions import ToolError
from mcp.server.transport_security import TransportSecuritySettings


PUBLIC_MCP_HOST = "127.0.0.1"
PUBLIC_MCP_PORT = 8003

LOCAL_PUBLIC_MCP_URL = (
    f"http://{PUBLIC_MCP_HOST}:"
    f"{PUBLIC_MCP_PORT}/mcp"
)


def is_port_open(host, port):
    """
    Return True when a TCP server is listening on the target port.
    """
    with socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    ) as connection_socket:
        connection_socket.settimeout(0.5)

        return (
            connection_socket.connect_ex(
                (host, port)
            )
            == 0
        )


def parse_mcp_tool_result(tool_result):
    """
    Parse structured MCP output or its JSON text fallback.
    """
    if tool_result.isError:
        error_messages = [
            content.text
            for content in tool_result.content
            if hasattr(content, "text")
        ]

        raise RuntimeError(
            "The MCP tool returned an error. "
            + " ".join(error_messages)
        )

    if tool_result.structuredContent:
        parsed_result = tool_result.structuredContent

        if (
            isinstance(parsed_result, dict)
            and isinstance(
                parsed_result.get("result"),
                dict,
            )
        ):
            return parsed_result["result"]

        if isinstance(parsed_result, dict):
            return parsed_result

    text_blocks = [
        content.text.strip()
        for content in tool_result.content
        if hasattr(content, "text")
        and content.text.strip()
    ]

    for text_block in text_blocks:
        try:
            parsed_result = json.loads(
                text_block
            )

        except json.JSONDecodeError:
            continue

        if (
            isinstance(parsed_result, dict)
            and isinstance(
                parsed_result.get("result"),
                dict,
            )
        ):
            return parsed_result["result"]

        if isinstance(parsed_result, dict):
            return parsed_result

    combined_text = "\n".join(
        text_blocks
    )

    try:
        parsed_result = json.loads(
            combined_text
        )

    except json.JSONDecodeError as error:
        raise RuntimeError(
            "The MCP response could not be parsed. "
            f"Raw content: {combined_text[:1000]}"
        ) from error

    if (
        isinstance(parsed_result, dict)
        and isinstance(
            parsed_result.get("result"),
            dict,
        )
    ):
        return parsed_result["result"]

    if not isinstance(parsed_result, dict):
        raise RuntimeError(
            "The parsed MCP response is not an object."
        )

    return parsed_result


def validate_mcp_payload(payload):
    """
    Validate the answer, tenant ID, and returned source files.
    """
    answer = str(
        payload.get(
            "answer",
            "",
        )
    ).strip()

    tenant_id = payload.get(
        "tenant_id"
    )

    sources = payload.get(
        "sources",
        [],
    )

    if not answer:
        raise RuntimeError(
            "The MCP response contained no answer."
        )

    if tenant_id != ACTIVE_DEMO_TENANT_ID:
        raise RuntimeError(
            "The MCP response returned the wrong tenant. "
            f"Expected: {ACTIVE_DEMO_TENANT_ID}, "
            f"received: {tenant_id}."
        )

    if not isinstance(sources, list):
        raise RuntimeError(
            "The MCP sources field is not a list."
        )

    unexpected_files = {
        source.get("filename")
        for source in sources
        if isinstance(source, dict)
        and source.get("filename")
        != "apple_fy2025_10k.pdf"
    }

    if unexpected_files:
        raise RuntimeError(
            "Unexpected tenant files were returned. "
            f"Files: {sorted(unexpected_files)}"
        )

    return {
        "answer": answer,
        "tenant_id": tenant_id,
        "sources": sources,
    }


public_vaultify_mcp = FastMCP(
    "Vaultify",
    host=PUBLIC_MCP_HOST,
    port=PUBLIC_MCP_PORT,
    stateless_http=True,
    json_response=True,
    transport_security=TransportSecuritySettings(
        # Quick Tunnel hostnames are random.
        # Disable this protection only for the temporary demo.
        enable_dns_rebinding_protection=False,
    ),
)


@public_vaultify_mcp.tool()
def ask_documents(question: str) -> dict:
    """
    Answer a question using only the active tenant's documents.

    The tenant identity is controlled by the server and cannot be
    provided or changed by the MCP client.
    """
    cleaned_question = question.strip()

    if not cleaned_question:
        raise ToolError(
            "The question cannot be empty."
        )

    try:
        result = answer_tenant_question(
            question=cleaned_question,
            tenant_id=ACTIVE_DEMO_TENANT_ID,
            retrieval_limit=MCP_RETRIEVAL_LIMIT,
        )

    except Exception as error:
        raise ToolError(
            "Vaultify could not process the question. "
            f"Reason: {error}"
        ) from error

    return {
        "answer": result["answer"],
        "tenant_id": ACTIVE_DEMO_TENANT_ID,
        "sources": serialize_retrieval_sources(
            result["results"]
        ),
    }


def run_public_mcp_server():
    """
    Run the stateless JSON MCP server.
    """
    public_vaultify_mcp.run(
        transport="streamable-http"
    )


if is_port_open(
    PUBLIC_MCP_HOST,
    PUBLIC_MCP_PORT,
):
    print(
        f"ℹ️ Port {PUBLIC_MCP_PORT} is already open. "
        "The existing server will be reused."
    )

else:
    public_mcp_server_thread = threading.Thread(
        target=run_public_mcp_server,
        name="vaultify-public-mcp-server",
        daemon=True,
    )

    public_mcp_server_thread.start()

    startup_deadline = time.time() + 15

    while (
        time.time() < startup_deadline
        and not is_port_open(
            PUBLIC_MCP_HOST,
            PUBLIC_MCP_PORT,
        )
    ):
        time.sleep(0.25)

    if not is_port_open(
        PUBLIC_MCP_HOST,
        PUBLIC_MCP_PORT,
    ):
        raise RuntimeError(
            "The Vaultify MCP server did not start "
            f"on port {PUBLIC_MCP_PORT}."
        )


print(
    f"🌐 Local MCP endpoint: "
    f"{LOCAL_PUBLIC_MCP_URL}"
)


async def test_public_mcp_locally():
    """
    Test the remote-compatible MCP server locally.
    """
    async with streamablehttp_client(
        LOCAL_PUBLIC_MCP_URL
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            initialization_result = (
                await session.initialize()
            )

            tools_result = (
                await session.list_tools()
            )

            tool_names = [
                tool.name
                for tool in tools_result.tools
            ]

            if "ask_documents" not in tool_names:
                raise RuntimeError(
                    "The ask_documents tool "
                    "was not registered."
                )

            tool_result = await session.call_tool(
                "ask_documents",
                {
                    "question": (
                        "What were Apple's total net sales "
                        "in fiscal year 2025?"
                    )
                },
            )

            return {
                "server_name": (
                    initialization_result
                    .serverInfo.name
                ),
                "server_version": (
                    initialization_result
                    .serverInfo.version
                ),
                "tool_names": tool_names,
                "tool_result": tool_result,
            }


local_test_result = (
    await test_public_mcp_locally()
)


local_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        local_test_result["tool_result"]
    )
)


print("\n📊 Local MCP test")
print(
    f"   Server: "
    f"{local_test_result['server_name']} "
    f"{local_test_result['server_version']}"
)
print(
    f"   Tools: "
    f"{local_test_result['tool_names']}"
)
print(
    f"   Tenant: "
    f"{local_payload['tenant_id']}"
)
print(
    f"   Sources: "
    f"{len(local_payload['sources'])}"
)

print("\n🤖 Local MCP answer:")
print(
    local_payload["answer"]
)

print(
    "\n✅ The final MCP server works locally."
)
print(
    "✅ Stateless JSON transport is enabled."
)
print(
    "✅ Tenant isolation remained active."
)
print(
    f"✅ Port {PUBLIC_MCP_PORT} is ready "
    "for Cloudflare."
)

INFO:     Started server process [619]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8003 (Press CTRL+C to quit)


🌐 Local MCP endpoint: http://127.0.0.1:8003/mcp
INFO:     127.0.0.1:33962 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:33962 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:33966 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:33966 - "POST /mcp HTTP/1.1" 200 OK

📊 Local MCP test
   Server: Vaultify 1.28.1
   Tools: ['ask_documents']
   Tenant: demo_apple_tenant
   Sources: 5

🤖 Local MCP answer:
$416,161 million
The answer is found in the table in Section "Note 2 - Revenue" of the file apple_fy2025_10k.pdf, which provides the total net sales for fiscal year 2025.

✅ The final MCP server works locally.
✅ Stateless JSON transport is enabled.
✅ Tenant isolation remained active.
✅ Port 8003 is ready for Cloudflare.


In [ ]:
# PHASE 2 - Cell 4A: Install Cloudflare Tunnel client

import shutil
import subprocess


if shutil.which("cloudflared") is None:
    print("📥 Installing cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
            "-O",
            "/tmp/cloudflared.deb",
        ],
        check=True,
    )

    subprocess.run(
        [
            "dpkg",
            "-i",
            "/tmp/cloudflared.deb",
        ],
        check=True,
    )

    print("✅ cloudflared installed successfully.")

else:
    print("ℹ️ cloudflared is already installed.")


cloudflared_path = shutil.which(
    "cloudflared"
)

if not cloudflared_path:
    raise RuntimeError(
        "cloudflared installation could not be verified."
    )


version_result = subprocess.run(
    [
        cloudflared_path,
        "--version",
    ],
    capture_output=True,
    text=True,
    check=True,
)


print(
    f"📦 Binary path: {cloudflared_path}"
)
print(
    f"📦 Version: {version_result.stdout.strip()}"
)

ℹ️ cloudflared is already installed.
📦 Binary path: /usr/local/bin/cloudflared
📦 Version: cloudflared version 2026.7.1 (built 2026-07-09-13:00 UTC)


In [ ]:
# PHASE 2 - Cell 4: Create and test the public Cloudflare MCP endpoint

import asyncio
import re
import shutil
import subprocess
import threading
import time

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


CLOUDFLARED_BINARY = "cloudflared"
CLOUDFLARE_STARTUP_TIMEOUT_SECONDS = 45
PUBLIC_TEST_ATTEMPTS = 10
PUBLIC_TEST_RETRY_SECONDS = 3

CLOUDFLARE_ORIGIN_URL = (
    f"http://{PUBLIC_MCP_HOST}:"
    f"{PUBLIC_MCP_PORT}"
)


def stream_cloudflared_logs(
    process,
    log_storage,
):
    """
    Read and display cloudflared logs.
    """
    if process.stderr is None:
        return

    for log_line in iter(
        process.stderr.readline,
        "",
    ):
        cleaned_line = log_line.strip()

        if cleaned_line:
            log_storage.append(
                cleaned_line
            )

            print(
                f"[cloudflared] "
                f"{cleaned_line}"
            )

        if process.poll() is not None:
            break


def find_cloudflare_public_url(
    log_lines,
):
    """
    Extract the TryCloudflare URL from cloudflared logs.
    """
    url_pattern = re.compile(
        r"https://[a-zA-Z0-9-]+"
        r"\.trycloudflare\.com"
    )

    for log_line in log_lines:
        url_match = url_pattern.search(
            log_line
        )

        if url_match:
            return url_match.group(0)

    return None


def tunnel_is_registered(log_lines):
    """
    Return True after Cloudflare registers the tunnel connection.
    """
    return any(
        "Registered tunnel connection"
        in log_line
        for log_line in log_lines
    )


def format_exception_tree(
    error,
    indent=0,
):
    """
    Convert nested exception groups into readable lines.
    """
    prefix = " " * indent

    error_lines = [
        f"{prefix}{type(error).__name__}: {error}"
    ]

    nested_errors = getattr(
        error,
        "exceptions",
        None,
    )

    if nested_errors:
        for nested_error in nested_errors:
            error_lines.extend(
                format_exception_tree(
                    nested_error,
                    indent + 2,
                )
            )

    return error_lines


if shutil.which(
    CLOUDFLARED_BINARY
) is None:
    raise RuntimeError(
        "cloudflared is not installed. "
        "Run Phase 2 - Cell 4A first."
    )


if not is_port_open(
    PUBLIC_MCP_HOST,
    PUBLIC_MCP_PORT,
):
    raise RuntimeError(
        "The final MCP server is not running. "
        "Run Phase 2 - Cell 3 first."
    )


if (
    "cloudflare_tunnel_process"
    in globals()
    and cloudflare_tunnel_process.poll()
    is None
):
    print(
        "ℹ️ Stopping the previous "
        "Cloudflare tunnel."
    )

    cloudflare_tunnel_process.terminate()

    try:
        cloudflare_tunnel_process.wait(
            timeout=5
        )

    except subprocess.TimeoutExpired:
        cloudflare_tunnel_process.kill()
        cloudflare_tunnel_process.wait(
            timeout=5
        )


cloudflare_tunnel_logs = []


cloudflare_command = [
    CLOUDFLARED_BINARY,
    "tunnel",
    "--url",
    CLOUDFLARE_ORIGIN_URL,
    "--no-autoupdate",
]


print(
    f"🌐 Cloudflare origin: "
    f"{CLOUDFLARE_ORIGIN_URL}"
)


cloudflare_tunnel_process = subprocess.Popen(
    cloudflare_command,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,
)


cloudflare_log_thread = threading.Thread(
    target=stream_cloudflared_logs,
    args=(
        cloudflare_tunnel_process,
        cloudflare_tunnel_logs,
    ),
    name="cloudflare-log-reader",
    daemon=True,
)

cloudflare_log_thread.start()


startup_deadline = (
    time.time()
    + CLOUDFLARE_STARTUP_TIMEOUT_SECONDS
)

CLOUDFLARE_PUBLIC_URL = None


while time.time() < startup_deadline:
    if cloudflare_tunnel_process.poll() is not None:
        raise RuntimeError(
            "cloudflared stopped unexpectedly. "
            f"Exit code: "
            f"{cloudflare_tunnel_process.returncode}. "
            f"Recent logs: "
            f"{cloudflare_tunnel_logs[-10:]}"
        )

    CLOUDFLARE_PUBLIC_URL = (
        find_cloudflare_public_url(
            cloudflare_tunnel_logs
        )
    )

    if (
        CLOUDFLARE_PUBLIC_URL
        and tunnel_is_registered(
            cloudflare_tunnel_logs
        )
    ):
        break

    time.sleep(0.25)


if not CLOUDFLARE_PUBLIC_URL:
    raise RuntimeError(
        "Cloudflare did not provide "
        "a public URL."
    )


if not tunnel_is_registered(
    cloudflare_tunnel_logs
):
    raise RuntimeError(
        "Cloudflare created a URL but "
        "did not register the tunnel connection."
    )


PUBLIC_MCP_URL = (
    CLOUDFLARE_PUBLIC_URL.rstrip("/")
    + "/mcp"
)


print(
    "\n✅ Cloudflare Quick Tunnel started."
)
print(
    f"🌍 Public tunnel URL: "
    f"{CLOUDFLARE_PUBLIC_URL}"
)
print(
    f"🔗 Public MCP endpoint: "
    f"{PUBLIC_MCP_URL}"
)


async def call_public_mcp_once():
    """
    Run one complete MCP request through Cloudflare.
    """
    async with streamablehttp_client(
        PUBLIC_MCP_URL
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            initialization_result = (
                await session.initialize()
            )

            tools_result = (
                await session.list_tools()
            )

            tool_names = [
                tool.name
                for tool in tools_result.tools
            ]

            if "ask_documents" not in tool_names:
                raise RuntimeError(
                    "The ask_documents tool "
                    "was not found publicly."
                )

            tool_result = await session.call_tool(
                "ask_documents",
                {
                    "question": (
                        "What were Apple's total net sales "
                        "in fiscal year 2025?"
                    )
                },
            )

            return {
                "server_name": (
                    initialization_result
                    .serverInfo.name
                ),
                "server_version": (
                    initialization_result
                    .serverInfo.version
                ),
                "tool_names": tool_names,
                "tool_result": tool_result,
            }


public_test_result = None
last_public_error = None


for attempt_number in range(
    1,
    PUBLIC_TEST_ATTEMPTS + 1,
):
    try:
        public_test_result = (
            await call_public_mcp_once()
        )

        break

    except BaseException as error:
        last_public_error = error

        print(
            f"\n⏳ Public MCP attempt "
            f"{attempt_number}/"
            f"{PUBLIC_TEST_ATTEMPTS} failed."
        )

        for error_line in format_exception_tree(
            error
        ):
            print(
                f"   {error_line}"
            )

        if (
            attempt_number
            < PUBLIC_TEST_ATTEMPTS
        ):
            await asyncio.sleep(
                PUBLIC_TEST_RETRY_SECONDS
            )


if public_test_result is None:
    raise RuntimeError(
        "The public MCP endpoint remained "
        "unreachable after all attempts."
    ) from last_public_error


public_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        public_test_result["tool_result"]
    )
)


print("\n📊 Public MCP test")
print(
    f"   Server: "
    f"{public_test_result['server_name']} "
    f"{public_test_result['server_version']}"
)
print(
    f"   Tools: "
    f"{public_test_result['tool_names']}"
)
print(
    f"   Tenant: "
    f"{public_payload['tenant_id']}"
)
print(
    f"   Sources: "
    f"{len(public_payload['sources'])}"
)

print("\n🤖 Public MCP answer:")
print(
    public_payload["answer"]
)

print(
    "\n✅ The Vaultify MCP server is "
    "reachable through Cloudflare."
)
print(
    "✅ Public MCP initialization, "
    "tool discovery, and execution succeeded."
)
print(
    "✅ Tenant isolation remained active."
)
print(
    f"🔗 Claude-ready MCP endpoint: "
    f"{PUBLIC_MCP_URL}"
)
print(
    "⚠️ This URL expires when the "
    "Colab runtime stops."
)

ℹ️ Stopping the previous Cloudflare tunnel.
[cloudflared] 2026-07-10T16:14:00Z INF Initiating graceful shutdown due to signal terminated ...
[cloudflared] 2026-07-10T16:14:00Z ERR failed to run the datagram handler error="context canceled" connIndex=0 event=0 ip=198.41.200.43
[cloudflared] 2026-07-10T16:14:00Z ERR failed to serve tunnel connection error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.200.43
[cloudflared] 2026-07-10T16:14:00Z ERR Serve tunnel error error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.200.43
[cloudflared] 2026-07-10T16:14:00Z INF Retrying connection in up to 1s connIndex=0 event=0 ip=198.41.200.43
[cloudflared] 2026-07-10T16:14:00Z ERR Connection terminated connIndex=0
[cloudflared] 2026-07-10T16:14:00Z ERR no more connections active and exiting
[cloudflared] 2026-07-10T16:14:00Z INF Tunnel server stopped
[cloudflared] 2026-07-10T16:14:00Z INF Metrics server stopped
🌐 

RuntimeError: The public MCP endpoint remained unreachable after all attempts.

In [ ]:
# PHASE 2 - Cell 4B: Resolve the Quick Tunnel hostname and retry the public MCP test

import asyncio
import json
import socket
import time
from pathlib import Path
from urllib.parse import urlparse

import requests


DNS_RETRY_ATTEMPTS = 12
DNS_RETRY_SECONDS = 5

PUBLIC_MCP_HOSTNAME = urlparse(
    PUBLIC_MCP_URL
).hostname


def query_google_dns(hostname):
    """
    Resolve an A record through Google DNS over HTTPS.
    """
    response = requests.get(
        "https://dns.google/resolve",
        params={
            "name": hostname,
            "type": "A",
        },
        timeout=10,
    )

    response.raise_for_status()

    payload = response.json()

    return [
        answer["data"]
        for answer in payload.get("Answer", [])
        if answer.get("type") == 1
    ]


def query_cloudflare_dns(hostname):
    """
    Resolve an A record through Cloudflare DNS over HTTPS.
    """
    response = requests.get(
        "https://cloudflare-dns.com/dns-query",
        params={
            "name": hostname,
            "type": "A",
        },
        headers={
            "Accept": "application/dns-json",
        },
        timeout=10,
    )

    response.raise_for_status()

    payload = response.json()

    return [
        answer["data"]
        for answer in payload.get("Answer", [])
        if answer.get("type") == 1
    ]


def resolve_with_public_dns(hostname):
    """
    Try multiple public DNS resolvers.
    """
    resolved_addresses = []

    for resolver_function in (
        query_cloudflare_dns,
        query_google_dns,
    ):
        try:
            resolved_addresses.extend(
                resolver_function(hostname)
            )

        except Exception as error:
            print(
                f"⚠️ {resolver_function.__name__} failed: "
                f"{error}"
            )

    return list(
        dict.fromkeys(resolved_addresses)
    )


def add_temporary_hosts_entry(
    hostname,
    ip_address,
):
    """
    Add an ephemeral hostname mapping inside the Colab runtime.
    """
    hosts_path = Path("/etc/hosts")
    marker = "# Vaultify Quick Tunnel"

    original_lines = hosts_path.read_text(
        encoding="utf-8"
    ).splitlines()

    cleaned_lines = [
        line
        for line in original_lines
        if marker not in line
        and hostname not in line
    ]

    cleaned_lines.append(
        f"{ip_address} {hostname} {marker}"
    )

    hosts_path.write_text(
        "\n".join(cleaned_lines) + "\n",
        encoding="utf-8",
    )


print(
    f"🔎 Resolving public hostname: "
    f"{PUBLIC_MCP_HOSTNAME}"
)


public_ip_addresses = []


for attempt_number in range(
    1,
    DNS_RETRY_ATTEMPTS + 1,
):
    public_ip_addresses = (
        resolve_with_public_dns(
            PUBLIC_MCP_HOSTNAME
        )
    )

    if public_ip_addresses:
        print(
            f"✅ Public DNS resolved on attempt "
            f"{attempt_number}."
        )
        break

    print(
        f"⏳ DNS attempt "
        f"{attempt_number}/"
        f"{DNS_RETRY_ATTEMPTS}: "
        "record is not visible yet."
    )

    if attempt_number < DNS_RETRY_ATTEMPTS:
        await asyncio.sleep(
            DNS_RETRY_SECONDS
        )


if not public_ip_addresses:
    raise RuntimeError(
        "The TryCloudflare hostname has not appeared "
        "in public DNS yet. The MCP server and tunnel "
        "are running, but Cloudflare did not publish "
        "a resolvable hostname for this Quick Tunnel."
    )


print(
    f"🌐 Public IP addresses: "
    f"{public_ip_addresses}"
)


try:
    system_dns_result = socket.getaddrinfo(
        PUBLIC_MCP_HOSTNAME,
        443,
    )

    print(
        "✅ Colab system DNS can now resolve "
        "the Quick Tunnel hostname."
    )

except socket.gaierror:
    selected_ip_address = (
        public_ip_addresses[0]
    )

    add_temporary_hosts_entry(
        hostname=PUBLIC_MCP_HOSTNAME,
        ip_address=selected_ip_address,
    )

    print(
        "🛠️ Colab system DNS still failed, "
        "so a temporary /etc/hosts entry was added."
    )
    print(
        f"   {PUBLIC_MCP_HOSTNAME} "
        f"→ {selected_ip_address}"
    )


verified_addresses = socket.getaddrinfo(
    PUBLIC_MCP_HOSTNAME,
    443,
)

print(
    f"✅ Colab hostname resolution verified: "
    f"{len(verified_addresses)} result(s)"
)


try:
    public_test_result = (
        await call_public_mcp_once()
    )

except BaseException as error:
    print(
        "\n❌ Public MCP request failed "
        "after DNS resolution:"
    )

    for error_line in format_exception_tree(
        error
    ):
        print(
            f"   {error_line}"
        )

    raise


public_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        public_test_result["tool_result"]
    )
)


print("\n📊 Public MCP test")
print(
    f"   Server: "
    f"{public_test_result['server_name']} "
    f"{public_test_result['server_version']}"
)
print(
    f"   Tools: "
    f"{public_test_result['tool_names']}"
)
print(
    f"   Tenant: "
    f"{public_payload['tenant_id']}"
)
print(
    f"   Sources: "
    f"{len(public_payload['sources'])}"
)

print("\n🤖 Public MCP answer:")
print(
    public_payload["answer"]
)

print(
    "\n✅ Public DNS resolution succeeded."
)
print(
    "✅ Vaultify is reachable through Cloudflare."
)
print(
    "✅ MCP initialization, discovery, "
    "and tool execution succeeded publicly."
)
print(
    "✅ Tenant isolation remained active."
)
print(
    f"🔗 Claude-ready MCP endpoint: "
    f"{PUBLIC_MCP_URL}"
)

🔎 Resolving public hostname: minute-volvo-foster-fiction.trycloudflare.com
✅ Public DNS resolved on attempt 1.
🌐 Public IP addresses: ['104.16.231.132', '104.16.230.132']
✅ Colab system DNS can now resolve the Quick Tunnel hostname.
✅ Colab hostname resolution verified: 12 result(s)
INFO:     35.197.112.94:0 - "POST /mcp HTTP/1.1" 200 OK
INFO:     35.197.112.94:0 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     35.197.112.94:0 - "POST /mcp HTTP/1.1" 200 OK
INFO:     35.197.112.94:0 - "POST /mcp HTTP/1.1" 200 OK

📊 Public MCP test
   Server: Vaultify 1.28.1
   Tools: ['ask_documents']
   Tenant: demo_apple_tenant
   Sources: 5

🤖 Public MCP answer:
$416,161 million
The answer is found in the table in Section: Note 2 - Revenue of the file apple_fy2025_10k.pdf, which provides the total net sales for fiscal year 2025.

✅ Public DNS resolution succeeded.
✅ Vaultify is reachable through Cloudflare.
✅ MCP initialization, discovery, and tool execution succeeded publicly.
✅ Tenant isolation remain

In [ ]:
# PHASE 3 - Cell 1: Install authentication and dashboard dependencies

%pip install -q \
    flask-sqlalchemy \
    flask-login \
    flask-wtf \
    flask-migrate \
    email-validator \
    flask-limiter \
    filetype

from importlib.metadata import version


PHASE_3_PACKAGES = {
    "Flask-SQLAlchemy": "flask-sqlalchemy",
    "Flask-Login": "flask-login",
    "Flask-WTF": "flask-wtf",
    "Flask-Migrate": "flask-migrate",
    "Email Validator": "email-validator",
    "Flask-Limiter": "flask-limiter",
    "Filetype": "filetype",
}


print("✅ Phase 3 dependencies installed successfully.")
print("\n📦 Installed package versions:")

for display_name, package_name in PHASE_3_PACKAGES.items():
    print(
        f"   {display_name}: "
        f"{version(package_name)}"
    )


print("\n🔐 Authentication capabilities:")
print("   - Secure password hashing")
print("   - Login and logout sessions")
print("   - CSRF-protected forms")
print("   - Email validation")
print("   - Request rate limiting")

print("\n📁 Dashboard capabilities:")
print("   - SQLAlchemy database models")
print("   - Database migrations")
print("   - File type validation")
print("   - User and tenant-scoped data")

print(
    "\nℹ️ No runtime restart is required "
    "unless Colab explicitly requests it."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 18.0 MB/s eta 0:00:00
✅ Phase 3 dependencies installed successfully.

📦 Installed package versions:
   Flask-SQLAlchemy: 3.1.1
   Flask-Login: 0.6.3
   Flask-WTF: 1.3.0
   Flask-Migrate: 4.1.0
   Email Validator: 2.3.0
   Flask-Limiter: 4.1.1
   Filetype: 1.2.0

🔐 Authentication capabilities:
   - Secure password hashing
   - Login and logout sessions
   - CSRF-protected forms
   - Email validation
   - Request rate limiting

📁 Dashboard capabilities:
   - SQLAlchemy database models
   - Database migrations
   - File type validation
   - User and tenant-scoped data

ℹ️ No runtime restart is required unless Colab explicitly requests it.


In [ ]:
# PHASE 3 - Cell 2: Create the Flask application structure and base configuration

import importlib
import sys
import textwrap
from pathlib import Path


PROJECT_ROOT = Path("/content/vaultify_app")

PROJECT_DIRECTORIES = [
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "app" / "auth",
    PROJECT_ROOT / "app" / "dashboard",
    PROJECT_ROOT / "app" / "documents",
    PROJECT_ROOT / "app" / "chat",
    PROJECT_ROOT / "app" / "mcp",
    PROJECT_ROOT / "app" / "templates",
    PROJECT_ROOT / "app" / "templates" / "auth",
    PROJECT_ROOT / "app" / "templates" / "dashboard",
    PROJECT_ROOT / "app" / "templates" / "documents",
    PROJECT_ROOT / "app" / "templates" / "chat",
    PROJECT_ROOT / "app" / "static",
    PROJECT_ROOT / "app" / "static" / "css",
    PROJECT_ROOT / "app" / "static" / "js",
    PROJECT_ROOT / "instance",
    PROJECT_ROOT / "uploads",
]


for directory in PROJECT_DIRECTORIES:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


PROJECT_FILES = {
    "app/config.py": """
        import os
        import secrets
        from pathlib import Path


        BASE_DIR = Path(__file__).resolve().parent.parent


        class Config:
            \"\"\"Base configuration for the Vaultify web application.\"\"\"

            SECRET_KEY = (
                os.environ.get("VAULTIFY_SECRET_KEY")
                or secrets.token_hex(32)
            )

            SQLALCHEMY_DATABASE_URI = (
                os.environ.get("DATABASE_URL")
                or f"sqlite:///{BASE_DIR / 'instance' / 'vaultify.db'}"
            )

            SQLALCHEMY_TRACK_MODIFICATIONS = False

            UPLOAD_FOLDER = str(
                BASE_DIR / "uploads"
            )

            MAX_CONTENT_LENGTH = (
                25 * 1024 * 1024
            )

            WTF_CSRF_ENABLED = True
            WTF_CSRF_TIME_LIMIT = 3600

            SESSION_COOKIE_HTTPONLY = True
            SESSION_COOKIE_SAMESITE = "Lax"

            # HTTPS is not enabled in the local Colab development app.
            # This must be True in production.
            SESSION_COOKIE_SECURE = False

            REMEMBER_COOKIE_HTTPONLY = True
            REMEMBER_COOKIE_SAMESITE = "Lax"
            REMEMBER_COOKIE_SECURE = False
    """,

    "app/extensions.py": """
        from flask_limiter import Limiter
        from flask_limiter.util import get_remote_address
        from flask_login import LoginManager
        from flask_migrate import Migrate
        from flask_sqlalchemy import SQLAlchemy
        from flask_wtf import CSRFProtect


        db = SQLAlchemy()

        login_manager = LoginManager()
        login_manager.login_view = "auth.login"
        login_manager.login_message = (
            "Please log in to access this page."
        )
        login_manager.login_message_category = "warning"

        csrf = CSRFProtect()

        migrate = Migrate()

        limiter = Limiter(
            key_func=get_remote_address,
            storage_uri="memory://",
        )
    """,

    "app/__init__.py": """
        from pathlib import Path

        from flask import Flask, jsonify

        from .config import Config
        from .extensions import (
            csrf,
            db,
            limiter,
            login_manager,
            migrate,
        )


        def create_app(config_class=Config):
            \"\"\"Create and configure the Vaultify Flask application.\"\"\"

            app = Flask(__name__)

            app.config.from_object(
                config_class
            )

            Path(
                app.config["UPLOAD_FOLDER"]
            ).mkdir(
                parents=True,
                exist_ok=True,
            )

            database_directory = (
                Path(app.root_path).parent
                / "instance"
            )

            database_directory.mkdir(
                parents=True,
                exist_ok=True,
            )

            db.init_app(app)
            login_manager.init_app(app)
            csrf.init_app(app)
            migrate.init_app(app, db)
            limiter.init_app(app)

            @app.get("/health")
            @limiter.exempt
            def health():
                \"\"\"Return the current web application status.\"\"\"

                return jsonify(
                    {
                        "status": "ok",
                        "service": "Vaultify",
                        "phase": 3,
                    }
                ), 200

            return app
    """,

    "run.py": """
        from app import create_app


        app = create_app()


        if __name__ == "__main__":
            app.run(
                host="127.0.0.1",
                port=5000,
                debug=True,
            )
    """,

    "app/auth/__init__.py": """
        \"\"\"Vaultify authentication package.\"\"\"
    """,

    "app/dashboard/__init__.py": """
        \"\"\"Vaultify dashboard package.\"\"\"
    """,

    "app/documents/__init__.py": """
        \"\"\"Vaultify document management package.\"\"\"
    """,

    "app/chat/__init__.py": """
        \"\"\"Vaultify question and answer package.\"\"\"
    """,

    "app/mcp/__init__.py": """
        \"\"\"Vaultify MCP integration package.\"\"\"
    """,

    "app/static/css/.gitkeep": "",
    "app/static/js/.gitkeep": "",
    "uploads/.gitkeep": "",
    "instance/.gitkeep": "",
}


for relative_path, file_content in PROJECT_FILES.items():
    target_path = (
        PROJECT_ROOT / relative_path
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    cleaned_content = textwrap.dedent(
        file_content
    ).lstrip()

    target_path.write_text(
        cleaned_content,
        encoding="utf-8",
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


importlib.invalidate_caches()


# Remove stale imports when this cell is executed again.
for module_name in list(sys.modules):
    if (
        module_name == "app"
        or module_name.startswith("app.")
    ):
        del sys.modules[module_name]


from app import create_app


vaultify_web_app = create_app()

test_client = (
    vaultify_web_app.test_client()
)

health_response = test_client.get(
    "/health"
)

health_payload = (
    health_response.get_json()
)


if health_response.status_code != 200:
    raise RuntimeError(
        "The Vaultify health endpoint failed. "
        f"Status code: {health_response.status_code}"
    )


if health_payload.get("status") != "ok":
    raise RuntimeError(
        "The Vaultify health endpoint returned "
        "an unexpected response."
    )


print("✅ Vaultify Flask application structure created.")
print(f"📁 Project root: {PROJECT_ROOT}")

print("\n📂 Created structure:")

for path in sorted(
    PROJECT_ROOT.rglob("*")
):
    relative_path = path.relative_to(
        PROJECT_ROOT
    )

    if "__pycache__" in relative_path.parts:
        continue

    path_type = (
        "DIR "
        if path.is_dir()
        else "FILE"
    )

    print(
        f"   [{path_type}] "
        f"{relative_path}"
    )


print("\n🩺 Health endpoint test:")
print(
    f"   Status code: "
    f"{health_response.status_code}"
)
print(
    f"   Response: "
    f"{health_payload}"
)

print("\n🔐 Base security configuration:")
print("   - CSRF protection enabled")
print("   - HTTP-only session cookies enabled")
print("   - SameSite cookie policy enabled")
print("   - 25 MB upload limit configured")
print("   - Database path configured")
print("   - Login manager initialized")
print("   - Rate limiter initialized")

print(
    "\n✅ Phase 3.1 application foundation "
    "is working correctly."
)
print(
    "ℹ️ The Flask server was not opened publicly; "
    "the app was tested safely in memory."
)

✅ Vaultify Flask application structure created.
📁 Project root: /content/vaultify_app

📂 Created structure:
   [DIR ] app
   [FILE] app/__init__.py
   [DIR ] app/auth
   [FILE] app/auth/__init__.py
   [DIR ] app/chat
   [FILE] app/chat/__init__.py
   [FILE] app/config.py
   [DIR ] app/dashboard
   [FILE] app/dashboard/__init__.py
   [DIR ] app/documents
   [FILE] app/documents/__init__.py
   [FILE] app/extensions.py
   [DIR ] app/mcp
   [FILE] app/mcp/__init__.py
   [DIR ] app/static
   [DIR ] app/static/css
   [FILE] app/static/css/.gitkeep
   [DIR ] app/static/js
   [FILE] app/static/js/.gitkeep
   [DIR ] app/templates
   [DIR ] app/templates/auth
   [DIR ] app/templates/chat
   [DIR ] app/templates/dashboard
   [DIR ] app/templates/documents
   [DIR ] instance
   [FILE] instance/.gitkeep
   [FILE] run.py
   [DIR ] uploads
   [FILE] uploads/.gitkeep

🩺 Health endpoint test:
   Status code: 200
   Response: {'phase': 3, 'service': 'Vaultify', 'status': 'ok'}

🔐 Base security configura

In [ ]:
# PHASE 3 - Cell 3: Create the database models and verify relationships

import importlib
import sys
import textwrap
from pathlib import Path


PROJECT_ROOT = Path("/content/vaultify_app")


MODELS_FILE_CONTENT = """
    import secrets
    from datetime import datetime, timezone

    from flask_login import UserMixin
    from werkzeug.security import (
        check_password_hash,
        generate_password_hash,
    )

    from .extensions import db, login_manager


    MEMBERSHIP_ROLES = (
        "owner",
        "admin",
        "member",
        "viewer",
    )

    DOCUMENT_STATUSES = (
        "uploaded",
        "parsing",
        "indexing",
        "ready",
        "failed",
        "deleting",
    )

    QUERY_STATUSES = (
        "success",
        "refused",
        "error",
    )


    def utc_now():
        \"\"\"Return the current UTC datetime.\"\"\"
        return datetime.now(timezone.utc)


    def generate_tenant_id():
        \"\"\"Generate a stable public-safe tenant identifier.\"\"\"
        return f"tenant_{secrets.token_hex(12)}"


    class TimestampMixin:
        \"\"\"Add created and updated timestamps to a model.\"\"\"

        created_at = db.Column(
            db.DateTime(timezone=True),
            nullable=False,
            default=utc_now,
        )

        updated_at = db.Column(
            db.DateTime(timezone=True),
            nullable=False,
            default=utc_now,
            onupdate=utc_now,
        )


    class User(
        UserMixin,
        TimestampMixin,
        db.Model,
    ):
        \"\"\"Represent a Vaultify user account.\"\"\"

        __tablename__ = "users"

        id = db.Column(
            db.Integer,
            primary_key=True,
        )

        email = db.Column(
            db.String(320),
            nullable=False,
            unique=True,
            index=True,
        )

        password_hash = db.Column(
            db.String(255),
            nullable=False,
        )

        display_name = db.Column(
            db.String(120),
            nullable=False,
        )

        is_active = db.Column(
            db.Boolean,
            nullable=False,
            default=True,
        )

        is_verified = db.Column(
            db.Boolean,
            nullable=False,
            default=False,
        )

        last_login_at = db.Column(
            db.DateTime(timezone=True),
            nullable=True,
        )

        memberships = db.relationship(
            "Membership",
            back_populates="user",
            cascade="all, delete-orphan",
            passive_deletes=True,
        )

        uploaded_documents = db.relationship(
            "Document",
            back_populates="uploaded_by",
            foreign_keys="Document.uploaded_by_user_id",
        )

        query_logs = db.relationship(
            "QueryLog",
            back_populates="user",
        )

        def set_password(self, password):
            \"\"\"Hash and store a plaintext password.\"\"\"
            self.password_hash = generate_password_hash(
                password
            )

        def check_password(self, password):
            \"\"\"Verify a plaintext password against the stored hash.\"\"\"
            return check_password_hash(
                self.password_hash,
                password,
            )

        def normalize_email(self):
            \"\"\"Normalize the user email before persistence.\"\"\"
            self.email = self.email.strip().lower()

        def __repr__(self):
            return (
                f"<User id={self.id} "
                f"email={self.email!r}>"
            )


    class Organization(
        TimestampMixin,
        db.Model,
    ):
        \"\"\"Represent a tenant organization.\"\"\"

        __tablename__ = "organizations"

        id = db.Column(
            db.Integer,
            primary_key=True,
        )

        tenant_id = db.Column(
            db.String(64),
            nullable=False,
            unique=True,
            index=True,
            default=generate_tenant_id,
        )

        name = db.Column(
            db.String(160),
            nullable=False,
        )

        slug = db.Column(
            db.String(160),
            nullable=False,
            unique=True,
            index=True,
        )

        is_active = db.Column(
            db.Boolean,
            nullable=False,
            default=True,
        )

        memberships = db.relationship(
            "Membership",
            back_populates="organization",
            cascade="all, delete-orphan",
            passive_deletes=True,
        )

        documents = db.relationship(
            "Document",
            back_populates="organization",
            cascade="all, delete-orphan",
            passive_deletes=True,
        )

        query_logs = db.relationship(
            "QueryLog",
            back_populates="organization",
            cascade="all, delete-orphan",
            passive_deletes=True,
        )

        connector_credentials = db.relationship(
            "ConnectorCredential",
            back_populates="organization",
            cascade="all, delete-orphan",
            passive_deletes=True,
        )

        def __repr__(self):
            return (
                f"<Organization id={self.id} "
                f"tenant_id={self.tenant_id!r}>"
            )


    class Membership(
        TimestampMixin,
        db.Model,
    ):
        \"\"\"Connect a user to an organization with a role.\"\"\"

        __tablename__ = "memberships"

        __table_args__ = (
            db.UniqueConstraint(
                "user_id",
                "organization_id",
                name="uq_membership_user_organization",
            ),
            db.CheckConstraint(
                "role IN "
                "('owner', 'admin', 'member', 'viewer')",
                name="ck_membership_valid_role",
            ),
        )

        id = db.Column(
            db.Integer,
            primary_key=True,
        )

        user_id = db.Column(
            db.Integer,
            db.ForeignKey(
                "users.id",
                ondelete="CASCADE",
            ),
            nullable=False,
            index=True,
        )

        organization_id = db.Column(
            db.Integer,
            db.ForeignKey(
                "organizations.id",
                ondelete="CASCADE",
            ),
            nullable=False,
            index=True,
        )

        role = db.Column(
            db.String(20),
            nullable=False,
            default="member",
        )

        user = db.relationship(
            "User",
            back_populates="memberships",
        )

        organization = db.relationship(
            "Organization",
            back_populates="memberships",
        )

        def __repr__(self):
            return (
                f"<Membership user_id={self.user_id} "
                f"organization_id={self.organization_id} "
                f"role={self.role!r}>"
            )


    class Document(
        TimestampMixin,
        db.Model,
    ):
        \"\"\"Represent a tenant-owned uploaded document.\"\"\"

        __tablename__ = "documents"

        __table_args__ = (
            db.UniqueConstraint(
                "organization_id",
                "document_hash",
                name="uq_document_organization_hash",
            ),
            db.CheckConstraint(
                "status IN "
                "('uploaded', 'parsing', 'indexing', "
                "'ready', 'failed', 'deleting')",
                name="ck_document_valid_status",
            ),
        )

        id = db.Column(
            db.Integer,
            primary_key=True,
        )

        organization_id = db.Column(
            db.Integer,
            db.ForeignKey(
                "organizations.id",
                ondelete="CASCADE",
            ),
            nullable=False,
            index=True,
        )

        filename = db.Column(
            db.String(255),
            nullable=False,
        )

        original_filename = db.Column(
            db.String(255),
            nullable=False,
        )

        stored_path = db.Column(
            db.String(1000),
            nullable=False,
        )

        mime_type = db.Column(
            db.String(120),
            nullable=False,
        )

        size_bytes = db.Column(
            db.BigInteger,
            nullable=False,
        )

        document_hash = db.Column(
            db.String(64),
            nullable=False,
            index=True,
        )

        status = db.Column(
            db.String(20),
            nullable=False,
            default="uploaded",
            index=True,
        )

        chunk_count = db.Column(
            db.Integer,
            nullable=False,
            default=0,
        )

        error_message = db.Column(
            db.Text,
            nullable=True,
        )

        uploaded_by_user_id = db.Column(
            db.Integer,
            db.ForeignKey(
                "users.id",
                ondelete="RESTRICT",
            ),
            nullable=False,
            index=True,
        )

        indexed_at = db.Column(
            db.DateTime(timezone=True),
            nullable=True,
        )

        organization = db.relationship(
            "Organization",
            back_populates="documents",
        )

        uploaded_by = db.relationship(
            "User",
            back_populates="uploaded_documents",
            foreign_keys=[uploaded_by_user_id],
        )

        def __repr__(self):
            return (
                f"<Document id={self.id} "
                f"filename={self.filename!r} "
                f"status={self.status!r}>"
            )


    class QueryLog(
        TimestampMixin,
        db.Model,
    ):
        \"\"\"Record a tenant-scoped Vaultify question and result.\"\"\"

        __tablename__ = "query_logs"

        __table_args__ = (
            db.CheckConstraint(
                "status IN ('success', 'refused', 'error')",
                name="ck_query_log_valid_status",
            ),
        )

        id = db.Column(
            db.Integer,
            primary_key=True,
        )

        organization_id = db.Column(
            db.Integer,
            db.ForeignKey(
                "organizations.id",
                ondelete="CASCADE",
            ),
            nullable=False,
            index=True,
        )

        user_id = db.Column(
            db.Integer,
            db.ForeignKey(
                "users.id",
                ondelete="SET NULL",
            ),
            nullable=True,
            index=True,
        )

        question = db.Column(
            db.Text,
            nullable=False,
        )

        answer = db.Column(
            db.Text,
            nullable=True,
        )

        source_count = db.Column(
            db.Integer,
            nullable=False,
            default=0,
        )

        latency_ms = db.Column(
            db.Integer,
            nullable=True,
        )

        status = db.Column(
            db.String(20),
            nullable=False,
            default="success",
            index=True,
        )

        organization = db.relationship(
            "Organization",
            back_populates="query_logs",
        )

        user = db.relationship(
            "User",
            back_populates="query_logs",
        )

        def __repr__(self):
            return (
                f"<QueryLog id={self.id} "
                f"status={self.status!r}>"
            )


    class ConnectorCredential(
        TimestampMixin,
        db.Model,
    ):
        \"\"\"Store a hashed organization-scoped MCP credential.\"\"\"

        __tablename__ = "connector_credentials"

        id = db.Column(
            db.Integer,
            primary_key=True,
        )

        organization_id = db.Column(
            db.Integer,
            db.ForeignKey(
                "organizations.id",
                ondelete="CASCADE",
            ),
            nullable=False,
            index=True,
        )

        name = db.Column(
            db.String(120),
            nullable=False,
        )

        token_hash = db.Column(
            db.String(64),
            nullable=False,
            unique=True,
            index=True,
        )

        token_prefix = db.Column(
            db.String(24),
            nullable=False,
            index=True,
        )

        is_active = db.Column(
            db.Boolean,
            nullable=False,
            default=True,
        )

        last_used_at = db.Column(
            db.DateTime(timezone=True),
            nullable=True,
        )

        revoked_at = db.Column(
            db.DateTime(timezone=True),
            nullable=True,
        )

        organization = db.relationship(
            "Organization",
            back_populates="connector_credentials",
        )

        def __repr__(self):
            return (
                f"<ConnectorCredential id={self.id} "
                f"prefix={self.token_prefix!r}>"
            )


    @login_manager.user_loader
    def load_user(user_id):
        \"\"\"Load a user from the authenticated session identifier.\"\"\"
        try:
            parsed_user_id = int(user_id)
        except (TypeError, ValueError):
            return None

        return db.session.get(
            User,
            parsed_user_id,
        )
"""


APP_INIT_FILE_CONTENT = """
    from pathlib import Path

    from flask import Flask, jsonify

    from .config import Config
    from .extensions import (
        csrf,
        db,
        limiter,
        login_manager,
        migrate,
    )


    def create_app(config_class=Config):
        \"\"\"Create and configure the Vaultify Flask application.\"\"\"

        app = Flask(__name__)

        app.config.from_object(
            config_class
        )

        Path(
            app.config["UPLOAD_FOLDER"]
        ).mkdir(
            parents=True,
            exist_ok=True,
        )

        database_directory = (
            Path(app.root_path).parent
            / "instance"
        )

        database_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        db.init_app(app)
        login_manager.init_app(app)
        csrf.init_app(app)
        migrate.init_app(app, db)
        limiter.init_app(app)

        # Import models before database initialization.
        from . import models

        @app.get("/health")
        @limiter.exempt
        def health():
            \"\"\"Return the current web application status.\"\"\"

            return jsonify(
                {
                    "status": "ok",
                    "service": "Vaultify",
                    "phase": 3,
                }
            ), 200

        return app
"""


models_path = (
    PROJECT_ROOT / "app" / "models.py"
)

app_init_path = (
    PROJECT_ROOT / "app" / "__init__.py"
)


models_path.write_text(
    textwrap.dedent(
        MODELS_FILE_CONTENT
    ).lstrip(),
    encoding="utf-8",
)

app_init_path.write_text(
    textwrap.dedent(
        APP_INIT_FILE_CONTENT
    ).lstrip(),
    encoding="utf-8",
)


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


importlib.invalidate_caches()


# Remove stale imports so Colab loads the new model definitions.
for module_name in list(sys.modules):
    if (
        module_name == "app"
        or module_name.startswith("app.")
    ):
        del sys.modules[module_name]


from sqlalchemy import inspect

from app import create_app
from app.extensions import db
from app.models import (
    ConnectorCredential,
    Document,
    Membership,
    Organization,
    QueryLog,
    User,
)


vaultify_web_app = create_app()


with vaultify_web_app.app_context():
    db.create_all()

    database_inspector = inspect(
        db.engine
    )

    created_tables = sorted(
        database_inspector.get_table_names()
    )

    required_tables = {
        "users",
        "organizations",
        "memberships",
        "documents",
        "query_logs",
        "connector_credentials",
    }

    missing_tables = (
        required_tables
        - set(created_tables)
    )

    if missing_tables:
        raise RuntimeError(
            "Required database tables are missing: "
            f"{sorted(missing_tables)}"
        )

    # Create temporary records to validate model behavior.
    test_user = User(
        email="phase3-model-test@example.com",
        display_name="Phase 3 Test User",
    )

    test_user.normalize_email()
    test_user.set_password(
        "Temporary-Test-Password-123!"
    )

    test_organization = Organization(
        name="Phase 3 Test Organization",
        slug="phase-3-model-test",
    )

    test_membership = Membership(
        user=test_user,
        organization=test_organization,
        role="owner",
    )

    db.session.add_all(
        [
            test_user,
            test_organization,
            test_membership,
        ]
    )

    db.session.flush()

    if not test_user.check_password(
        "Temporary-Test-Password-123!"
    ):
        raise RuntimeError(
            "Password hashing verification failed."
        )

    if test_user.check_password(
        "Incorrect-Password"
    ):
        raise RuntimeError(
            "An incorrect password was accepted."
        )

    if (
        test_membership.organization.tenant_id
        != test_organization.tenant_id
    ):
        raise RuntimeError(
            "Membership organization relationship failed."
        )

    if test_membership.user.id != test_user.id:
        raise RuntimeError(
            "Membership user relationship failed."
        )

    loaded_user = db.session.get(
        User,
        test_user.id,
    )

    if loaded_user is None:
        raise RuntimeError(
            "The test user could not be loaded."
        )

    generated_tenant_id = (
        test_organization.tenant_id
    )

    generated_password_hash = (
        test_user.password_hash
    )

    temporary_user_id = test_user.id
    temporary_organization_id = (
        test_organization.id
    )

    # Roll back so temporary verification data is not persisted.
    db.session.rollback()

    persisted_test_user = (
        db.session.scalar(
            db.select(User).where(
                User.email
                == "phase3-model-test@example.com"
            )
        )
    )

    if persisted_test_user is not None:
        raise RuntimeError(
            "Temporary test data was unexpectedly persisted."
        )


print("✅ Vaultify database models created successfully.")
print(
    f"🗄️ Database: "
    f"{vaultify_web_app.config['SQLALCHEMY_DATABASE_URI']}"
)

print("\n📋 Created database tables:")

for table_name in created_tables:
    print(
        f"   - {table_name}"
    )


print("\n🔗 Verified relationships:")
print(
    f"   User ID generated: "
    f"{temporary_user_id}"
)
print(
    f"   Organization ID generated: "
    f"{temporary_organization_id}"
)
print(
    f"   Tenant ID generated: "
    f"{generated_tenant_id}"
)
print(
    "   User ↔ Membership: working"
)
print(
    "   Organization ↔ Membership: working"
)

print("\n🔐 Verified security behavior:")
print(
    "   Password stored as a hash: "
    f"{generated_password_hash != 'Temporary-Test-Password-123!'}"
)
print(
    "   Correct password accepted: True"
)
print(
    "   Incorrect password rejected: True"
)
print(
    "   Temporary test data persisted: False"
)

print("\n🏢 Tenant model rules:")
print(
    "   - Every organization receives "
    "a unique tenant ID"
)
print(
    "   - Users connect to organizations "
    "through memberships"
)
print(
    "   - Documents belong to organizations"
)
print(
    "   - Query logs belong to organizations"
)
print(
    "   - MCP credentials belong to organizations"
)

print(
    "\n✅ Phase 3.2 database foundation "
    "is working correctly."
)

✅ Vaultify database models created successfully.
🗄️ Database: sqlite:////content/vaultify_app/instance/vaultify.db

📋 Created database tables:
   - connector_credentials
   - documents
   - memberships
   - organizations
   - query_logs
   - users

🔗 Verified relationships:
   User ID generated: 1
   Organization ID generated: 1
   Tenant ID generated: tenant_7fe8d28e1262ef81cfdb7c66
   User ↔ Membership: working
   Organization ↔ Membership: working

🔐 Verified security behavior:
   Password stored as a hash: True
   Correct password accepted: True
   Incorrect password rejected: True
   Temporary test data persisted: False

🏢 Tenant model rules:
   - Every organization receives a unique tenant ID
   - Users connect to organizations through memberships
   - Documents belong to organizations
   - Query logs belong to organizations
   - MCP credentials belong to organizations

✅ Phase 3.2 database foundation is working correctly.


In [ ]:
# PHASE 3 - Cell 4: Build and test registration, login, logout, and the first dashboard

import importlib
import sys
import textwrap
from pathlib import Path


PROJECT_ROOT = Path("/content/vaultify_app")


AUTH_FORMS_CONTENT = """
    import re

    from flask_wtf import FlaskForm
    from wtforms import (
        BooleanField,
        PasswordField,
        StringField,
        SubmitField,
    )
    from wtforms.validators import (
        DataRequired,
        Email,
        EqualTo,
        Length,
        ValidationError,
    )

    from app.extensions import db
    from app.models import User


    class RegistrationForm(FlaskForm):
        \"\"\"Collect and validate new Vaultify account details.\"\"\"

        display_name = StringField(
            "Full name",
            validators=[
                DataRequired(),
                Length(min=2, max=120),
            ],
        )

        organization_name = StringField(
            "Organization name",
            validators=[
                DataRequired(),
                Length(min=2, max=160),
            ],
        )

        email = StringField(
            "Email address",
            validators=[
                DataRequired(),
                Email(),
                Length(max=320),
            ],
        )

        password = PasswordField(
            "Password",
            validators=[
                DataRequired(),
                Length(min=10, max=128),
            ],
        )

        confirm_password = PasswordField(
            "Confirm password",
            validators=[
                DataRequired(),
                EqualTo(
                    "password",
                    message="Passwords must match.",
                ),
            ],
        )

        submit = SubmitField(
            "Create account"
        )

        def validate_email(self, field):
            \"\"\"Reject email addresses that already exist.\"\"\"
            normalized_email = (
                field.data.strip().lower()
            )

            existing_user = db.session.scalar(
                db.select(User).where(
                    User.email == normalized_email
                )
            )

            if existing_user is not None:
                raise ValidationError(
                    "An account with this email already exists."
                )

        def validate_password(self, field):
            \"\"\"Enforce the Vaultify password policy.\"\"\"
            password = field.data or ""

            required_patterns = [
                (
                    r"[a-z]",
                    "Password must contain a lowercase letter.",
                ),
                (
                    r"[A-Z]",
                    "Password must contain an uppercase letter.",
                ),
                (
                    r"[0-9]",
                    "Password must contain a number.",
                ),
                (
                    r"[^A-Za-z0-9]",
                    "Password must contain a special character.",
                ),
            ]

            for pattern, error_message in required_patterns:
                if not re.search(
                    pattern,
                    password,
                ):
                    raise ValidationError(
                        error_message
                    )


    class LoginForm(FlaskForm):
        \"\"\"Collect Vaultify login credentials.\"\"\"

        email = StringField(
            "Email address",
            validators=[
                DataRequired(),
                Email(),
                Length(max=320),
            ],
        )

        password = PasswordField(
            "Password",
            validators=[
                DataRequired(),
                Length(max=128),
            ],
        )

        remember = BooleanField(
            "Keep me signed in"
        )

        submit = SubmitField(
            "Sign in"
        )
"""


AUTH_ROUTES_CONTENT = """
    import re
    import secrets
    import unicodedata
    from urllib.parse import urlsplit

    from flask import (
        Blueprint,
        flash,
        redirect,
        render_template,
        request,
        session,
        url_for,
    )
    from flask_login import (
        current_user,
        login_user,
        logout_user,
    )
    from sqlalchemy.exc import IntegrityError

    from app.extensions import db, limiter
    from app.models import (
        Membership,
        Organization,
        User,
        utc_now,
    )

    from .forms import (
        LoginForm,
        RegistrationForm,
    )


    auth_bp = Blueprint(
        "auth",
        __name__,
    )


    def create_slug(value):
        \"\"\"Convert an organization name into a URL-safe slug.\"\"\"
        normalized_value = unicodedata.normalize(
            "NFKD",
            value,
        )

        ascii_value = normalized_value.encode(
            "ascii",
            "ignore",
        ).decode("ascii")

        slug = re.sub(
            r"[^a-zA-Z0-9]+",
            "-",
            ascii_value,
        ).strip("-").lower()

        return slug or "organization"


    def create_unique_organization_slug(name):
        \"\"\"Create a unique organization slug.\"\"\"
        base_slug = create_slug(name)
        candidate_slug = base_slug

        while db.session.scalar(
            db.select(Organization).where(
                Organization.slug
                == candidate_slug
            )
        ) is not None:
            candidate_slug = (
                f"{base_slug}-"
                f"{secrets.token_hex(3)}"
            )

        return candidate_slug


    def is_safe_local_redirect(target):
        \"\"\"Allow redirects only to local application paths.\"\"\"
        if not target:
            return False

        parsed_target = urlsplit(target)

        return (
            parsed_target.scheme == ""
            and parsed_target.netloc == ""
            and target.startswith("/")
        )


    @auth_bp.route(
        "/register",
        methods=["GET", "POST"],
    )
    @limiter.limit("5 per minute")
    def register():
        \"\"\"Create a user, organization, and owner membership.\"\"\"
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard.home")
            )

        form = RegistrationForm()

        if form.validate_on_submit():
            normalized_email = (
                form.email.data.strip().lower()
            )

            user = User(
                email=normalized_email,
                display_name=(
                    form.display_name.data.strip()
                ),
            )

            user.set_password(
                form.password.data
            )

            organization = Organization(
                name=(
                    form.organization_name.data.strip()
                ),
                slug=create_unique_organization_slug(
                    form.organization_name.data
                ),
            )

            membership = Membership(
                user=user,
                organization=organization,
                role="owner",
            )

            try:
                db.session.add_all(
                    [
                        user,
                        organization,
                        membership,
                    ]
                )

                db.session.commit()

            except IntegrityError:
                db.session.rollback()

                flash(
                    "The account could not be created. "
                    "Please check your details and try again.",
                    "danger",
                )

                return render_template(
                    "auth/register.html",
                    form=form,
                )

            login_user(user)

            session.clear()
            session["organization_id"] = (
                organization.id
            )

            flash(
                "Your Vaultify account was created successfully.",
                "success",
            )

            return redirect(
                url_for("dashboard.home")
            )

        return render_template(
            "auth/register.html",
            form=form,
        )


    @auth_bp.route(
        "/login",
        methods=["GET", "POST"],
    )
    @limiter.limit("5 per minute")
    def login():
        \"\"\"Authenticate a Vaultify user.\"\"\"
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard.home")
            )

        form = LoginForm()

        if form.validate_on_submit():
            normalized_email = (
                form.email.data.strip().lower()
            )

            user = db.session.scalar(
                db.select(User).where(
                    User.email
                    == normalized_email
                )
            )

            credentials_are_valid = (
                user is not None
                and user.is_active
                and user.check_password(
                    form.password.data
                )
            )

            if not credentials_are_valid:
                flash(
                    "Invalid email address or password.",
                    "danger",
                )

                return render_template(
                    "auth/login.html",
                    form=form,
                )

            first_membership = db.session.scalar(
                db.select(Membership)
                .where(
                    Membership.user_id
                    == user.id
                )
                .order_by(
                    Membership.id.asc()
                )
            )

            if first_membership is None:
                flash(
                    "Your account is not connected "
                    "to an organization.",
                    "danger",
                )

                return render_template(
                    "auth/login.html",
                    form=form,
                )

            login_user(
                user,
                remember=form.remember.data,
            )

            session.clear()
            session["organization_id"] = (
                first_membership.organization_id
            )

            user.last_login_at = utc_now()
            db.session.commit()

            requested_next_page = request.args.get(
                "next"
            )

            if is_safe_local_redirect(
                requested_next_page
            ):
                destination = requested_next_page
            else:
                destination = url_for(
                    "dashboard.home"
                )

            flash(
                "Welcome back to Vaultify.",
                "success",
            )

            return redirect(destination)

        return render_template(
            "auth/login.html",
            form=form,
        )


    @auth_bp.post("/logout")
    @limiter.limit("20 per minute")
    def logout():
        \"\"\"End the authenticated Vaultify session.\"\"\"
        if current_user.is_authenticated:
            logout_user()

        session.clear()

        flash(
            "You have been signed out.",
            "success",
        )

        return redirect(
            url_for("auth.login")
        )
"""


DASHBOARD_ROUTES_CONTENT = """
    from flask import (
        Blueprint,
        abort,
        render_template,
        session,
    )
    from flask_login import (
        current_user,
        login_required,
    )

    from app.extensions import db
    from app.models import Membership


    dashboard_bp = Blueprint(
        "dashboard",
        __name__,
    )


    @dashboard_bp.get("/dashboard")
    @login_required
    def home():
        \"\"\"Render the authenticated user's first dashboard.\"\"\"
        organization_id = session.get(
            "organization_id"
        )

        membership = None

        if organization_id is not None:
            membership = db.session.scalar(
                db.select(Membership).where(
                    Membership.user_id
                    == current_user.id,
                    Membership.organization_id
                    == organization_id,
                )
            )

        if membership is None:
            membership = db.session.scalar(
                db.select(Membership)
                .where(
                    Membership.user_id
                    == current_user.id
                )
                .order_by(
                    Membership.id.asc()
                )
            )

            if membership is not None:
                session["organization_id"] = (
                    membership.organization_id
                )

        if membership is None:
            abort(403)

        return render_template(
            "dashboard/home.html",
            membership=membership,
            organization=membership.organization,
        )
"""


APP_INIT_CONTENT = """
    from pathlib import Path

    from flask import (
        Flask,
        jsonify,
        redirect,
        url_for,
    )
    from flask_login import current_user

    from .config import Config
    from .extensions import (
        csrf,
        db,
        limiter,
        login_manager,
        migrate,
    )


    def create_app(config_class=Config):
        \"\"\"Create and configure the Vaultify Flask application.\"\"\"
        app = Flask(__name__)

        app.config.from_object(
            config_class
        )

        Path(
            app.config["UPLOAD_FOLDER"]
        ).mkdir(
            parents=True,
            exist_ok=True,
        )

        database_directory = (
            Path(app.root_path).parent
            / "instance"
        )

        database_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        db.init_app(app)
        login_manager.init_app(app)
        csrf.init_app(app)
        migrate.init_app(app, db)
        limiter.init_app(app)

        from . import models
        from .auth.routes import auth_bp
        from .dashboard.routes import dashboard_bp

        app.register_blueprint(auth_bp)
        app.register_blueprint(dashboard_bp)

        @app.get("/")
        def index():
            \"\"\"Send users to the appropriate Vaultify page.\"\"\"
            if current_user.is_authenticated:
                return redirect(
                    url_for("dashboard.home")
                )

            return redirect(
                url_for("auth.login")
            )

        @app.get("/health")
        @limiter.exempt
        def health():
            \"\"\"Return the application health status.\"\"\"
            return jsonify(
                {
                    "status": "ok",
                    "service": "Vaultify",
                    "phase": 3,
                }
            ), 200

        return app
"""


BASE_TEMPLATE_CONTENT = """
    <!doctype html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <meta
            name="viewport"
            content="width=device-width, initial-scale=1"
        >
        <title>
            {% block title %}Vaultify{% endblock %}
        </title>
        <link
            rel="stylesheet"
            href="{{ url_for('static', filename='css/app.css') }}"
        >
    </head>
    <body>
        <header class="site-header">
            <a
                class="brand"
                href="{{ url_for('index') }}"
            >
                Vaultify
            </a>

            {% if current_user.is_authenticated %}
                <nav>
                    <a href="{{ url_for('dashboard.home') }}">
                        Dashboard
                    </a>

                    <form
                        method="post"
                        action="{{ url_for('auth.logout') }}"
                        class="inline-form"
                    >
                        <input
                            type="hidden"
                            name="csrf_token"
                            value="{{ csrf_token() }}"
                        >
                        <button
                            type="submit"
                            class="nav-button"
                        >
                            Sign out
                        </button>
                    </form>
                </nav>
            {% endif %}
        </header>

        <main class="page-shell">
            {% with messages = get_flashed_messages(with_categories=true) %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ category }}">
                        {{ message }}
                    </div>
                {% endfor %}
            {% endwith %}

            {% block content %}{% endblock %}
        </main>
    </body>
    </html>
"""


REGISTER_TEMPLATE_CONTENT = """
    {% extends "base.html" %}

    {% block title %}Create account — Vaultify{% endblock %}

    {% block content %}
        <section class="auth-card">
            <p class="eyebrow">Start using Vaultify</p>
            <h1>Create your account</h1>
            <p class="muted">
                Your first secure organization will be created automatically.
            </p>

            <form method="post" novalidate>
                {{ form.hidden_tag() }}

                <div class="form-group">
                    {{ form.display_name.label }}
                    {{ form.display_name(class="input") }}

                    {% for error in form.display_name.errors %}
                        <p class="field-error">{{ error }}</p>
                    {% endfor %}
                </div>

                <div class="form-group">
                    {{ form.organization_name.label }}
                    {{ form.organization_name(class="input") }}

                    {% for error in form.organization_name.errors %}
                        <p class="field-error">{{ error }}</p>
                    {% endfor %}
                </div>

                <div class="form-group">
                    {{ form.email.label }}
                    {{ form.email(class="input") }}

                    {% for error in form.email.errors %}
                        <p class="field-error">{{ error }}</p>
                    {% endfor %}
                </div>

                <div class="form-group">
                    {{ form.password.label }}
                    {{ form.password(class="input") }}

                    {% for error in form.password.errors %}
                        <p class="field-error">{{ error }}</p>
                    {% endfor %}
                </div>

                <div class="form-group">
                    {{ form.confirm_password.label }}
                    {{ form.confirm_password(class="input") }}

                    {% for error in form.confirm_password.errors %}
                        <p class="field-error">{{ error }}</p>
                    {% endfor %}
                </div>

                {{ form.submit(class="primary-button") }}
            </form>

            <p class="auth-footer">
                Already have an account?
                <a href="{{ url_for('auth.login') }}">Sign in</a>
            </p>
        </section>
    {% endblock %}
"""


LOGIN_TEMPLATE_CONTENT = """
    {% extends "base.html" %}

    {% block title %}Sign in — Vaultify{% endblock %}

    {% block content %}
        <section class="auth-card">
            <p class="eyebrow">Welcome back</p>
            <h1>Sign in to Vaultify</h1>
            <p class="muted">
                Access your organization's private document workspace.
            </p>

            <form method="post" novalidate>
                {{ form.hidden_tag() }}

                <div class="form-group">
                    {{ form.email.label }}
                    {{ form.email(class="input") }}

                    {% for error in form.email.errors %}
                        <p class="field-error">{{ error }}</p>
                    {% endfor %}
                </div>

                <div class="form-group">
                    {{ form.password.label }}
                    {{ form.password(class="input") }}

                    {% for error in form.password.errors %}
                        <p class="field-error">{{ error }}</p>
                    {% endfor %}
                </div>

                <label class="checkbox-row">
                    {{ form.remember() }}
                    <span>{{ form.remember.label.text }}</span>
                </label>

                {{ form.submit(class="primary-button") }}
            </form>

            <p class="auth-footer">
                New to Vaultify?
                <a href="{{ url_for('auth.register') }}">
                    Create an account
                </a>
            </p>
        </section>
    {% endblock %}
"""


DASHBOARD_TEMPLATE_CONTENT = """
    {% extends "base.html" %}

    {% block title %}Dashboard — Vaultify{% endblock %}

    {% block content %}
        <section class="dashboard-hero">
            <p class="eyebrow">Private AI workspace</p>
            <h1>
                Welcome, {{ current_user.display_name }}
            </h1>
            <p class="muted">
                Your Vaultify organization and tenant context
                are now resolved from the authenticated session.
            </p>
        </section>

        <section class="dashboard-grid">
            <article class="info-card">
                <span class="card-label">Organization</span>
                <strong>{{ organization.name }}</strong>
            </article>

            <article class="info-card">
                <span class="card-label">Role</span>
                <strong>{{ membership.role|title }}</strong>
            </article>

            <article class="info-card">
                <span class="card-label">Tenant ID</span>
                <strong class="code-value">
                    {{ organization.tenant_id }}
                </strong>
            </article>

            <article class="info-card">
                <span class="card-label">Account</span>
                <strong>{{ current_user.email }}</strong>
            </article>
        </section>

        <section class="next-step-card">
            <h2>Authentication is active</h2>
            <p>
                The next step is trusted tenant resolution,
                organization switching, and role-based authorization.
            </p>
        </section>
    {% endblock %}
"""


CSS_CONTENT = """
    :root {
        color-scheme: dark;
        font-family:
            Inter,
            ui-sans-serif,
            system-ui,
            -apple-system,
            BlinkMacSystemFont,
            "Segoe UI",
            sans-serif;
        background: #080b12;
        color: #f5f7fb;
    }

    * {
        box-sizing: border-box;
    }

    body {
        margin: 0;
        min-height: 100vh;
        background:
            radial-gradient(
                circle at top left,
                rgba(82, 104, 255, 0.18),
                transparent 36%
            ),
            #080b12;
    }

    a {
        color: #9aa8ff;
        text-decoration: none;
    }

    a:hover {
        color: #c4cbff;
    }

    .site-header {
        min-height: 68px;
        padding: 0 5vw;
        display: flex;
        align-items: center;
        justify-content: space-between;
        border-bottom: 1px solid rgba(255, 255, 255, 0.08);
        background: rgba(8, 11, 18, 0.76);
        backdrop-filter: blur(18px);
    }

    .brand {
        font-size: 1.3rem;
        font-weight: 800;
        color: #ffffff;
        letter-spacing: -0.04em;
    }

    nav {
        display: flex;
        align-items: center;
        gap: 20px;
    }

    .inline-form {
        display: inline;
    }

    .nav-button {
        border: 0;
        background: transparent;
        color: #9aa8ff;
        cursor: pointer;
        font: inherit;
    }

    .page-shell {
        width: min(1100px, 92vw);
        margin: 0 auto;
        padding: 70px 0;
    }

    .auth-card {
        width: min(500px, 100%);
        margin: 0 auto;
        padding: 36px;
        border: 1px solid rgba(255, 255, 255, 0.10);
        border-radius: 24px;
        background: rgba(17, 22, 35, 0.92);
        box-shadow: 0 30px 80px rgba(0, 0, 0, 0.35);
    }

    h1,
    h2,
    p {
        margin-top: 0;
    }

    h1 {
        margin-bottom: 12px;
        font-size: clamp(2rem, 5vw, 3rem);
        letter-spacing: -0.05em;
    }

    .eyebrow {
        margin-bottom: 10px;
        color: #8fa0ff;
        font-size: 0.78rem;
        font-weight: 800;
        text-transform: uppercase;
        letter-spacing: 0.15em;
    }

    .muted {
        color: #aab1c2;
        line-height: 1.65;
    }

    .form-group {
        margin: 22px 0;
    }

    .form-group label {
        display: block;
        margin-bottom: 8px;
        font-weight: 700;
    }

    .input {
        width: 100%;
        padding: 14px 16px;
        border: 1px solid rgba(255, 255, 255, 0.13);
        border-radius: 12px;
        outline: none;
        background: #0e1320;
        color: #ffffff;
        font: inherit;
    }

    .input:focus {
        border-color: #7788ff;
        box-shadow: 0 0 0 4px rgba(119, 136, 255, 0.12);
    }

    .primary-button {
        width: 100%;
        padding: 14px 18px;
        border: 0;
        border-radius: 12px;
        background: linear-gradient(135deg, #6f7cff, #8d61ff);
        color: #ffffff;
        font-weight: 800;
        cursor: pointer;
    }

    .checkbox-row {
        display: flex;
        align-items: center;
        gap: 9px;
        margin: 18px 0 22px;
        color: #c4c9d5;
    }

    .field-error {
        margin-top: 7px;
        color: #ff9b9b;
        font-size: 0.9rem;
    }

    .auth-footer {
        margin: 24px 0 0;
        text-align: center;
        color: #aab1c2;
    }

    .alert {
        width: min(700px, 100%);
        margin: 0 auto 22px;
        padding: 14px 16px;
        border-radius: 12px;
    }

    .alert-success {
        background: rgba(52, 211, 153, 0.14);
        color: #8df3cb;
    }

    .alert-danger {
        background: rgba(248, 113, 113, 0.14);
        color: #ffaaaa;
    }

    .alert-warning {
        background: rgba(251, 191, 36, 0.14);
        color: #ffd978;
    }

    .dashboard-hero {
        margin-bottom: 35px;
    }

    .dashboard-grid {
        display: grid;
        grid-template-columns:
            repeat(auto-fit, minmax(220px, 1fr));
        gap: 18px;
    }

    .info-card,
    .next-step-card {
        padding: 24px;
        border: 1px solid rgba(255, 255, 255, 0.09);
        border-radius: 18px;
        background: rgba(17, 22, 35, 0.84);
    }

    .card-label {
        display: block;
        margin-bottom: 10px;
        color: #939caf;
        font-size: 0.82rem;
        text-transform: uppercase;
        letter-spacing: 0.08em;
    }

    .code-value {
        overflow-wrap: anywhere;
        font-family: ui-monospace, monospace;
        font-size: 0.9rem;
    }

    .next-step-card {
        margin-top: 24px;
    }

    @media (max-width: 600px) {
        .auth-card {
            padding: 25px;
        }

        .site-header {
            padding: 0 4vw;
        }
    }
"""


PROJECT_FILES = {
    "app/auth/forms.py": AUTH_FORMS_CONTENT,
    "app/auth/routes.py": AUTH_ROUTES_CONTENT,
    "app/dashboard/routes.py": DASHBOARD_ROUTES_CONTENT,
    "app/__init__.py": APP_INIT_CONTENT,
    "app/templates/base.html": BASE_TEMPLATE_CONTENT,
    "app/templates/auth/register.html": REGISTER_TEMPLATE_CONTENT,
    "app/templates/auth/login.html": LOGIN_TEMPLATE_CONTENT,
    "app/templates/dashboard/home.html": DASHBOARD_TEMPLATE_CONTENT,
    "app/static/css/app.css": CSS_CONTENT,
}


for relative_path, content in PROJECT_FILES.items():
    target_path = (
        PROJECT_ROOT / relative_path
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    target_path.write_text(
        textwrap.dedent(content).lstrip(),
        encoding="utf-8",
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


importlib.invalidate_caches()


for module_name in list(sys.modules):
    if (
        module_name == "app"
        or module_name.startswith("app.")
    ):
        del sys.modules[module_name]


from app import create_app
from app.config import Config
from app.extensions import db
from app.models import (
    Membership,
    Organization,
    User,
)


AUTH_TEST_DATABASE_PATH = (
    PROJECT_ROOT
    / "instance"
    / "vaultify_auth_test.db"
)


if AUTH_TEST_DATABASE_PATH.exists():
    AUTH_TEST_DATABASE_PATH.unlink()


class AuthTestConfig(Config):
    """Configuration used only for automated authentication tests."""

    TESTING = True
    WTF_CSRF_ENABLED = False
    RATELIMIT_ENABLED = False
    SQLALCHEMY_DATABASE_URI = (
        f"sqlite:///{AUTH_TEST_DATABASE_PATH}"
    )


auth_test_app = create_app(
    AuthTestConfig
)


with auth_test_app.app_context():
    db.create_all()


auth_test_client = (
    auth_test_app.test_client()
)


register_page_response = (
    auth_test_client.get("/register")
)

if register_page_response.status_code != 200:
    raise RuntimeError(
        "The registration page did not load."
    )


registration_response = (
    auth_test_client.post(
        "/register",
        data={
            "display_name": "Vaultify Test User",
            "organization_name": "Vaultify Test Company",
            "email": "auth-test@example.com",
            "password": "Secure-Test-123!",
            "confirm_password": "Secure-Test-123!",
        },
        follow_redirects=True,
    )
)


if registration_response.status_code != 200:
    raise RuntimeError(
        "The registration request failed."
    )


if b"Vaultify Test Company" not in registration_response.data:
    raise RuntimeError(
        "Registration did not redirect "
        "to the authenticated dashboard."
    )


with auth_test_app.app_context():
    created_user = db.session.scalar(
        db.select(User).where(
            User.email
            == "auth-test@example.com"
        )
    )

    if created_user is None:
        raise RuntimeError(
            "The registered user was not stored."
        )

    created_membership = db.session.scalar(
        db.select(Membership).where(
            Membership.user_id
            == created_user.id
        )
    )

    if created_membership is None:
        raise RuntimeError(
            "The owner membership was not created."
        )

    created_organization = db.session.get(
        Organization,
        created_membership.organization_id,
    )

    if created_organization is None:
        raise RuntimeError(
            "The organization was not created."
        )

    if created_membership.role != "owner":
        raise RuntimeError(
            "The registered user was not made owner."
        )

    tested_tenant_id = (
        created_organization.tenant_id
    )


logout_response = (
    auth_test_client.post(
        "/logout",
        follow_redirects=True,
    )
)


if b"Sign in to Vaultify" not in logout_response.data:
    raise RuntimeError(
        "Logout did not return to the login page."
    )


protected_response = (
    auth_test_client.get(
        "/dashboard",
        follow_redirects=False,
    )
)


if protected_response.status_code not in {
    302,
    401,
}:
    raise RuntimeError(
        "The dashboard was accessible "
        "without authentication."
    )


invalid_login_response = (
    auth_test_client.post(
        "/login",
        data={
            "email": "auth-test@example.com",
            "password": "Wrong-Password-123!",
        },
        follow_redirects=True,
    )
)


if b"Invalid email address or password" not in (
    invalid_login_response.data
):
    raise RuntimeError(
        "Invalid login did not fail safely."
    )


valid_login_response = (
    auth_test_client.post(
        "/login",
        data={
            "email": "auth-test@example.com",
            "password": "Secure-Test-123!",
            "remember": False,
        },
        follow_redirects=True,
    )
)


if b"Vaultify Test Company" not in valid_login_response.data:
    raise RuntimeError(
        "Valid login did not reach the dashboard."
    )


with auth_test_app.app_context():
    db.session.remove()
    db.drop_all()
    db.engine.dispose()


if AUTH_TEST_DATABASE_PATH.exists():
    AUTH_TEST_DATABASE_PATH.unlink()


vaultify_web_app = create_app()


print("✅ Vaultify authentication frontend created.")

print("\n🧑‍💻 Authentication routes:")
print("   GET/POST /register")
print("   GET/POST /login")
print("   POST     /logout")
print("   GET      /dashboard")

print("\n🔐 Verified behavior:")
print("   Registration page loads: True")
print("   User registration succeeds: True")
print("   Organization creation succeeds: True")
print("   Owner membership creation succeeds: True")
print("   Password hashing works: True")
print("   Invalid password is rejected: True")
print("   Valid login succeeds: True")
print("   Logout succeeds: True")
print("   Anonymous dashboard access blocked: True")

print("\n🏢 Verified tenant creation:")
print(
    f"   Generated tenant ID: "
    f"{tested_tenant_id}"
)

print("\n🎨 Frontend created:")
print("   Responsive login page")
print("   Responsive registration page")
print("   Authenticated dashboard page")
print("   Flash message components")
print("   Navigation and logout form")
print("   Dark Vaultify design system")

print(
    "\n✅ Phase 3.3 authentication flow "
    "is working correctly."
)
print(
    "ℹ️ The tests used a temporary database "
    "and did not create a permanent test account."
)

RuntimeError: Registration did not redirect to the authenticated dashboard.

In [ ]:
# PHASE 3 - Cell 4A: Rewrite the corrected auth routes and retest authentication

import importlib
import sys
import textwrap
from pathlib import Path


PROJECT_ROOT = Path("/content/vaultify_app")

AUTH_ROUTES_PATH = (
    PROJECT_ROOT
    / "app"
    / "auth"
    / "routes.py"
)


CORRECTED_AUTH_ROUTES_CONTENT = """
    import re
    import secrets
    import unicodedata
    from urllib.parse import urlsplit

    from flask import (
        Blueprint,
        flash,
        redirect,
        render_template,
        request,
        session,
        url_for,
    )
    from flask_login import (
        current_user,
        login_user,
        logout_user,
    )
    from sqlalchemy.exc import IntegrityError

    from app.extensions import db, limiter
    from app.models import (
        Membership,
        Organization,
        User,
        utc_now,
    )

    from .forms import (
        LoginForm,
        RegistrationForm,
    )


    auth_bp = Blueprint(
        "auth",
        __name__,
    )


    def create_slug(value):
        \"\"\"Convert an organization name into a URL-safe slug.\"\"\"
        normalized_value = unicodedata.normalize(
            "NFKD",
            value,
        )

        ascii_value = normalized_value.encode(
            "ascii",
            "ignore",
        ).decode("ascii")

        slug = re.sub(
            r"[^a-zA-Z0-9]+",
            "-",
            ascii_value,
        ).strip("-").lower()

        return slug or "organization"


    def create_unique_organization_slug(name):
        \"\"\"Create a unique organization slug.\"\"\"
        base_slug = create_slug(name)
        candidate_slug = base_slug

        while db.session.scalar(
            db.select(Organization).where(
                Organization.slug
                == candidate_slug
            )
        ) is not None:
            candidate_slug = (
                f"{base_slug}-"
                f"{secrets.token_hex(3)}"
            )

        return candidate_slug


    def is_safe_local_redirect(target):
        \"\"\"Allow redirects only to local application paths.\"\"\"
        if not target:
            return False

        parsed_target = urlsplit(target)

        return (
            parsed_target.scheme == ""
            and parsed_target.netloc == ""
            and target.startswith("/")
        )


    @auth_bp.route(
        "/register",
        methods=["GET", "POST"],
    )
    @limiter.limit("5 per minute")
    def register():
        \"\"\"Create a user, organization, and owner membership.\"\"\"
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard.home")
            )

        form = RegistrationForm()

        if form.validate_on_submit():
            normalized_email = (
                form.email.data.strip().lower()
            )

            user = User(
                email=normalized_email,
                display_name=(
                    form.display_name.data.strip()
                ),
            )

            user.set_password(
                form.password.data
            )

            organization = Organization(
                name=(
                    form.organization_name.data.strip()
                ),
                slug=create_unique_organization_slug(
                    form.organization_name.data
                ),
            )

            membership = Membership(
                user=user,
                organization=organization,
                role="owner",
            )

            try:
                db.session.add_all(
                    [
                        user,
                        organization,
                        membership,
                    ]
                )

                db.session.commit()

            except IntegrityError:
                db.session.rollback()

                flash(
                    "The account could not be created. "
                    "Please check your details and try again.",
                    "danger",
                )

                return render_template(
                    "auth/register.html",
                    form=form,
                )

            # Clear any previous anonymous session data first.
            session.clear()

            # Then create the authenticated Flask-Login session.
            login_user(user)

            # Store only the trusted database organization ID.
            session["organization_id"] = (
                organization.id
            )

            flash(
                "Your Vaultify account was created successfully.",
                "success",
            )

            return redirect(
                url_for("dashboard.home")
            )

        return render_template(
            "auth/register.html",
            form=form,
        )


    @auth_bp.route(
        "/login",
        methods=["GET", "POST"],
    )
    @limiter.limit("5 per minute")
    def login():
        \"\"\"Authenticate a Vaultify user.\"\"\"
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard.home")
            )

        form = LoginForm()

        if form.validate_on_submit():
            normalized_email = (
                form.email.data.strip().lower()
            )

            user = db.session.scalar(
                db.select(User).where(
                    User.email
                    == normalized_email
                )
            )

            credentials_are_valid = (
                user is not None
                and user.is_active
                and user.check_password(
                    form.password.data
                )
            )

            if not credentials_are_valid:
                flash(
                    "Invalid email address or password.",
                    "danger",
                )

                return render_template(
                    "auth/login.html",
                    form=form,
                )

            first_membership = db.session.scalar(
                db.select(Membership)
                .where(
                    Membership.user_id
                    == user.id
                )
                .order_by(
                    Membership.id.asc()
                )
            )

            if first_membership is None:
                flash(
                    "Your account is not connected "
                    "to an organization.",
                    "danger",
                )

                return render_template(
                    "auth/login.html",
                    form=form,
                )

            # Clear stale session data before authentication.
            session.clear()

            # Create the authenticated user session.
            login_user(
                user,
                remember=form.remember.data,
            )

            # Resolve the organization from the trusted membership.
            session["organization_id"] = (
                first_membership.organization_id
            )

            user.last_login_at = utc_now()
            db.session.commit()

            requested_next_page = request.args.get(
                "next"
            )

            if is_safe_local_redirect(
                requested_next_page
            ):
                destination = requested_next_page

            else:
                destination = url_for(
                    "dashboard.home"
                )

            flash(
                "Welcome back to Vaultify.",
                "success",
            )

            return redirect(destination)

        return render_template(
            "auth/login.html",
            form=form,
        )


    @auth_bp.post("/logout")
    @limiter.limit("20 per minute")
    def logout():
        \"\"\"End the authenticated Vaultify session.\"\"\"
        if current_user.is_authenticated:
            logout_user()

        session.clear()

        flash(
            "You have been signed out.",
            "success",
        )

        return redirect(
            url_for("auth.login")
        )
"""


AUTH_ROUTES_PATH.write_text(
    textwrap.dedent(
        CORRECTED_AUTH_ROUTES_CONTENT
    ).lstrip(),
    encoding="utf-8",
)


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


importlib.invalidate_caches()


# Remove stale modules so Colab loads the rewritten routes.
for module_name in list(sys.modules):
    if (
        module_name == "app"
        or module_name.startswith("app.")
    ):
        del sys.modules[module_name]


from app import create_app
from app.config import Config
from app.extensions import db
from app.models import (
    Membership,
    Organization,
    User,
)


AUTH_TEST_DATABASE_PATH = (
    PROJECT_ROOT
    / "instance"
    / "vaultify_auth_regression_test.db"
)


if AUTH_TEST_DATABASE_PATH.exists():
    AUTH_TEST_DATABASE_PATH.unlink()


class AuthRegressionTestConfig(Config):
    """Configuration used only for authentication tests."""

    TESTING = True
    WTF_CSRF_ENABLED = False
    RATELIMIT_ENABLED = False

    SQLALCHEMY_DATABASE_URI = (
        f"sqlite:///"
        f"{AUTH_TEST_DATABASE_PATH}"
    )


auth_test_app = create_app(
    AuthRegressionTestConfig
)


with auth_test_app.app_context():
    db.create_all()


auth_test_client = (
    auth_test_app.test_client()
)


# Test the registration page.
register_page_response = (
    auth_test_client.get("/register")
)


if register_page_response.status_code != 200:
    raise RuntimeError(
        "The registration page did not load. "
        f"Status: {register_page_response.status_code}"
    )


# Test account registration.
registration_response = (
    auth_test_client.post(
        "/register",
        data={
            "display_name": (
                "Vaultify Test User"
            ),
            "organization_name": (
                "Vaultify Test Company"
            ),
            "email": (
                "auth-regression@example.com"
            ),
            "password": (
                "Secure-Test-123!"
            ),
            "confirm_password": (
                "Secure-Test-123!"
            ),
        },
        follow_redirects=True,
    )
)


if registration_response.status_code != 200:
    raise RuntimeError(
        "Registration returned an unexpected status. "
        f"Status: {registration_response.status_code}"
    )


registration_html = (
    registration_response.data.decode(
        "utf-8",
        errors="replace",
    )
)


if "Vaultify Test Company" not in registration_html:
    raise RuntimeError(
        "Registration did not reach the dashboard. "
        f"Response preview: "
        f"{registration_html[:2000]}"
    )


# Verify the authenticated session.
with auth_test_client.session_transaction() as session_data:
    registered_user_id = session_data.get(
        "_user_id"
    )

    registered_organization_id = (
        session_data.get(
            "organization_id"
        )
    )


if registered_user_id is None:
    raise RuntimeError(
        "The registration session contains "
        "no Flask-Login user ID."
    )


if registered_organization_id is None:
    raise RuntimeError(
        "The registration session contains "
        "no organization ID."
    )


# Verify the database records and relationships.
with auth_test_app.app_context():
    created_user = db.session.scalar(
        db.select(User).where(
            User.email
            == "auth-regression@example.com"
        )
    )

    if created_user is None:
        raise RuntimeError(
            "The registered user was not stored."
        )

    created_membership = db.session.scalar(
        db.select(Membership).where(
            Membership.user_id
            == created_user.id
        )
    )

    if created_membership is None:
        raise RuntimeError(
            "The owner membership was not stored."
        )

    created_organization = db.session.get(
        Organization,
        created_membership.organization_id,
    )

    if created_organization is None:
        raise RuntimeError(
            "The organization was not stored."
        )

    if created_membership.role != "owner":
        raise RuntimeError(
            "The registered user was not assigned "
            "the owner role."
        )

    if (
        registered_organization_id
        != created_organization.id
    ):
        raise RuntimeError(
            "The session organization does not match "
            "the user's membership."
        )

    tested_tenant_id = (
        created_organization.tenant_id
    )


# Test logout.
logout_response = (
    auth_test_client.post(
        "/logout",
        follow_redirects=True,
    )
)


if b"Sign in to Vaultify" not in (
    logout_response.data
):
    raise RuntimeError(
        "Logout did not return to the login page."
    )


# Verify that anonymous users cannot access the dashboard.
anonymous_dashboard_response = (
    auth_test_client.get(
        "/dashboard",
        follow_redirects=False,
    )
)


if anonymous_dashboard_response.status_code != 302:
    raise RuntimeError(
        "Anonymous dashboard access was not blocked. "
        f"Status: "
        f"{anonymous_dashboard_response.status_code}"
    )


# Test an incorrect password.
invalid_login_response = (
    auth_test_client.post(
        "/login",
        data={
            "email": (
                "auth-regression@example.com"
            ),
            "password": (
                "Wrong-Password-123!"
            ),
        },
        follow_redirects=True,
    )
)


if b"Invalid email address or password" not in (
    invalid_login_response.data
):
    raise RuntimeError(
        "An invalid login did not fail safely."
    )


# Test valid login.
valid_login_response = (
    auth_test_client.post(
        "/login",
        data={
            "email": (
                "auth-regression@example.com"
            ),
            "password": (
                "Secure-Test-123!"
            ),
        },
        follow_redirects=True,
    )
)


valid_login_html = (
    valid_login_response.data.decode(
        "utf-8",
        errors="replace",
    )
)


if "Vaultify Test Company" not in valid_login_html:
    raise RuntimeError(
        "Valid login did not reach the dashboard. "
        f"Response preview: "
        f"{valid_login_html[:2000]}"
    )


# Verify the login session again.
with auth_test_client.session_transaction() as session_data:
    login_user_id = session_data.get(
        "_user_id"
    )

    login_organization_id = session_data.get(
        "organization_id"
    )


if login_user_id is None:
    raise RuntimeError(
        "The login session contains no user ID."
    )


if login_organization_id is None:
    raise RuntimeError(
        "The login session contains "
        "no organization ID."
    )


# Remove the temporary authentication test database.
with auth_test_app.app_context():
    db.session.remove()
    db.drop_all()
    db.engine.dispose()


if AUTH_TEST_DATABASE_PATH.exists():
    AUTH_TEST_DATABASE_PATH.unlink()


# Restore the normal Vaultify application.
vaultify_web_app = create_app()


print("✅ Authentication route file rewritten successfully.")

print("\n🔐 Registration verification:")
print("   Registration page loads: True")
print("   User created: True")
print("   Organization created: True")
print("   Owner membership created: True")
print("   Flask-Login session created: True")
print("   Trusted organization session created: True")
print("   Dashboard reached after registration: True")

print("\n🔑 Login verification:")
print("   Logout succeeds: True")
print("   Anonymous dashboard access blocked: True")
print("   Incorrect password rejected: True")
print("   Valid login succeeds: True")
print("   Dashboard reached after login: True")

print("\n🏢 Tenant verification:")
print(
    f"   Generated tenant ID: "
    f"{tested_tenant_id}"
)

print(
    "\n✅ Phase 3.3 authentication flow "
    "is working correctly."
)
print(
    "ℹ️ The temporary authentication "
    "test database was deleted."
)

✅ Authentication route file rewritten successfully.

🔐 Registration verification:
   Registration page loads: True
   User created: True
   Organization created: True
   Owner membership created: True
   Flask-Login session created: True
   Trusted organization session created: True
   Dashboard reached after registration: True

🔑 Login verification:
   Logout succeeds: True
   Anonymous dashboard access blocked: True
   Incorrect password rejected: True
   Valid login succeeds: True
   Dashboard reached after login: True

🏢 Tenant verification:
   Generated tenant ID: tenant_4fdf3d1c7d39a72ad69fdac2

✅ Phase 3.3 authentication flow is working correctly.
ℹ️ The temporary authentication test database was deleted.


In [ ]:
# PHASE 3 - Cell 5: Add trusted tenant resolution, organization switching, and role authorization

import importlib
import sys
import textwrap
from pathlib import Path


PROJECT_ROOT = Path("/content/vaultify_app")


AUTHORIZATION_CONTENT = """
    from functools import wraps

    from flask import abort, g, session
    from flask_login import current_user

    from .extensions import db
    from .models import Membership, Organization


    VALID_ROLES = {
        "owner",
        "admin",
        "member",
        "viewer",
    }


    def get_user_memberships():
        \"\"\"Return active organizations belonging to the current user.\"\"\"
        if not current_user.is_authenticated:
            return []

        return list(
            db.session.scalars(
                db.select(Membership)
                .join(
                    Organization,
                    Membership.organization_id
                    == Organization.id,
                )
                .where(
                    Membership.user_id
                    == current_user.id,
                    Organization.is_active.is_(True),
                )
                .order_by(
                    Organization.name.asc()
                )
            )
        )


    def resolve_current_membership(required=True):
        \"\"\"
        Resolve the current membership from trusted session state.

        This function never accepts tenant_id or organization ownership
        information from a form, URL, prompt, or MCP tool argument.
        \"\"\"
        cached_membership = getattr(
            g,
            "vaultify_current_membership",
            None,
        )

        if cached_membership is not None:
            return cached_membership

        if not current_user.is_authenticated:
            if required:
                abort(401)

            return None

        organization_id = session.get(
            "organization_id"
        )

        if not isinstance(
            organization_id,
            int,
        ):
            if required:
                abort(403)

            return None

        membership = db.session.scalar(
            db.select(Membership).where(
                Membership.user_id
                == current_user.id,
                Membership.organization_id
                == organization_id,
            )
        )

        if (
            membership is None
            or membership.organization is None
            or not membership.organization.is_active
        ):
            if required:
                abort(403)

            return None

        g.vaultify_current_membership = (
            membership
        )

        g.vaultify_current_organization = (
            membership.organization
        )

        return membership


    def resolve_current_organization(required=True):
        \"\"\"Return the trusted current organization.\"\"\"
        membership = resolve_current_membership(
            required=required
        )

        if membership is None:
            return None

        return membership.organization


    def resolve_current_tenant_id(required=True):
        \"\"\"Return the trusted Qdrant tenant identifier.\"\"\"
        organization = resolve_current_organization(
            required=required
        )

        if organization is None:
            return None

        return organization.tenant_id


    def roles_required(*allowed_roles):
        \"\"\"Restrict a route to selected organization roles.\"\"\"
        invalid_roles = (
            set(allowed_roles)
            - VALID_ROLES
        )

        if invalid_roles:
            raise ValueError(
                "Unknown Vaultify roles: "
                f"{sorted(invalid_roles)}"
            )

        def decorator(view_function):
            @wraps(view_function)
            def wrapped_view(*args, **kwargs):
                membership = (
                    resolve_current_membership()
                )

                if (
                    membership.role
                    not in allowed_roles
                ):
                    abort(403)

                return view_function(
                    *args,
                    **kwargs,
                )

            return wrapped_view

        return decorator
"""


ORGANIZATION_FORMS_CONTENT = """
    from flask_wtf import FlaskForm
    from wtforms import SelectField, SubmitField
    from wtforms.validators import DataRequired


    class OrganizationSwitchForm(FlaskForm):
        \"\"\"Select one organization belonging to the current user.\"\"\"

        organization_id = SelectField(
            "Organization",
            coerce=int,
            validators=[
                DataRequired(),
            ],
        )

        submit = SubmitField(
            "Switch"
        )
"""


ORGANIZATION_ROUTES_CONTENT = """
    from flask import (
        Blueprint,
        abort,
        flash,
        redirect,
        session,
        url_for,
    )
    from flask_login import (
        current_user,
        login_required,
    )

    from app.authorization import (
        get_user_memberships,
        resolve_current_membership,
    )
    from app.extensions import db, limiter
    from app.models import Membership

    from .forms import OrganizationSwitchForm


    organizations_bp = Blueprint(
        "organizations",
        __name__,
    )


    def build_organization_switch_form():
        \"\"\"Build a switch form using only trusted memberships.\"\"\"
        form = OrganizationSwitchForm()

        memberships = get_user_memberships()

        form.organization_id.choices = [
            (
                membership.organization_id,
                membership.organization.name,
            )
            for membership in memberships
        ]

        if not form.is_submitted():
            form.organization_id.data = (
                session.get(
                    "organization_id"
                )
            )

        return form


    def organization_navigation_context():
        \"\"\"Inject trusted organization navigation into templates.\"\"\"
        if not current_user.is_authenticated:
            return {
                "organization_switch_form": None,
                "navigation_membership": None,
                "navigation_organization": None,
            }

        membership = resolve_current_membership(
            required=False
        )

        return {
            "organization_switch_form": (
                build_organization_switch_form()
            ),
            "navigation_membership": membership,
            "navigation_organization": (
                membership.organization
                if membership is not None
                else None
            ),
        }


    @organizations_bp.post(
        "/organization/switch"
    )
    @login_required
    @limiter.limit("30 per minute")
    def switch_organization():
        \"\"\"Switch only to an organization owned by a membership.\"\"\"
        form = build_organization_switch_form()

        if not form.validate_on_submit():
            abort(400)

        requested_organization_id = (
            form.organization_id.data
        )

        membership = db.session.scalar(
            db.select(Membership).where(
                Membership.user_id
                == current_user.id,
                Membership.organization_id
                == requested_organization_id,
            )
        )

        if (
            membership is None
            or membership.organization is None
            or not membership.organization.is_active
        ):
            abort(403)

        session["organization_id"] = (
            membership.organization_id
        )

        flash(
            f"Switched to "
            f"{membership.organization.name}.",
            "success",
        )

        return redirect(
            url_for("dashboard.home")
        )
"""


DASHBOARD_ROUTES_CONTENT = """
    from flask import (
        Blueprint,
        jsonify,
        render_template,
    )
    from flask_login import login_required

    from app.authorization import (
        resolve_current_membership,
        resolve_current_organization,
        resolve_current_tenant_id,
        roles_required,
    )


    dashboard_bp = Blueprint(
        "dashboard",
        __name__,
    )


    @dashboard_bp.get("/dashboard")
    @login_required
    def home():
        \"\"\"Render the tenant-scoped Vaultify dashboard.\"\"\"
        membership = resolve_current_membership()
        organization = resolve_current_organization()
        tenant_id = resolve_current_tenant_id()

        return render_template(
            "dashboard/home.html",
            membership=membership,
            organization=organization,
            tenant_id=tenant_id,
        )


    @dashboard_bp.get(
        "/dashboard/management"
    )
    @login_required
    @roles_required(
        "owner",
        "admin",
    )
    def management():
        \"\"\"Render organization management for privileged roles.\"\"\"
        membership = resolve_current_membership()
        organization = resolve_current_organization()

        return render_template(
            "dashboard/management.html",
            membership=membership,
            organization=organization,
        )


    @dashboard_bp.get(
        "/api/current-tenant"
    )
    @login_required
    def current_tenant():
        \"\"\"Return the trusted tenant resolved from membership.\"\"\"
        membership = resolve_current_membership()
        organization = resolve_current_organization()

        return jsonify(
            {
                "organization_id": organization.id,
                "organization_name": organization.name,
                "tenant_id": organization.tenant_id,
                "role": membership.role,
            }
        )
"""


APP_INIT_CONTENT = """
    from pathlib import Path

    from flask import (
        Flask,
        jsonify,
        redirect,
        url_for,
    )
    from flask_login import current_user

    from .config import Config
    from .extensions import (
        csrf,
        db,
        limiter,
        login_manager,
        migrate,
    )


    def create_app(config_class=Config):
        \"\"\"Create and configure the Vaultify Flask application.\"\"\"
        app = Flask(__name__)

        app.config.from_object(
            config_class
        )

        Path(
            app.config["UPLOAD_FOLDER"]
        ).mkdir(
            parents=True,
            exist_ok=True,
        )

        database_directory = (
            Path(app.root_path).parent
            / "instance"
        )

        database_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        db.init_app(app)
        login_manager.init_app(app)
        csrf.init_app(app)
        migrate.init_app(app, db)
        limiter.init_app(app)

        from . import models

        from .auth.routes import auth_bp
        from .dashboard.routes import dashboard_bp
        from .organizations.routes import (
            organization_navigation_context,
            organizations_bp,
        )

        app.register_blueprint(auth_bp)
        app.register_blueprint(
            dashboard_bp
        )
        app.register_blueprint(
            organizations_bp
        )

        app.context_processor(
            organization_navigation_context
        )

        @app.get("/")
        def index():
            \"\"\"Send users to the appropriate Vaultify page.\"\"\"
            if current_user.is_authenticated:
                return redirect(
                    url_for("dashboard.home")
                )

            return redirect(
                url_for("auth.login")
            )

        @app.get("/health")
        @limiter.exempt
        def health():
            \"\"\"Return the application health status.\"\"\"
            return jsonify(
                {
                    "status": "ok",
                    "service": "Vaultify",
                    "phase": 3,
                }
            ), 200

        return app
"""


BASE_TEMPLATE_CONTENT = """
    <!doctype html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <meta
            name="viewport"
            content="width=device-width, initial-scale=1"
        >
        <title>
            {% block title %}Vaultify{% endblock %}
        </title>
        <link
            rel="stylesheet"
            href="{{ url_for('static', filename='css/app.css') }}"
        >
    </head>
    <body>
        <header class="site-header">
            <a
                class="brand"
                href="{{ url_for('index') }}"
            >
                Vaultify
            </a>

            {% if current_user.is_authenticated %}
                <nav class="main-navigation">
                    {% if organization_switch_form %}
                        <form
                            method="post"
                            action="{{ url_for('organizations.switch_organization') }}"
                            class="organization-switcher"
                        >
                            {{ organization_switch_form.hidden_tag() }}

                            {{
                                organization_switch_form.organization_id(
                                    class="organization-select",
                                    onchange="this.form.submit()"
                                )
                            }}

                            <noscript>
                                {{
                                    organization_switch_form.submit(
                                        class="secondary-button"
                                    )
                                }}
                            </noscript>
                        </form>
                    {% endif %}

                    <a href="{{ url_for('dashboard.home') }}">
                        Dashboard
                    </a>

                    {% if
                        navigation_membership
                        and navigation_membership.role
                        in ["owner", "admin"]
                    %}
                        <a href="{{ url_for('dashboard.management') }}">
                            Organization
                        </a>
                    {% endif %}

                    <span class="navigation-user">
                        {{ current_user.display_name }}
                    </span>

                    <form
                        method="post"
                        action="{{ url_for('auth.logout') }}"
                        class="inline-form"
                    >
                        <input
                            type="hidden"
                            name="csrf_token"
                            value="{{ csrf_token() }}"
                        >

                        <button
                            type="submit"
                            class="nav-button"
                        >
                            Sign out
                        </button>
                    </form>
                </nav>
            {% endif %}
        </header>

        <main class="page-shell">
            {% with messages = get_flashed_messages(with_categories=true) %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ category }}">
                        {{ message }}
                    </div>
                {% endfor %}
            {% endwith %}

            {% block content %}{% endblock %}
        </main>
    </body>
    </html>
"""


DASHBOARD_TEMPLATE_CONTENT = """
    {% extends "base.html" %}

    {% block title %}Dashboard — Vaultify{% endblock %}

    {% block content %}
        <section class="dashboard-hero">
            <p class="eyebrow">
                Private AI workspace
            </p>

            <h1>
                Welcome, {{ current_user.display_name }}
            </h1>

            <p class="muted">
                Your tenant context is resolved from your
                authenticated organization membership.
            </p>
        </section>

        <section class="dashboard-grid">
            <article class="info-card">
                <span class="card-label">
                    Organization
                </span>

                <strong>
                    {{ organization.name }}
                </strong>
            </article>

            <article class="info-card">
                <span class="card-label">
                    Role
                </span>

                <strong>
                    {{ membership.role|title }}
                </strong>
            </article>

            <article class="info-card">
                <span class="card-label">
                    Tenant ID
                </span>

                <strong class="code-value">
                    {{ tenant_id }}
                </strong>
            </article>

            <article class="info-card">
                <span class="card-label">
                    Account
                </span>

                <strong>
                    {{ current_user.email }}
                </strong>
            </article>
        </section>

        <section class="next-step-card">
            <h2>
                Trusted tenant resolution is active
            </h2>

            <p>
                Dashboard requests, document operations,
                Qdrant searches, and MCP credentials can now
                obtain their tenant context from this verified
                organization membership.
            </p>

            {% if membership.role in ["owner", "admin"] %}
                <a
                    class="text-link"
                    href="{{ url_for('dashboard.management') }}"
                >
                    Open organization management →
                </a>
            {% endif %}
        </section>
    {% endblock %}
"""


MANAGEMENT_TEMPLATE_CONTENT = """
    {% extends "base.html" %}

    {% block title %}
        Organization — Vaultify
    {% endblock %}

    {% block content %}
        <section class="dashboard-hero">
            <p class="eyebrow">
                Organization management
            </p>

            <h1>
                {{ organization.name }}
            </h1>

            <p class="muted">
                This area is restricted to organization
                owners and administrators.
            </p>
        </section>

        <section class="dashboard-grid">
            <article class="info-card">
                <span class="card-label">
                    Your role
                </span>

                <strong>
                    {{ membership.role|title }}
                </strong>
            </article>

            <article class="info-card">
                <span class="card-label">
                    Organization slug
                </span>

                <strong class="code-value">
                    {{ organization.slug }}
                </strong>
            </article>

            <article class="info-card">
                <span class="card-label">
                    Trusted tenant
                </span>

                <strong class="code-value">
                    {{ organization.tenant_id }}
                </strong>
            </article>
        </section>
    {% endblock %}
"""


PROJECT_FILES = {
    "app/authorization.py": AUTHORIZATION_CONTENT,
    "app/organizations/__init__.py": (
        '"""Vaultify organization management package."""'
    ),
    "app/organizations/forms.py": (
        ORGANIZATION_FORMS_CONTENT
    ),
    "app/organizations/routes.py": (
        ORGANIZATION_ROUTES_CONTENT
    ),
    "app/dashboard/routes.py": (
        DASHBOARD_ROUTES_CONTENT
    ),
    "app/__init__.py": APP_INIT_CONTENT,
    "app/templates/base.html": (
        BASE_TEMPLATE_CONTENT
    ),
    "app/templates/dashboard/home.html": (
        DASHBOARD_TEMPLATE_CONTENT
    ),
    "app/templates/dashboard/management.html": (
        MANAGEMENT_TEMPLATE_CONTENT
    ),
}


for relative_path, content in PROJECT_FILES.items():
    target_path = (
        PROJECT_ROOT / relative_path
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    target_path.write_text(
        textwrap.dedent(content).lstrip(),
        encoding="utf-8",
    )


CSS_PATH = (
    PROJECT_ROOT
    / "app"
    / "static"
    / "css"
    / "app.css"
)


css_content = CSS_PATH.read_text(
    encoding="utf-8"
)


ORGANIZATION_CSS_MARKER = (
    "/* Vaultify organization navigation */"
)


if ORGANIZATION_CSS_MARKER not in css_content:
    css_content += """

/* Vaultify organization navigation */

.main-navigation {
    display: flex;
    align-items: center;
    gap: 18px;
}

.organization-switcher {
    display: flex;
    align-items: center;
}

.organization-select {
    max-width: 210px;
    padding: 9px 34px 9px 12px;
    border: 1px solid rgba(255, 255, 255, 0.12);
    border-radius: 10px;
    background: #0e1320;
    color: #ffffff;
    font: inherit;
    cursor: pointer;
}

.navigation-user {
    color: #aab1c2;
    font-size: 0.9rem;
}

.secondary-button {
    padding: 9px 12px;
    border: 1px solid rgba(255, 255, 255, 0.14);
    border-radius: 10px;
    background: transparent;
    color: #ffffff;
    cursor: pointer;
}

.text-link {
    display: inline-block;
    margin-top: 8px;
    font-weight: 700;
}

@media (max-width: 800px) {
    .site-header {
        align-items: flex-start;
        gap: 15px;
        padding-top: 16px;
        padding-bottom: 16px;
    }

    .main-navigation {
        flex-wrap: wrap;
        justify-content: flex-end;
    }

    .organization-select {
        max-width: 170px;
    }

    .navigation-user {
        display: none;
    }
}
"""

    CSS_PATH.write_text(
        css_content,
        encoding="utf-8",
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


importlib.invalidate_caches()


for module_name in list(sys.modules):
    if (
        module_name == "app"
        or module_name.startswith("app.")
    ):
        del sys.modules[module_name]


from app import create_app
from app.config import Config
from app.extensions import db
from app.models import (
    Membership,
    Organization,
    User,
)


TENANT_TEST_DATABASE_PATH = (
    PROJECT_ROOT
    / "instance"
    / "vaultify_tenant_test.db"
)


if TENANT_TEST_DATABASE_PATH.exists():
    TENANT_TEST_DATABASE_PATH.unlink()


class TenantTestConfig(Config):
    """Configuration used only for tenant authorization tests."""

    TESTING = True
    WTF_CSRF_ENABLED = False
    RATELIMIT_ENABLED = False

    SQLALCHEMY_DATABASE_URI = (
        f"sqlite:///"
        f"{TENANT_TEST_DATABASE_PATH}"
    )


tenant_test_app = create_app(
    TenantTestConfig
)


with tenant_test_app.app_context():
    db.create_all()

    test_user = User(
        email="tenant-test@example.com",
        display_name="Tenant Test User",
        is_verified=True,
    )

    test_user.set_password(
        "Secure-Test-123!"
    )

    owner_organization = Organization(
        name="Owner Organization",
        slug="owner-organization",
    )

    viewer_organization = Organization(
        name="Viewer Organization",
        slug="viewer-organization",
    )

    forbidden_organization = Organization(
        name="Forbidden Organization",
        slug="forbidden-organization",
    )

    owner_membership = Membership(
        user=test_user,
        organization=owner_organization,
        role="owner",
    )

    viewer_membership = Membership(
        user=test_user,
        organization=viewer_organization,
        role="viewer",
    )

    db.session.add_all(
        [
            test_user,
            owner_organization,
            viewer_organization,
            forbidden_organization,
            owner_membership,
            viewer_membership,
        ]
    )

    db.session.commit()

    owner_organization_id = (
        owner_organization.id
    )

    viewer_organization_id = (
        viewer_organization.id
    )

    forbidden_organization_id = (
        forbidden_organization.id
    )

    owner_tenant_id = (
        owner_organization.tenant_id
    )

    viewer_tenant_id = (
        viewer_organization.tenant_id
    )


tenant_test_client = (
    tenant_test_app.test_client()
)


login_response = tenant_test_client.post(
    "/login",
    data={
        "email": "tenant-test@example.com",
        "password": "Secure-Test-123!",
    },
    follow_redirects=True,
)


if (
    login_response.status_code != 200
    or b"Owner Organization"
    not in login_response.data
):
    raise RuntimeError(
        "The tenant test user could not log in "
        "to the owner organization."
    )


owner_api_response = (
    tenant_test_client.get(
        "/api/current-tenant"
    )
)


owner_api_payload = (
    owner_api_response.get_json()
)


if (
    owner_api_response.status_code != 200
    or owner_api_payload["tenant_id"]
    != owner_tenant_id
    or owner_api_payload["role"]
    != "owner"
):
    raise RuntimeError(
        "Trusted owner tenant resolution failed."
    )


owner_management_response = (
    tenant_test_client.get(
        "/dashboard/management"
    )
)


if owner_management_response.status_code != 200:
    raise RuntimeError(
        "The owner could not access "
        "organization management."
    )


viewer_switch_response = (
    tenant_test_client.post(
        "/organization/switch",
        data={
            "organization_id": (
                viewer_organization_id
            ),
        },
        follow_redirects=True,
    )
)


if (
    viewer_switch_response.status_code != 200
    or b"Viewer Organization"
    not in viewer_switch_response.data
):
    raise RuntimeError(
        "Switching to the viewer organization failed."
    )


viewer_api_response = (
    tenant_test_client.get(
        "/api/current-tenant"
    )
)


viewer_api_payload = (
    viewer_api_response.get_json()
)


if (
    viewer_api_payload["tenant_id"]
    != viewer_tenant_id
    or viewer_api_payload["role"]
    != "viewer"
):
    raise RuntimeError(
        "Trusted viewer tenant resolution failed."
    )


viewer_management_response = (
    tenant_test_client.get(
        "/dashboard/management"
    )
)


if viewer_management_response.status_code != 403:
    raise RuntimeError(
        "A viewer unexpectedly accessed "
        "owner/admin management."
    )


forbidden_switch_response = (
    tenant_test_client.post(
        "/organization/switch",
        data={
            "organization_id": (
                forbidden_organization_id
            ),
        },
        follow_redirects=False,
    )
)


if forbidden_switch_response.status_code != 400:
    raise RuntimeError(
        "A non-member organization switch "
        "was not rejected."
    )


with tenant_test_client.session_transaction() as session_data:
    session_data["organization_id"] = (
        forbidden_organization_id
    )


tampered_dashboard_response = (
    tenant_test_client.get(
        "/dashboard",
        follow_redirects=False,
    )
)


if tampered_dashboard_response.status_code != 403:
    raise RuntimeError(
        "A manipulated organization session "
        "was not rejected."
    )


with tenant_test_client.session_transaction() as session_data:
    session_data["organization_id"] = (
        owner_organization_id
    )


restored_owner_response = (
    tenant_test_client.get(
        "/api/current-tenant"
    )
)


if (
    restored_owner_response.status_code != 200
    or restored_owner_response.get_json()[
        "tenant_id"
    ]
    != owner_tenant_id
):
    raise RuntimeError(
        "The owner tenant could not be restored."
    )


with tenant_test_app.app_context():
    db.session.remove()
    db.drop_all()
    db.engine.dispose()


if TENANT_TEST_DATABASE_PATH.exists():
    TENANT_TEST_DATABASE_PATH.unlink()


vaultify_web_app = create_app()


print(
    "✅ Trusted tenant authorization layer created."
)

print("\n🏢 Organization behavior:")
print(
    "   Owner organization resolved: True"
)
print(
    "   Viewer organization resolved: True"
)
print(
    "   Organization switching works: True"
)
print(
    "   Non-member organization rejected: True"
)

print("\n🔐 Role authorization:")
print(
    "   Owner management access: Allowed"
)
print(
    "   Viewer management access: Denied"
)

print("\n🛡️ Tenant isolation:")
print(
    "   Tenant ID resolved from membership: True"
)
print(
    "   Raw tenant ID input required: False"
)
print(
    "   Manipulated session rejected: True"
)
print(
    "   Cross-organization switch rejected: True"
)

print("\n🎨 Dashboard additions:")
print(
    "   Organization switcher created"
)
print(
    "   Current organization navigation created"
)
print(
    "   Owner/admin management page created"
)

print(
    "\n✅ Phase 3.4 trusted tenant "
    "resolution is working correctly."
)

✅ Trusted tenant authorization layer created.

🏢 Organization behavior:
   Owner organization resolved: True
   Viewer organization resolved: True
   Organization switching works: True
   Non-member organization rejected: True

🔐 Role authorization:
   Owner management access: Allowed
   Viewer management access: Denied

🛡️ Tenant isolation:
   Tenant ID resolved from membership: True
   Raw tenant ID input required: False
   Manipulated session rejected: True
   Cross-organization switch rejected: True

🎨 Dashboard additions:
   Organization switcher created
   Current organization navigation created
   Owner/admin management page created

✅ Phase 3.4 trusted tenant resolution is working correctly.


In [ ]:
# PHASE 3 - Cell 6: Add secure tenant-scoped PDF upload and document dashboard

import importlib
import io
import shutil
import sys
import textwrap
from pathlib import Path


PROJECT_ROOT = Path("/content/vaultify_app")


DOCUMENT_FORMS_CONTENT = """
    from flask_wtf import FlaskForm
    from flask_wtf.file import (
        FileField,
        FileRequired,
    )
    from wtforms import SubmitField


    class DocumentUploadForm(FlaskForm):
        \"\"\"Collect a PDF document for secure tenant-scoped upload.\"\"\"

        file = FileField(
            "PDF document",
            validators=[
                FileRequired(
                    message="Please select a PDF document."
                ),
            ],
        )

        submit = SubmitField(
            "Upload document"
        )
"""


DOCUMENT_SERVICE_CONTENT = """
    import hashlib
    import os
    import uuid
    from pathlib import Path

    import filetype
    from flask import current_app
    from sqlalchemy.exc import IntegrityError
    from werkzeug.utils import secure_filename

    from app.extensions import db
    from app.models import Document


    ALLOWED_EXTENSIONS = {
        ".pdf",
    }

    ALLOWED_MIME_TYPES = {
        "application/pdf",
        "application/x-pdf",
    }

    STREAM_CHUNK_SIZE = (
        1024 * 1024
    )


    class InvalidUploadError(ValueError):
        \"\"\"Raised when an uploaded file fails validation.\"\"\"


    class DuplicateDocumentError(ValueError):
        \"\"\"Raised when the tenant already owns the same document.\"\"\"

        def __init__(self, existing_document):
            self.existing_document = existing_document

            super().__init__(
                "This document already exists "
                "in the current organization."
            )


    def normalize_uploaded_filename(filename):
        \"\"\"Return a safe user-facing filename.\"\"\"
        original_filename = (
            filename or ""
        ).strip()

        safe_filename = secure_filename(
            original_filename
        )

        if not safe_filename:
            raise InvalidUploadError(
                "The selected file has an invalid filename."
            )

        extension = Path(
            safe_filename
        ).suffix.lower()

        if extension not in ALLOWED_EXTENSIONS:
            raise InvalidUploadError(
                "Only PDF documents are currently supported."
            )

        return (
            original_filename,
            safe_filename,
        )


    def validate_declared_mime_type(file_storage):
        \"\"\"Validate the client-declared MIME type before saving.\"\"\"
        declared_mime_type = (
            file_storage.mimetype or ""
        ).lower()

        if declared_mime_type not in ALLOWED_MIME_TYPES:
            raise InvalidUploadError(
                "The uploaded file does not declare "
                "a supported PDF MIME type."
            )

        return declared_mime_type


    def save_stream_to_temporary_file(
        file_storage,
        temporary_path,
        maximum_size_bytes,
    ):
        \"\"\"
        Save the upload while calculating its hash and enforcing size limits.
        \"\"\"
        document_hasher = hashlib.sha256()
        total_size = 0
        header_bytes = bytearray()

        file_storage.stream.seek(0)

        with temporary_path.open("wb") as output_file:
            while True:
                chunk = file_storage.stream.read(
                    STREAM_CHUNK_SIZE
                )

                if not chunk:
                    break

                total_size += len(chunk)

                if total_size > maximum_size_bytes:
                    raise InvalidUploadError(
                        "The uploaded document exceeds "
                        "the configured file-size limit."
                    )

                if len(header_bytes) < 512:
                    remaining_header_bytes = (
                        512 - len(header_bytes)
                    )

                    header_bytes.extend(
                        chunk[:remaining_header_bytes]
                    )

                document_hasher.update(chunk)
                output_file.write(chunk)

        if total_size == 0:
            raise InvalidUploadError(
                "The uploaded document is empty."
            )

        return {
            "size_bytes": total_size,
            "document_hash": document_hasher.hexdigest(),
            "header_bytes": bytes(header_bytes),
        }


    def validate_pdf_signature(header_bytes):
        \"\"\"Verify that the file content is genuinely recognized as PDF.\"\"\"
        detected_type = filetype.guess(
            header_bytes
        )

        if (
            detected_type is None
            or detected_type.extension != "pdf"
            or detected_type.mime != "application/pdf"
        ):
            raise InvalidUploadError(
                "The uploaded file is not a valid PDF document."
            )


    def find_duplicate_document(
        organization_id,
        document_hash,
    ):
        \"\"\"Find an existing document hash inside the current tenant.\"\"\"
        return db.session.scalar(
            db.select(Document).where(
                Document.organization_id
                == organization_id,
                Document.document_hash
                == document_hash,
            )
        )


    def create_uploaded_document(
        file_storage,
        organization,
        uploaded_by_user,
    ):
        \"\"\"
        Validate, store, and register a tenant-owned PDF document.

        The organization is supplied by trusted membership resolution.
        \"\"\"
        (
            original_filename,
            safe_filename,
        ) = normalize_uploaded_filename(
            file_storage.filename
        )

        declared_mime_type = (
            validate_declared_mime_type(
                file_storage
            )
        )

        maximum_size_bytes = int(
            current_app.config[
                "MAX_CONTENT_LENGTH"
            ]
        )

        tenant_upload_directory = (
            Path(
                current_app.config[
                    "UPLOAD_FOLDER"
                ]
            )
            / organization.tenant_id
        )

        tenant_upload_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        temporary_path = (
            tenant_upload_directory
            / f".upload-{uuid.uuid4().hex}.tmp"
        )

        final_path = None

        try:
            stream_result = (
                save_stream_to_temporary_file(
                    file_storage=file_storage,
                    temporary_path=temporary_path,
                    maximum_size_bytes=(
                        maximum_size_bytes
                    ),
                )
            )

            validate_pdf_signature(
                stream_result["header_bytes"]
            )

            duplicate_document = (
                find_duplicate_document(
                    organization_id=organization.id,
                    document_hash=(
                        stream_result[
                            "document_hash"
                        ]
                    ),
                )
            )

            if duplicate_document is not None:
                raise DuplicateDocumentError(
                    duplicate_document
                )

            stored_filename = (
                f"{uuid.uuid4().hex}.pdf"
            )

            final_path = (
                tenant_upload_directory
                / stored_filename
            )

            os.replace(
                temporary_path,
                final_path,
            )

            document = Document(
                organization_id=organization.id,
                filename=safe_filename,
                original_filename=original_filename,
                stored_path=str(final_path),
                mime_type=declared_mime_type,
                size_bytes=(
                    stream_result[
                        "size_bytes"
                    ]
                ),
                document_hash=(
                    stream_result[
                        "document_hash"
                    ]
                ),
                status="uploaded",
                chunk_count=0,
                uploaded_by_user_id=(
                    uploaded_by_user.id
                ),
            )

            db.session.add(document)
            db.session.commit()

            return document

        except DuplicateDocumentError:
            db.session.rollback()
            raise

        except InvalidUploadError:
            db.session.rollback()
            raise

        except IntegrityError as error:
            db.session.rollback()

            raise DuplicateDocumentError(
                find_duplicate_document(
                    organization_id=organization.id,
                    document_hash=(
                        stream_result[
                            "document_hash"
                        ]
                    ),
                )
            ) from error

        except Exception:
            db.session.rollback()
            raise

        finally:
            if temporary_path.exists():
                temporary_path.unlink()

            if (
                final_path is not None
                and final_path.exists()
                and "document" not in locals()
            ):
                final_path.unlink()
"""


DOCUMENT_ROUTES_CONTENT = """
    from flask import (
        Blueprint,
        flash,
        redirect,
        render_template,
        url_for,
    )
    from flask_login import (
        current_user,
        login_required,
    )

    from app.authorization import (
        resolve_current_membership,
        resolve_current_organization,
        roles_required,
    )
    from app.extensions import (
        db,
        limiter,
    )
    from app.models import Document

    from .forms import DocumentUploadForm
    from .service import (
        DuplicateDocumentError,
        InvalidUploadError,
        create_uploaded_document,
    )


    documents_bp = Blueprint(
        "documents",
        __name__,
    )


    @documents_bp.get("/documents")
    @login_required
    def list_documents():
        \"\"\"List documents belonging to the active organization.\"\"\"
        membership = (
            resolve_current_membership()
        )

        organization = (
            resolve_current_organization()
        )

        documents = list(
            db.session.scalars(
                db.select(Document)
                .where(
                    Document.organization_id
                    == organization.id
                )
                .order_by(
                    Document.created_at.desc()
                )
            )
        )

        return render_template(
            "documents/index.html",
            membership=membership,
            organization=organization,
            documents=documents,
        )


    @documents_bp.route(
        "/documents/upload",
        methods=["GET", "POST"],
    )
    @login_required
    @roles_required(
        "owner",
        "admin",
        "member",
    )
    @limiter.limit("10 per hour")
    def upload_document():
        \"\"\"Upload a PDF into the trusted active organization.\"\"\"
        organization = (
            resolve_current_organization()
        )

        form = DocumentUploadForm()

        if form.validate_on_submit():
            try:
                document = (
                    create_uploaded_document(
                        file_storage=form.file.data,
                        organization=organization,
                        uploaded_by_user=(
                            current_user
                        ),
                    )
                )

            except InvalidUploadError as error:
                flash(
                    str(error),
                    "danger",
                )

            except DuplicateDocumentError as error:
                flash(
                    str(error),
                    "warning",
                )

            else:
                flash(
                    f"{document.filename} was uploaded "
                    "successfully.",
                    "success",
                )

                return redirect(
                    url_for(
                        "documents.list_documents"
                    )
                )

        return render_template(
            "documents/upload.html",
            form=form,
            organization=organization,
        )
"""


APP_INIT_CONTENT = """
    from pathlib import Path

    from flask import (
        Flask,
        jsonify,
        redirect,
        url_for,
    )
    from flask_login import current_user

    from .config import Config
    from .extensions import (
        csrf,
        db,
        limiter,
        login_manager,
        migrate,
    )


    def create_app(config_class=Config):
        \"\"\"Create and configure the Vaultify Flask application.\"\"\"
        app = Flask(__name__)

        app.config.from_object(
            config_class
        )

        Path(
            app.config["UPLOAD_FOLDER"]
        ).mkdir(
            parents=True,
            exist_ok=True,
        )

        database_directory = (
            Path(app.root_path).parent
            / "instance"
        )

        database_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        db.init_app(app)
        login_manager.init_app(app)
        csrf.init_app(app)
        migrate.init_app(app, db)
        limiter.init_app(app)

        from . import models

        from .auth.routes import auth_bp
        from .dashboard.routes import dashboard_bp
        from .documents.routes import documents_bp
        from .organizations.routes import (
            organization_navigation_context,
            organizations_bp,
        )

        app.register_blueprint(auth_bp)
        app.register_blueprint(
            dashboard_bp
        )
        app.register_blueprint(
            documents_bp
        )
        app.register_blueprint(
            organizations_bp
        )

        app.context_processor(
            organization_navigation_context
        )

        @app.get("/")
        def index():
            \"\"\"Send users to the appropriate Vaultify page.\"\"\"
            if current_user.is_authenticated:
                return redirect(
                    url_for("dashboard.home")
                )

            return redirect(
                url_for("auth.login")
            )

        @app.get("/health")
        @limiter.exempt
        def health():
            \"\"\"Return the application health status.\"\"\"
            return jsonify(
                {
                    "status": "ok",
                    "service": "Vaultify",
                    "phase": 3,
                }
            ), 200

        return app
"""


BASE_TEMPLATE_CONTENT = """
    <!doctype html>
    <html lang="en">
    <head>
        <meta charset="utf-8">

        <meta
            name="viewport"
            content="width=device-width, initial-scale=1"
        >

        <title>
            {% block title %}Vaultify{% endblock %}
        </title>

        <link
            rel="stylesheet"
            href="{{ url_for('static', filename='css/app.css') }}"
        >
    </head>

    <body>
        <header class="site-header">
            <a
                class="brand"
                href="{{ url_for('index') }}"
            >
                Vaultify
            </a>

            {% if current_user.is_authenticated %}
                <nav class="main-navigation">
                    {% if organization_switch_form %}
                        <form
                            method="post"
                            action="{{ url_for('organizations.switch_organization') }}"
                            class="organization-switcher"
                        >
                            {{ organization_switch_form.hidden_tag() }}

                            {{
                                organization_switch_form.organization_id(
                                    class="organization-select",
                                    onchange="this.form.submit()"
                                )
                            }}

                            <noscript>
                                {{
                                    organization_switch_form.submit(
                                        class="secondary-button"
                                    )
                                }}
                            </noscript>
                        </form>
                    {% endif %}

                    <a href="{{ url_for('dashboard.home') }}">
                        Dashboard
                    </a>

                    <a href="{{ url_for('documents.list_documents') }}">
                        Documents
                    </a>

                    {% if
                        navigation_membership
                        and navigation_membership.role
                        in ["owner", "admin"]
                    %}
                        <a href="{{ url_for('dashboard.management') }}">
                            Organization
                        </a>
                    {% endif %}

                    <span class="navigation-user">
                        {{ current_user.display_name }}
                    </span>

                    <form
                        method="post"
                        action="{{ url_for('auth.logout') }}"
                        class="inline-form"
                    >
                        <input
                            type="hidden"
                            name="csrf_token"
                            value="{{ csrf_token() }}"
                        >

                        <button
                            type="submit"
                            class="nav-button"
                        >
                            Sign out
                        </button>
                    </form>
                </nav>
            {% endif %}
        </header>

        <main class="page-shell">
            {% with messages = get_flashed_messages(with_categories=true) %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ category }}">
                        {{ message }}
                    </div>
                {% endfor %}
            {% endwith %}

            {% block content %}{% endblock %}
        </main>
    </body>
    </html>
"""


DOCUMENT_INDEX_TEMPLATE_CONTENT = """
    {% extends "base.html" %}

    {% block title %}
        Documents — Vaultify
    {% endblock %}

    {% block content %}
        <section class="page-heading-row">
            <div>
                <p class="eyebrow">
                    Knowledge base
                </p>

                <h1>
                    Documents
                </h1>

                <p class="muted">
                    Files belonging to {{ organization.name }}.
                </p>
            </div>

            {% if membership.role in ["owner", "admin", "member"] %}
                <a
                    class="primary-link-button"
                    href="{{ url_for('documents.upload_document') }}"
                >
                    Upload PDF
                </a>
            {% endif %}
        </section>

        {% if documents %}
            <section class="document-list">
                {% for document in documents %}
                    <article class="document-row">
                        <div class="document-icon">
                            PDF
                        </div>

                        <div class="document-details">
                            <strong>
                                {{ document.filename }}
                            </strong>

                            <span>
                                {{ "%.2f"|format(document.size_bytes / 1048576) }}
                                MB
                                ·
                                Uploaded by
                                {{ document.uploaded_by.display_name }}
                            </span>
                        </div>

                        <div class="document-metadata">
                            <span class="status-badge status-{{ document.status }}">
                                {{ document.status|title }}
                            </span>

                            <span>
                                {{ document.chunk_count }}
                                chunks
                            </span>
                        </div>
                    </article>
                {% endfor %}
            </section>

        {% else %}
            <section class="empty-state">
                <div class="empty-state-icon">
                    PDF
                </div>

                <h2>
                    No documents yet
                </h2>

                <p class="muted">
                    Upload the first PDF for this organization.
                </p>

                {% if membership.role in ["owner", "admin", "member"] %}
                    <a
                        class="primary-link-button"
                        href="{{ url_for('documents.upload_document') }}"
                    >
                        Upload first document
                    </a>
                {% endif %}
            </section>
        {% endif %}
    {% endblock %}
"""


DOCUMENT_UPLOAD_TEMPLATE_CONTENT = """
    {% extends "base.html" %}

    {% block title %}
        Upload document — Vaultify
    {% endblock %}

    {% block content %}
        <section class="upload-card">
            <p class="eyebrow">
                {{ organization.name }}
            </p>

            <h1>
                Upload a PDF
            </h1>

            <p class="muted">
                The document will be securely assigned
                to the active organization.
            </p>

            <form
                method="post"
                enctype="multipart/form-data"
                novalidate
            >
                {{ form.hidden_tag() }}

                <label class="upload-dropzone">
                    {{ form.file(
                        accept="application/pdf"
                    ) }}

                    <span class="upload-dropzone-title">
                        Select a PDF document
                    </span>

                    <span class="upload-dropzone-description">
                        Maximum file size: 25 MB
                    </span>
                </label>

                {% for error in form.file.errors %}
                    <p class="field-error">
                        {{ error }}
                    </p>
                {% endfor %}

                {{ form.submit(
                    class="primary-button"
                ) }}
            </form>

            <a
                class="back-link"
                href="{{ url_for('documents.list_documents') }}"
            >
                ← Back to documents
            </a>
        </section>
    {% endblock %}
"""


PROJECT_FILES = {
    "app/documents/forms.py": (
        DOCUMENT_FORMS_CONTENT
    ),
    "app/documents/service.py": (
        DOCUMENT_SERVICE_CONTENT
    ),
    "app/documents/routes.py": (
        DOCUMENT_ROUTES_CONTENT
    ),
    "app/__init__.py": (
        APP_INIT_CONTENT
    ),
    "app/templates/base.html": (
        BASE_TEMPLATE_CONTENT
    ),
    "app/templates/documents/index.html": (
        DOCUMENT_INDEX_TEMPLATE_CONTENT
    ),
    "app/templates/documents/upload.html": (
        DOCUMENT_UPLOAD_TEMPLATE_CONTENT
    ),
}


for relative_path, content in PROJECT_FILES.items():
    target_path = (
        PROJECT_ROOT / relative_path
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    target_path.write_text(
        textwrap.dedent(content).lstrip(),
        encoding="utf-8",
    )


CSS_PATH = (
    PROJECT_ROOT
    / "app"
    / "static"
    / "css"
    / "app.css"
)


css_content = CSS_PATH.read_text(
    encoding="utf-8"
)


DOCUMENT_CSS_MARKER = (
    "/* Vaultify document dashboard */"
)


if DOCUMENT_CSS_MARKER not in css_content:
    css_content += """

/* Vaultify document dashboard */

.page-heading-row {
    display: flex;
    align-items: flex-end;
    justify-content: space-between;
    gap: 24px;
    margin-bottom: 32px;
}

.primary-link-button {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    padding: 13px 18px;
    border-radius: 12px;
    background: linear-gradient(
        135deg,
        #6f7cff,
        #8d61ff
    );
    color: #ffffff;
    font-weight: 800;
}

.primary-link-button:hover {
    color: #ffffff;
    transform: translateY(-1px);
}

.document-list {
    display: grid;
    gap: 14px;
}

.document-row {
    display: grid;
    grid-template-columns:
        auto
        minmax(0, 1fr)
        auto;
    align-items: center;
    gap: 18px;
    padding: 20px;
    border: 1px solid rgba(255, 255, 255, 0.09);
    border-radius: 16px;
    background: rgba(17, 22, 35, 0.84);
}

.document-icon,
.empty-state-icon {
    display: grid;
    place-items: center;
    width: 48px;
    height: 48px;
    border-radius: 13px;
    background: rgba(119, 136, 255, 0.13);
    color: #aab5ff;
    font-size: 0.76rem;
    font-weight: 900;
}

.document-details {
    min-width: 0;
}

.document-details strong {
    display: block;
    overflow: hidden;
    margin-bottom: 6px;
    text-overflow: ellipsis;
    white-space: nowrap;
}

.document-details span,
.document-metadata span {
    color: #9da5b8;
    font-size: 0.88rem;
}

.document-metadata {
    display: flex;
    align-items: flex-end;
    flex-direction: column;
    gap: 7px;
}

.status-badge {
    display: inline-flex;
    padding: 5px 9px;
    border-radius: 999px;
    font-size: 0.76rem;
    font-weight: 800;
}

.status-uploaded {
    background: rgba(96, 165, 250, 0.14);
    color: #93c5fd !important;
}

.status-parsing,
.status-indexing {
    background: rgba(251, 191, 36, 0.14);
    color: #fcd34d !important;
}

.status-ready {
    background: rgba(52, 211, 153, 0.14);
    color: #6ee7b7 !important;
}

.status-failed {
    background: rgba(248, 113, 113, 0.14);
    color: #fca5a5 !important;
}

.empty-state {
    padding: 70px 25px;
    border: 1px dashed rgba(255, 255, 255, 0.14);
    border-radius: 20px;
    text-align: center;
    background: rgba(17, 22, 35, 0.55);
}

.empty-state-icon {
    margin: 0 auto 18px;
}

.upload-card {
    width: min(620px, 100%);
    margin: 0 auto;
    padding: 36px;
    border: 1px solid rgba(255, 255, 255, 0.10);
    border-radius: 24px;
    background: rgba(17, 22, 35, 0.92);
}

.upload-dropzone {
    display: grid;
    place-items: center;
    min-height: 210px;
    margin: 28px 0 22px;
    padding: 30px;
    border: 1px dashed rgba(154, 168, 255, 0.42);
    border-radius: 18px;
    background: rgba(119, 136, 255, 0.06);
    text-align: center;
    cursor: pointer;
}

.upload-dropzone input {
    width: min(100%, 360px);
    margin-bottom: 20px;
}

.upload-dropzone-title {
    display: block;
    margin-bottom: 7px;
    font-weight: 800;
}

.upload-dropzone-description {
    color: #9da5b8;
    font-size: 0.9rem;
}

.back-link {
    display: inline-block;
    margin-top: 20px;
}

@media (max-width: 700px) {
    .page-heading-row {
        align-items: flex-start;
        flex-direction: column;
    }

    .document-row {
        grid-template-columns:
            auto
            minmax(0, 1fr);
    }

    .document-metadata {
        grid-column: 1 / -1;
        align-items: center;
        flex-direction: row;
        justify-content: space-between;
    }

    .upload-card {
        padding: 24px;
    }
}
"""

    CSS_PATH.write_text(
        css_content,
        encoding="utf-8",
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


importlib.invalidate_caches()


for module_name in list(sys.modules):
    if (
        module_name == "app"
        or module_name.startswith("app.")
    ):
        del sys.modules[module_name]


from app import create_app
from app.config import Config
from app.extensions import db
from app.models import (
    Document,
    Membership,
    Organization,
    User,
)


UPLOAD_TEST_DATABASE_PATH = (
    PROJECT_ROOT
    / "instance"
    / "vaultify_upload_test.db"
)

UPLOAD_TEST_DIRECTORY = (
    PROJECT_ROOT
    / "uploads"
    / "_phase3_upload_test"
)


if UPLOAD_TEST_DATABASE_PATH.exists():
    UPLOAD_TEST_DATABASE_PATH.unlink()


if UPLOAD_TEST_DIRECTORY.exists():
    shutil.rmtree(
        UPLOAD_TEST_DIRECTORY
    )


UPLOAD_TEST_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


class UploadTestConfig(Config):
    """Configuration used only for document upload tests."""

    TESTING = True
    WTF_CSRF_ENABLED = False
    RATELIMIT_ENABLED = False

    SQLALCHEMY_DATABASE_URI = (
        f"sqlite:///"
        f"{UPLOAD_TEST_DATABASE_PATH}"
    )

    UPLOAD_FOLDER = str(
        UPLOAD_TEST_DIRECTORY
    )

    MAX_CONTENT_LENGTH = (
        2 * 1024 * 1024
    )


upload_test_app = create_app(
    UploadTestConfig
)


with upload_test_app.app_context():
    db.create_all()

    owner_user = User(
        email="upload-owner@example.com",
        display_name="Upload Owner",
        is_verified=True,
    )

    owner_user.set_password(
        "Secure-Test-123!"
    )

    viewer_user = User(
        email="upload-viewer@example.com",
        display_name="Upload Viewer",
        is_verified=True,
    )

    viewer_user.set_password(
        "Secure-Test-123!"
    )

    test_organization = Organization(
        name="Upload Test Organization",
        slug="upload-test-organization",
    )

    owner_membership = Membership(
        user=owner_user,
        organization=test_organization,
        role="owner",
    )

    viewer_membership = Membership(
        user=viewer_user,
        organization=test_organization,
        role="viewer",
    )

    db.session.add_all(
        [
            owner_user,
            viewer_user,
            test_organization,
            owner_membership,
            viewer_membership,
        ]
    )

    db.session.commit()

    test_organization_id = (
        test_organization.id
    )

    test_tenant_id = (
        test_organization.tenant_id
    )


upload_test_client = (
    upload_test_app.test_client()
)


owner_login_response = (
    upload_test_client.post(
        "/login",
        data={
            "email": (
                "upload-owner@example.com"
            ),
            "password": (
                "Secure-Test-123!"
            ),
        },
        follow_redirects=True,
    )
)


if owner_login_response.status_code != 200:
    raise RuntimeError(
        "The upload test owner could not log in."
    )


VALID_TEST_PDF = (
    b"%PDF-1.4\n"
    b"%\xe2\xe3\xcf\xd3\n"
    b"1 0 obj\n"
    b"<< /Type /Catalog >>\n"
    b"endobj\n"
    b"trailer\n"
    b"<< /Root 1 0 R >>\n"
    b"%%EOF\n"
)


valid_upload_response = (
    upload_test_client.post(
        "/documents/upload",
        data={
            "file": (
                io.BytesIO(
                    VALID_TEST_PDF
                ),
                "annual-report.pdf",
                "application/pdf",
            ),
        },
        content_type="multipart/form-data",
        follow_redirects=True,
    )
)


if (
    valid_upload_response.status_code != 200
    or b"annual-report.pdf"
    not in valid_upload_response.data
):
    raise RuntimeError(
        "A valid PDF upload did not succeed."
    )


with upload_test_app.app_context():
    stored_documents = list(
        db.session.scalars(
            db.select(Document)
        )
    )

    if len(stored_documents) != 1:
        raise RuntimeError(
            "The valid upload did not create "
            "exactly one document record."
        )

    stored_document = (
        stored_documents[0]
    )

    if (
        stored_document.organization_id
        != test_organization_id
    ):
        raise RuntimeError(
            "The uploaded document was assigned "
            "to the wrong organization."
        )

    if stored_document.status != "uploaded":
        raise RuntimeError(
            "The uploaded document has "
            "an unexpected initial status."
        )

    if not Path(
        stored_document.stored_path
    ).exists():
        raise RuntimeError(
            "The uploaded PDF was not stored."
        )

    stored_document_hash = (
        stored_document.document_hash
    )

    stored_file_path = (
        stored_document.stored_path
    )


duplicate_upload_response = (
    upload_test_client.post(
        "/documents/upload",
        data={
            "file": (
                io.BytesIO(
                    VALID_TEST_PDF
                ),
                "duplicate-name.pdf",
                "application/pdf",
            ),
        },
        content_type="multipart/form-data",
        follow_redirects=True,
    )
)


if b"already exists" not in (
    duplicate_upload_response.data
):
    raise RuntimeError(
        "A duplicate document was not rejected."
    )


invalid_upload_response = (
    upload_test_client.post(
        "/documents/upload",
        data={
            "file": (
                io.BytesIO(
                    b"This is not a PDF file."
                ),
                "fake-document.pdf",
                "application/pdf",
            ),
        },
        content_type="multipart/form-data",
        follow_redirects=True,
    )
)


if b"not a valid PDF" not in (
    invalid_upload_response.data
):
    raise RuntimeError(
        "A fake PDF file was not rejected."
    )


with upload_test_app.app_context():
    document_count_after_rejections = (
        db.session.scalar(
            db.select(
                db.func.count(
                    Document.id
                )
            )
        )
    )


if document_count_after_rejections != 1:
    raise RuntimeError(
        "Rejected uploads unexpectedly "
        "created document records."
    )


upload_test_client.post(
    "/logout",
    follow_redirects=True,
)


viewer_login_response = (
    upload_test_client.post(
        "/login",
        data={
            "email": (
                "upload-viewer@example.com"
            ),
            "password": (
                "Secure-Test-123!"
            ),
        },
        follow_redirects=True,
    )
)


if viewer_login_response.status_code != 200:
    raise RuntimeError(
        "The viewer could not log in."
    )


viewer_upload_response = (
    upload_test_client.post(
        "/documents/upload",
        data={
            "file": (
                io.BytesIO(
                    VALID_TEST_PDF
                    + b"viewer"
                ),
                "viewer-upload.pdf",
                "application/pdf",
            ),
        },
        content_type="multipart/form-data",
        follow_redirects=False,
    )
)


if viewer_upload_response.status_code != 403:
    raise RuntimeError(
        "A viewer unexpectedly gained "
        "document upload permission."
    )


viewer_list_response = (
    upload_test_client.get(
        "/documents"
    )
)


if (
    viewer_list_response.status_code != 200
    or b"annual-report.pdf"
    not in viewer_list_response.data
):
    raise RuntimeError(
        "The viewer could not read "
        "the tenant document list."
    )


upload_test_client.post(
    "/logout",
    follow_redirects=True,
)


anonymous_documents_response = (
    upload_test_client.get(
        "/documents",
        follow_redirects=False,
    )
)


if anonymous_documents_response.status_code != 302:
    raise RuntimeError(
        "Anonymous document access "
        "was not blocked."
    )


with upload_test_app.app_context():
    db.session.remove()
    db.drop_all()
    db.engine.dispose()


if UPLOAD_TEST_DATABASE_PATH.exists():
    UPLOAD_TEST_DATABASE_PATH.unlink()


if UPLOAD_TEST_DIRECTORY.exists():
    shutil.rmtree(
        UPLOAD_TEST_DIRECTORY
    )


vaultify_web_app = create_app()


print(
    "✅ Secure tenant-scoped document upload created."
)

print("\n📁 Upload validation:")
print(
    "   PDF extension validation: Passed"
)
print(
    "   Declared MIME validation: Passed"
)
print(
    "   PDF file signature validation: Passed"
)
print(
    "   Empty file rejection: Enabled"
)
print(
    "   File-size enforcement: Enabled"
)
print(
    "   Server-generated storage names: Enabled"
)

print("\n🏢 Tenant ownership:")
print(
    "   Document assigned to trusted organization: True"
)
print(
    f"   Test tenant ID: "
    f"{test_tenant_id}"
)
print(
    "   Raw tenant ID accepted from upload form: False"
)

print("\n🔐 Authorization:")
print(
    "   Owner upload: Allowed"
)
print(
    "   Viewer upload: Denied"
)
print(
    "   Viewer tenant document list: Allowed"
)
print(
    "   Anonymous document list: Denied"
)

print("\n🛡️ Integrity:")
print(
    "   SHA-256 document hash generated: True"
)
print(
    f"   Hash length: "
    f"{len(stored_document_hash)}"
)
print(
    "   Duplicate document rejected: True"
)
print(
    "   Fake PDF rejected: True"
)
print(
    "   Rejected uploads stored in database: False"
)

print("\n🎨 Dashboard additions:")
print(
    "   Document list page created"
)
print(
    "   PDF upload page created"
)
print(
    "   Empty document state created"
)
print(
    "   Upload and status components created"
)

print(
    "\n✅ Phase 3.5A secure upload "
    "foundation is working correctly."
)
print(
    "ℹ️ Uploaded documents begin with "
    "status='uploaded'."
)
print(
    "➡️ Next: connect uploaded documents "
    "to Docling, embeddings, and Qdrant."
)

✅ Secure tenant-scoped document upload created.

📁 Upload validation:
   PDF extension validation: Passed
   Declared MIME validation: Passed
   PDF file signature validation: Passed
   Empty file rejection: Enabled
   File-size enforcement: Enabled
   Server-generated storage names: Enabled

🏢 Tenant ownership:
   Document assigned to trusted organization: True
   Test tenant ID: tenant_b1b1155442afc01e8875d00e
   Raw tenant ID accepted from upload form: False

🔐 Authorization:
   Owner upload: Allowed
   Viewer upload: Denied
   Viewer tenant document list: Allowed
   Anonymous document list: Denied

🛡️ Integrity:
   SHA-256 document hash generated: True
   Hash length: 64
   Duplicate document rejected: True
   Fake PDF rejected: True
   Rejected uploads stored in database: False

🎨 Dashboard additions:
   Document list page created
   PDF upload page created
   Empty document state created
   Upload and status components created

✅ Phase 3.5A secure upload foundation is working cor

In [ ]:
# PHASE 3 - Cell 7: Connect web uploads to Docling, embeddings, and Qdrant

import hashlib
import importlib
import shutil
import sys
import textwrap
import time
from pathlib import Path


PROJECT_ROOT = Path("/content/vaultify_app")


REQUIRED_NOTEBOOK_GLOBALS = [
    "parse_pdf_to_markdown",
    "smart_chunk_markdown",
    "embedding_model",
    "qdrant",
    "COLLECTION_NAME",
    "VECTOR_SIZE",
    "TENANT_ID_FIELD",
    "DOCUMENT_HASH_FIELD",
    "FILENAME_FIELD",
    "CHUNK_INDEX_FIELD",
    "CHUNK_TYPE_FIELD",
    "TEXT_FIELD",
    "seed_document_paths",
]


missing_notebook_globals = [
    variable_name
    for variable_name in REQUIRED_NOTEBOOK_GLOBALS
    if variable_name not in globals()
]


if missing_notebook_globals:
    raise RuntimeError(
        "Required Phase 1 objects are missing: "
        f"{missing_notebook_globals}. "
        "Run the Phase 1 cells before this cell."
    )


RAG_RUNTIME_CONTENT = """
    import uuid
    from pathlib import Path

    import numpy as np
    from qdrant_client.models import (
        FieldCondition,
        Filter,
        FilterSelector,
        MatchValue,
        PointStruct,
    )


    _runtime = {}


    class RAGRuntimeNotConfiguredError(RuntimeError):
        \"\"\"Raised when the Colab RAG runtime has not been registered.\"\"\"


    def configure_rag_runtime(
        *,
        parser,
        chunker,
        embedding_model,
        qdrant_client,
        collection_name,
        vector_size,
        field_names,
        embedding_batch_size=64,
        qdrant_batch_size=100,
    ):
        \"\"\"Register the working Phase 1 pipeline for the Flask app.\"\"\"
        _runtime.clear()

        _runtime.update(
            {
                "parser": parser,
                "chunker": chunker,
                "embedding_model": embedding_model,
                "qdrant": qdrant_client,
                "collection_name": collection_name,
                "vector_size": vector_size,
                "field_names": field_names,
                "embedding_batch_size": embedding_batch_size,
                "qdrant_batch_size": qdrant_batch_size,
            }
        )


    def rag_runtime_is_ready():
        \"\"\"Return True when every required RAG component is registered.\"\"\"
        required_keys = {
            "parser",
            "chunker",
            "embedding_model",
            "qdrant",
            "collection_name",
            "vector_size",
            "field_names",
        }

        return required_keys.issubset(
            _runtime.keys()
        )


    def require_runtime():
        \"\"\"Return the registered runtime or fail safely.\"\"\"
        if not rag_runtime_is_ready():
            raise RAGRuntimeNotConfiguredError(
                "The Vaultify RAG runtime is not configured."
            )

        return _runtime


    def build_document_filter(
        tenant_id,
        document_hash,
    ):
        \"\"\"Build the mandatory tenant and document Qdrant filter.\"\"\"
        runtime = require_runtime()
        fields = runtime["field_names"]

        return Filter(
            must=[
                FieldCondition(
                    key=fields["tenant_id"],
                    match=MatchValue(
                        value=tenant_id
                    ),
                ),
                FieldCondition(
                    key=fields["document_hash"],
                    match=MatchValue(
                        value=document_hash
                    ),
                ),
            ]
        )


    def build_point_id(
        tenant_id,
        document_hash,
        chunk_index,
    ):
        \"\"\"Create a deterministic tenant-safe Qdrant point ID.\"\"\"
        identity = (
            f"{tenant_id}:"
            f"{document_hash}:"
            f"{chunk_index}"
        )

        return str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                identity,
            )
        )


    def delete_document_vectors(
        tenant_id,
        document_hash,
    ):
        \"\"\"Delete vectors for exactly one tenant-owned document.\"\"\"
        runtime = require_runtime()

        runtime["qdrant"].delete(
            collection_name=runtime[
                "collection_name"
            ],
            points_selector=FilterSelector(
                filter=build_document_filter(
                    tenant_id=tenant_id,
                    document_hash=document_hash,
                )
            ),
            wait=True,
        )


    def count_document_vectors(
        tenant_id,
        document_hash,
    ):
        \"\"\"Count vectors for exactly one tenant-owned document.\"\"\"
        runtime = require_runtime()

        return runtime["qdrant"].count(
            collection_name=runtime[
                "collection_name"
            ],
            count_filter=build_document_filter(
                tenant_id=tenant_id,
                document_hash=document_hash,
            ),
            exact=True,
        ).count


    def build_qdrant_points(
        *,
        chunks,
        embeddings,
        tenant_id,
        document_hash,
        filename,
    ):
        \"\"\"Convert chunks and embeddings into deterministic Qdrant points.\"\"\"
        runtime = require_runtime()
        fields = runtime["field_names"]

        if len(chunks) != len(embeddings):
            raise RuntimeError(
                "Chunk and embedding counts do not match."
            )

        points = []

        for chunk, embedding in zip(
            chunks,
            embeddings,
        ):
            chunk_index = chunk.get(
                "chunk_index"
            )

            if chunk_index is None:
                raise RuntimeError(
                    "A generated chunk has no chunk_index."
                )

            chunk_text = str(
                chunk.get(
                    "text",
                    "",
                )
            ).strip()

            if not chunk_text:
                raise RuntimeError(
                    "A generated chunk contains no text."
                )

            point_payload = {
                fields["tenant_id"]: tenant_id,
                fields["document_hash"]: (
                    document_hash
                ),
                fields["filename"]: filename,
                fields["chunk_index"]: (
                    chunk_index
                ),
                fields["chunk_type"]: (
                    chunk.get(
                        "chunk_type",
                        "text",
                    )
                ),
                "section": chunk.get(
                    "section",
                    "Document introduction",
                ),
                fields["text"]: chunk_text,
            }

            points.append(
                PointStruct(
                    id=build_point_id(
                        tenant_id=tenant_id,
                        document_hash=(
                            document_hash
                        ),
                        chunk_index=chunk_index,
                    ),
                    vector=embedding.tolist(),
                    payload=point_payload,
                )
            )

        return points


    def ingest_document_record(document_id):
        \"\"\"
        Parse, chunk, embed, and index one database document.

        The document's trusted organization supplies the Qdrant tenant ID.
        \"\"\"
        from .extensions import db
        from .models import (
            Document,
            utc_now,
        )

        runtime = require_runtime()

        document = db.session.get(
            Document,
            document_id,
        )

        if document is None:
            raise LookupError(
                "The document record does not exist."
            )

        if document.organization is None:
            raise RuntimeError(
                "The document has no organization."
            )

        tenant_id = (
            document.organization.tenant_id
        )

        document_hash = (
            document.document_hash
        )

        document_path = Path(
            document.stored_path
        )

        if not document_path.is_file():
            document.status = "failed"
            document.error_message = (
                "The stored PDF file could not be found."
            )

            db.session.commit()

            raise FileNotFoundError(
                document.error_message
            )

        try:
            document.status = "parsing"
            document.error_message = None
            document.chunk_count = 0
            document.indexed_at = None
            db.session.commit()

            markdown_content = runtime[
                "parser"
            ](
                document_path
            )

            if not str(
                markdown_content
            ).strip():
                raise RuntimeError(
                    "The PDF parser returned no readable content."
                )

            chunks = runtime["chunker"](
                markdown_content
            )

            if not chunks:
                raise RuntimeError(
                    "The document produced no indexable chunks."
                )

            document.status = "indexing"
            db.session.commit()

            chunk_texts = [
                chunk["text"]
                for chunk in chunks
            ]

            embeddings = runtime[
                "embedding_model"
            ].encode(
                chunk_texts,
                batch_size=runtime[
                    "embedding_batch_size"
                ],
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
            )

            if embeddings.ndim != 2:
                raise RuntimeError(
                    "The embedding model returned "
                    "an invalid array shape."
                )

            if embeddings.shape[0] != len(
                chunks
            ):
                raise RuntimeError(
                    "The number of embeddings does not "
                    "match the number of chunks."
                )

            if (
                embeddings.shape[1]
                != runtime["vector_size"]
            ):
                raise RuntimeError(
                    "The embedding vector size does not "
                    "match the Qdrant collection."
                )

            if not np.isfinite(
                embeddings
            ).all():
                raise RuntimeError(
                    "The generated embeddings contain "
                    "invalid numeric values."
                )

            points = build_qdrant_points(
                chunks=chunks,
                embeddings=embeddings,
                tenant_id=tenant_id,
                document_hash=document_hash,
                filename=document.filename,
            )

            # Remove a previous partial or retry result first.
            delete_document_vectors(
                tenant_id=tenant_id,
                document_hash=document_hash,
            )

            batch_size = runtime[
                "qdrant_batch_size"
            ]

            for batch_start in range(
                0,
                len(points),
                batch_size,
            ):
                batch = points[
                    batch_start:
                    batch_start + batch_size
                ]

                runtime["qdrant"].upsert(
                    collection_name=runtime[
                        "collection_name"
                    ],
                    points=batch,
                    wait=True,
                )

            indexed_count = (
                count_document_vectors(
                    tenant_id=tenant_id,
                    document_hash=document_hash,
                )
            )

            if indexed_count != len(points):
                raise RuntimeError(
                    "Qdrant verification failed. "
                    f"Expected {len(points)} points, "
                    f"found {indexed_count}."
                )

            document.status = "ready"
            document.chunk_count = len(
                points
            )
            document.error_message = None
            document.indexed_at = utc_now()

            db.session.commit()

            return {
                "document_id": document.id,
                "tenant_id": tenant_id,
                "document_hash": (
                    document_hash
                ),
                "filename": document.filename,
                "chunk_count": len(points),
                "status": document.status,
            }

        except Exception as error:
            db.session.rollback()

            cleanup_error = None

            try:
                delete_document_vectors(
                    tenant_id=tenant_id,
                    document_hash=document_hash,
                )

            except Exception as vector_error:
                cleanup_error = vector_error

            failed_document = db.session.get(
                Document,
                document_id,
            )

            if failed_document is not None:
                failed_document.status = "failed"
                failed_document.chunk_count = 0
                failed_document.indexed_at = None

                safe_error_message = (
                    f"{type(error).__name__}: {error}"
                )[:2000]

                if cleanup_error is not None:
                    safe_error_message += (
                        " | Vector cleanup also failed."
                    )

                failed_document.error_message = (
                    safe_error_message
                )

                db.session.commit()

            raise
"""


DOCUMENT_ROUTES_CONTENT = """
    from flask import (
        Blueprint,
        abort,
        flash,
        redirect,
        render_template,
        url_for,
    )
    from flask_login import (
        current_user,
        login_required,
    )

    from app.authorization import (
        resolve_current_membership,
        resolve_current_organization,
        roles_required,
    )
    from app.extensions import (
        db,
        limiter,
    )
    from app.models import Document
    from app.rag_runtime import (
        ingest_document_record,
    )

    from .forms import DocumentUploadForm
    from .service import (
        DuplicateDocumentError,
        InvalidUploadError,
        create_uploaded_document,
    )


    documents_bp = Blueprint(
        "documents",
        __name__,
    )


    @documents_bp.get("/documents")
    @login_required
    def list_documents():
        \"\"\"List documents belonging to the active organization.\"\"\"
        membership = (
            resolve_current_membership()
        )

        organization = (
            resolve_current_organization()
        )

        documents = list(
            db.session.scalars(
                db.select(Document)
                .where(
                    Document.organization_id
                    == organization.id
                )
                .order_by(
                    Document.created_at.desc()
                )
            )
        )

        return render_template(
            "documents/index.html",
            membership=membership,
            organization=organization,
            documents=documents,
        )


    @documents_bp.route(
        "/documents/upload",
        methods=["GET", "POST"],
    )
    @login_required
    @roles_required(
        "owner",
        "admin",
        "member",
    )
    @limiter.limit("10 per hour")
    def upload_document():
        \"\"\"Upload and synchronously index a tenant-owned PDF.\"\"\"
        organization = (
            resolve_current_organization()
        )

        form = DocumentUploadForm()

        if form.validate_on_submit():
            try:
                document = (
                    create_uploaded_document(
                        file_storage=form.file.data,
                        organization=organization,
                        uploaded_by_user=(
                            current_user
                        ),
                    )
                )

            except InvalidUploadError as error:
                flash(
                    str(error),
                    "danger",
                )

            except DuplicateDocumentError as error:
                flash(
                    str(error),
                    "warning",
                )

            else:
                try:
                    ingestion_result = (
                        ingest_document_record(
                            document.id
                        )
                    )

                except Exception:
                    flash(
                        f"{document.filename} was uploaded, "
                        "but processing failed. "
                        "You can retry it from the document list.",
                        "danger",
                    )

                else:
                    flash(
                        f"{document.filename} is ready "
                        f"with "
                        f"{ingestion_result['chunk_count']} "
                        "indexed chunks.",
                        "success",
                    )

                return redirect(
                    url_for(
                        "documents.list_documents"
                    )
                )

        return render_template(
            "documents/upload.html",
            form=form,
            organization=organization,
        )


    @documents_bp.post(
        "/documents/<int:document_id>/retry"
    )
    @login_required
    @roles_required(
        "owner",
        "admin",
        "member",
    )
    @limiter.limit("10 per hour")
    def retry_document(document_id):
        \"\"\"Retry ingestion for a document owned by the active tenant.\"\"\"
        organization = (
            resolve_current_organization()
        )

        document = db.session.scalar(
            db.select(Document).where(
                Document.id == document_id,
                Document.organization_id
                == organization.id,
            )
        )

        if document is None:
            abort(404)

        if document.status not in {
            "uploaded",
            "failed",
        }:
            flash(
                "Only uploaded or failed documents "
                "can be retried.",
                "warning",
            )

            return redirect(
                url_for(
                    "documents.list_documents"
                )
            )

        try:
            ingestion_result = (
                ingest_document_record(
                    document.id
                )
            )

        except Exception:
            flash(
                f"{document.filename} could not be processed.",
                "danger",
            )

        else:
            flash(
                f"{document.filename} is ready "
                f"with "
                f"{ingestion_result['chunk_count']} "
                "indexed chunks.",
                "success",
            )

        return redirect(
            url_for(
                "documents.list_documents"
            )
        )
"""


DOCUMENT_INDEX_TEMPLATE_CONTENT = """
    {% extends "base.html" %}

    {% block title %}
        Documents — Vaultify
    {% endblock %}

    {% block content %}
        <section class="page-heading-row">
            <div>
                <p class="eyebrow">
                    Knowledge base
                </p>

                <h1>
                    Documents
                </h1>

                <p class="muted">
                    Files belonging to {{ organization.name }}.
                </p>
            </div>

            {% if membership.role in ["owner", "admin", "member"] %}
                <a
                    class="primary-link-button"
                    href="{{ url_for('documents.upload_document') }}"
                >
                    Upload PDF
                </a>
            {% endif %}
        </section>

        {% if documents %}
            <section class="document-list">
                {% for document in documents %}
                    <article class="document-row">
                        <div class="document-icon">
                            PDF
                        </div>

                        <div class="document-details">
                            <strong>
                                {{ document.filename }}
                            </strong>

                            <span>
                                {{ "%.2f"|format(document.size_bytes / 1048576) }}
                                MB
                                ·
                                Uploaded by
                                {{ document.uploaded_by.display_name }}
                            </span>

                            {% if document.error_message %}
                                <span class="document-error">
                                    Processing failed. Retry is available.
                                </span>
                            {% endif %}
                        </div>

                        <div class="document-metadata">
                            <span class="status-badge status-{{ document.status }}">
                                {{ document.status|title }}
                            </span>

                            <span>
                                {{ document.chunk_count }}
                                chunks
                            </span>

                            {% if
                                document.status in ["uploaded", "failed"]
                                and membership.role
                                in ["owner", "admin", "member"]
                            %}
                                <form
                                    method="post"
                                    action="{{ url_for(
                                        'documents.retry_document',
                                        document_id=document.id
                                    ) }}"
                                >
                                    <input
                                        type="hidden"
                                        name="csrf_token"
                                        value="{{ csrf_token() }}"
                                    >

                                    <button
                                        type="submit"
                                        class="small-action-button"
                                    >
                                        Retry
                                    </button>
                                </form>
                            {% endif %}
                        </div>
                    </article>
                {% endfor %}
            </section>

        {% else %}
            <section class="empty-state">
                <div class="empty-state-icon">
                    PDF
                </div>

                <h2>
                    No documents yet
                </h2>

                <p class="muted">
                    Upload the first PDF for this organization.
                </p>

                {% if membership.role in ["owner", "admin", "member"] %}
                    <a
                        class="primary-link-button"
                        href="{{ url_for('documents.upload_document') }}"
                    >
                        Upload first document
                    </a>
                {% endif %}
            </section>
        {% endif %}
    {% endblock %}
"""


PROJECT_FILES = {
    "app/rag_runtime.py": (
        RAG_RUNTIME_CONTENT
    ),
    "app/documents/routes.py": (
        DOCUMENT_ROUTES_CONTENT
    ),
    "app/templates/documents/index.html": (
        DOCUMENT_INDEX_TEMPLATE_CONTENT
    ),
}


for relative_path, content in PROJECT_FILES.items():
    target_path = (
        PROJECT_ROOT / relative_path
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    target_path.write_text(
        textwrap.dedent(content).lstrip(),
        encoding="utf-8",
    )


CSS_PATH = (
    PROJECT_ROOT
    / "app"
    / "static"
    / "css"
    / "app.css"
)


css_content = CSS_PATH.read_text(
    encoding="utf-8"
)


RAG_CSS_MARKER = (
    "/* Vaultify ingestion status */"
)


if RAG_CSS_MARKER not in css_content:
    css_content += """

/* Vaultify ingestion status */

.document-error {
    display: block;
    margin-top: 7px;
    color: #fca5a5 !important;
}

.small-action-button {
    padding: 7px 11px;
    border: 1px solid rgba(255, 255, 255, 0.14);
    border-radius: 9px;
    background: transparent;
    color: #ffffff;
    font: inherit;
    font-size: 0.82rem;
    cursor: pointer;
}

.small-action-button:hover {
    border-color: #8795ff;
    background: rgba(119, 136, 255, 0.10);
}
"""

    CSS_PATH.write_text(
        css_content,
        encoding="utf-8",
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


importlib.invalidate_caches()


for module_name in list(sys.modules):
    if (
        module_name == "app"
        or module_name.startswith("app.")
    ):
        del sys.modules[module_name]


from app import create_app
from app.config import Config
from app.extensions import db
from app.models import (
    Document,
    Membership,
    Organization,
    User,
)
from app.rag_runtime import (
    configure_rag_runtime,
    count_document_vectors,
    delete_document_vectors,
    ingest_document_record,
    rag_runtime_is_ready,
)


configure_rag_runtime(
    parser=parse_pdf_to_markdown,
    chunker=smart_chunk_markdown,
    embedding_model=embedding_model,
    qdrant_client=qdrant,
    collection_name=COLLECTION_NAME,
    vector_size=VECTOR_SIZE,
    field_names={
        "tenant_id": TENANT_ID_FIELD,
        "document_hash": (
            DOCUMENT_HASH_FIELD
        ),
        "filename": FILENAME_FIELD,
        "chunk_index": CHUNK_INDEX_FIELD,
        "chunk_type": CHUNK_TYPE_FIELD,
        "text": TEXT_FIELD,
    },
    embedding_batch_size=64,
    qdrant_batch_size=100,
)


if not rag_runtime_is_ready():
    raise RuntimeError(
        "The Flask RAG runtime could not be configured."
    )


INGESTION_TEST_DATABASE_PATH = (
    PROJECT_ROOT
    / "instance"
    / "vaultify_ingestion_test.db"
)

INGESTION_TEST_UPLOAD_DIRECTORY = (
    PROJECT_ROOT
    / "uploads"
    / "_phase3_ingestion_test"
)


if INGESTION_TEST_DATABASE_PATH.exists():
    INGESTION_TEST_DATABASE_PATH.unlink()


if INGESTION_TEST_UPLOAD_DIRECTORY.exists():
    shutil.rmtree(
        INGESTION_TEST_UPLOAD_DIRECTORY
    )


INGESTION_TEST_UPLOAD_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


class IngestionTestConfig(Config):
    """Configuration used only for the full ingestion test."""

    TESTING = True
    WTF_CSRF_ENABLED = False
    RATELIMIT_ENABLED = False

    SQLALCHEMY_DATABASE_URI = (
        f"sqlite:///"
        f"{INGESTION_TEST_DATABASE_PATH}"
    )

    UPLOAD_FOLDER = str(
        INGESTION_TEST_UPLOAD_DIRECTORY
    )


ingestion_test_app = create_app(
    IngestionTestConfig
)


test_tenant_id = None
test_document_hash = None
ingestion_result = None
indexed_point_count = None
test_error = None


try:
    test_seed_filename = next(
        filename
        for filename in seed_document_paths
        if "tesla" in filename.lower()
    )

    test_source_path = Path(
        seed_document_paths[
            test_seed_filename
        ]
    )

    if not test_source_path.is_file():
        raise FileNotFoundError(
            f"Test PDF was not found: "
            f"{test_source_path}"
        )

    test_stored_path = (
        INGESTION_TEST_UPLOAD_DIRECTORY
        / "integration-test-tesla.pdf"
    )

    shutil.copy2(
        test_source_path,
        test_stored_path,
    )

    test_document_hash = hashlib.sha256(
        test_stored_path.read_bytes()
    ).hexdigest()

    with ingestion_test_app.app_context():
        db.create_all()

        test_user = User(
            email=(
                "ingestion-test@example.com"
            ),
            display_name=(
                "Ingestion Test User"
            ),
            is_verified=True,
        )

        test_user.set_password(
            "Secure-Test-123!"
        )

        test_organization = Organization(
            name=(
                "Ingestion Test Organization"
            ),
            slug=(
                "ingestion-test-organization"
            ),
        )

        test_membership = Membership(
            user=test_user,
            organization=test_organization,
            role="owner",
        )

        db.session.add_all(
            [
                test_user,
                test_organization,
                test_membership,
            ]
        )

        db.session.flush()

        test_document = Document(
            organization_id=(
                test_organization.id
            ),
            filename=(
                "integration-test-tesla.pdf"
            ),
            original_filename=(
                test_seed_filename
            ),
            stored_path=str(
                test_stored_path
            ),
            mime_type="application/pdf",
            size_bytes=(
                test_stored_path.stat().st_size
            ),
            document_hash=(
                test_document_hash
            ),
            status="uploaded",
            chunk_count=0,
            uploaded_by_user_id=(
                test_user.id
            ),
        )

        db.session.add(
            test_document
        )

        db.session.commit()

        test_document_id = (
            test_document.id
        )

        test_tenant_id = (
            test_organization.tenant_id
        )

        print(
            "🚀 Running full web ingestion test..."
        )
        print(
            f"📄 Test file: "
            f"{test_seed_filename}"
        )
        print(
            f"🏢 Temporary tenant: "
            f"{test_tenant_id}"
        )

        ingestion_start_time = (
            time.time()
        )

        ingestion_result = (
            ingest_document_record(
                test_document_id
            )
        )

        ingestion_elapsed_seconds = (
            time.time()
            - ingestion_start_time
        )

        refreshed_document = db.session.get(
            Document,
            test_document_id,
        )

        if refreshed_document.status != "ready":
            raise RuntimeError(
                "The test document did not reach "
                "the ready status."
            )

        if refreshed_document.chunk_count <= 0:
            raise RuntimeError(
                "The ready document has no chunks."
            )

        indexed_point_count = (
            count_document_vectors(
                tenant_id=test_tenant_id,
                document_hash=(
                    test_document_hash
                ),
            )
        )

        if (
            indexed_point_count
            != refreshed_document.chunk_count
        ):
            raise RuntimeError(
                "Database and Qdrant chunk counts "
                "do not match."
            )

except Exception as error:
    test_error = error

finally:
    if (
        test_tenant_id is not None
        and test_document_hash is not None
    ):
        try:
            delete_document_vectors(
                tenant_id=test_tenant_id,
                document_hash=(
                    test_document_hash
                ),
            )

        except Exception as cleanup_error:
            print(
                "⚠️ Temporary Qdrant cleanup failed: "
                f"{cleanup_error}"
            )

    try:
        with ingestion_test_app.app_context():
            db.session.remove()
            db.drop_all()
            db.engine.dispose()

    except Exception as cleanup_error:
        print(
            "⚠️ Temporary database cleanup failed: "
            f"{cleanup_error}"
        )

    if INGESTION_TEST_DATABASE_PATH.exists():
        INGESTION_TEST_DATABASE_PATH.unlink()

    if INGESTION_TEST_UPLOAD_DIRECTORY.exists():
        shutil.rmtree(
            INGESTION_TEST_UPLOAD_DIRECTORY
        )


if test_error is not None:
    raise RuntimeError(
        "The full tenant-scoped ingestion test failed. "
        f"Reason: {type(test_error).__name__}: "
        f"{test_error}"
    ) from test_error


vaultify_web_app = create_app()


print(
    "\n✅ Web upload ingestion pipeline created."
)

print("\n🔄 Verified status flow:")
print("   uploaded → parsing: Passed")
print("   parsing → indexing: Passed")
print("   indexing → ready: Passed")

print("\n🧠 Verified RAG processing:")
print("   Docling parsing: Passed")
print("   Token-safe chunking: Passed")
print("   Normalized embeddings: Passed")
print("   Deterministic Qdrant points: Passed")

print("\n🏢 Verified tenant behavior:")
print(
    "   Tenant resolved from document organization: True"
)
print(
    "   Raw tenant ID accepted from upload request: False"
)
print(
    f"   Temporary tenant: "
    f"{test_tenant_id}"
)

print("\n📊 Verified indexing:")
print(
    f"   Indexed chunks: "
    f"{ingestion_result['chunk_count']}"
)
print(
    f"   Exact Qdrant count: "
    f"{indexed_point_count}"
)
print(
    f"   Processing time: "
    f"{ingestion_elapsed_seconds:.1f} seconds"
)

print("\n🛡️ Verified failure protection:")
print(
    "   Partial vectors cleaned on failure: Enabled"
)
print(
    "   Failed status and safe error storage: Enabled"
)
print(
    "   Retry endpoint: Enabled"
)
print(
    "   Temporary test vectors removed: True"
)

print(
    "\n✅ Phase 3.5B tenant-scoped ingestion "
    "is working correctly."
)
print(
    "➡️ Next: add tenant-scoped document "
    "deletion from database, storage, and Qdrant."
)

🚀 Running full web ingestion test...
📄 Test file: tesla_q4_2025_update.pdf
🏢 Temporary tenant: tenant_d4e172230e5bb706c3cfacd6


[WARNING] 2026-07-10 17:21:19,514 [RapidOCR] main.py:125: The text detection result is empty


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


✅ Web upload ingestion pipeline created.

🔄 Verified status flow:
   uploaded → parsing: Passed
   parsing → indexing: Passed
   indexing → ready: Passed

🧠 Verified RAG processing:
   Docling parsing: Passed
   Token-safe chunking: Passed
   Normalized embeddings: Passed
   Deterministic Qdrant points: Passed

🏢 Verified tenant behavior:
   Tenant resolved from document organization: True
   Raw tenant ID accepted from upload request: False
   Temporary tenant: tenant_d4e172230e5bb706c3cfacd6

📊 Verified indexing:
   Indexed chunks: 140
   Exact Qdrant count: 140
   Processing time: 99.4 seconds

🛡️ Verified failure protection:
   Partial vectors cleaned on failure: Enabled
   Failed status and safe error storage: Enabled
   Retry endpoint: Enabled
   Temporary test vectors removed: True

✅ Phase 3.5B tenant-scoped ingestion is working correctly.
➡️ Next: add tenant-scoped document deletion from database, storage, and Qdrant.


In [ ]:
# PHASE 3 - Cell 8: Launch the Vaultify web app and expose it through Cloudflare

import re
import shutil
import socket
import subprocess
import threading
import time

import requests
from werkzeug.serving import make_server


WEB_HOST = "127.0.0.1"

WEB_PORT_CANDIDATES = range(
    5000,
    5011,
)

WEB_TUNNEL_STARTUP_TIMEOUT = 45
WEB_PUBLIC_TEST_ATTEMPTS = 12
WEB_PUBLIC_TEST_RETRY_SECONDS = 3


def web_port_is_open(host, port):
    """
    Return True when a TCP server is listening on the target port.
    """
    with socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    ) as connection_socket:
        connection_socket.settimeout(0.5)

        return (
            connection_socket.connect_ex(
                (host, port)
            )
            == 0
        )


def find_available_web_port():
    """
    Return the first available web port.
    """
    for candidate_port in WEB_PORT_CANDIDATES:
        if not web_port_is_open(
            WEB_HOST,
            candidate_port,
        ):
            return candidate_port

    raise RuntimeError(
        "No available Vaultify web port was found "
        "between 5000 and 5010."
    )


def stream_web_cloudflare_logs(
    process,
    log_storage,
):
    """
    Store and display web tunnel logs.
    """
    if process.stderr is None:
        return

    for log_line in iter(
        process.stderr.readline,
        "",
    ):
        cleaned_line = log_line.strip()

        if cleaned_line:
            log_storage.append(
                cleaned_line
            )

            print(
                f"[web-cloudflared] "
                f"{cleaned_line}"
            )

        if process.poll() is not None:
            break


def find_trycloudflare_url(
    log_lines,
):
    """
    Extract the generated TryCloudflare URL.
    """
    url_pattern = re.compile(
        r"https://[a-zA-Z0-9-]+"
        r"\.trycloudflare\.com"
    )

    for log_line in log_lines:
        url_match = url_pattern.search(
            log_line
        )

        if url_match:
            return url_match.group(0)

    return None


def web_tunnel_is_registered(
    log_lines,
):
    """
    Return True after Cloudflare registers the web tunnel.
    """
    return any(
        "Registered tunnel connection"
        in log_line
        for log_line in log_lines
    )


if "vaultify_web_app" not in globals():
    raise RuntimeError(
        "vaultify_web_app is missing. "
        "Run the previous Phase 3 cells first."
    )


if shutil.which("cloudflared") is None:
    raise RuntimeError(
        "cloudflared is not installed."
    )


# Stop only the previous Vaultify web server.
# The MCP server and MCP Cloudflare tunnel are not touched.
if (
    "vaultify_web_server" in globals()
    and vaultify_web_server is not None
):
    try:
        print(
            "ℹ️ Stopping the previous "
            "Vaultify web server."
        )

        vaultify_web_server.shutdown()

    except Exception as error:
        print(
            "⚠️ Previous web server shutdown warning: "
            f"{error}"
        )


# Stop only the previous web tunnel.
# Do not stop cloudflare_tunnel_process because it belongs to MCP.
if (
    "web_cloudflare_tunnel_process"
    in globals()
    and web_cloudflare_tunnel_process.poll()
    is None
):
    print(
        "ℹ️ Stopping the previous "
        "Vaultify web tunnel."
    )

    web_cloudflare_tunnel_process.terminate()

    try:
        web_cloudflare_tunnel_process.wait(
            timeout=5
        )

    except subprocess.TimeoutExpired:
        web_cloudflare_tunnel_process.kill()

        web_cloudflare_tunnel_process.wait(
            timeout=5
        )


WEB_PORT = find_available_web_port()

LOCAL_WEB_URL = (
    f"http://{WEB_HOST}:{WEB_PORT}"
)


# Ensure the normal application database tables exist.
with vaultify_web_app.app_context():
    db.create_all()


vaultify_web_server = make_server(
    WEB_HOST,
    WEB_PORT,
    vaultify_web_app,
    threaded=True,
)


vaultify_web_server_thread = threading.Thread(
    target=vaultify_web_server.serve_forever,
    name="vaultify-web-server",
    daemon=True,
)


vaultify_web_server_thread.start()


server_deadline = time.time() + 15


while (
    time.time() < server_deadline
    and not web_port_is_open(
        WEB_HOST,
        WEB_PORT,
    )
):
    time.sleep(0.25)


if not web_port_is_open(
    WEB_HOST,
    WEB_PORT,
):
    raise RuntimeError(
        "The Vaultify web server "
        f"did not start on port {WEB_PORT}."
    )


local_health_response = requests.get(
    f"{LOCAL_WEB_URL}/health",
    timeout=10,
)


if local_health_response.status_code != 200:
    raise RuntimeError(
        "The local Vaultify health check failed. "
        f"Status: "
        f"{local_health_response.status_code}"
    )


print(
    "✅ Vaultify web server started locally."
)
print(
    f"🌐 Local web URL: "
    f"{LOCAL_WEB_URL}"
)
print(
    f"🩺 Local health: "
    f"{local_health_response.json()}"
)


WEB_CLOUDFLARE_ORIGIN = LOCAL_WEB_URL

web_cloudflare_logs = []


web_cloudflare_command = [
    "cloudflared",
    "tunnel",
    "--url",
    WEB_CLOUDFLARE_ORIGIN,
    "--no-autoupdate",
]


print(
    f"\n🌐 Web Cloudflare origin: "
    f"{WEB_CLOUDFLARE_ORIGIN}"
)


web_cloudflare_tunnel_process = subprocess.Popen(
    web_cloudflare_command,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,
)


web_cloudflare_log_thread = threading.Thread(
    target=stream_web_cloudflare_logs,
    args=(
        web_cloudflare_tunnel_process,
        web_cloudflare_logs,
    ),
    name="vaultify-web-cloudflare-logs",
    daemon=True,
)


web_cloudflare_log_thread.start()


tunnel_deadline = (
    time.time()
    + WEB_TUNNEL_STARTUP_TIMEOUT
)


VAULTIFY_WEB_PUBLIC_URL = None


while time.time() < tunnel_deadline:
    if (
        web_cloudflare_tunnel_process.poll()
        is not None
    ):
        raise RuntimeError(
            "The Vaultify web tunnel "
            "stopped unexpectedly. "
            f"Exit code: "
            f"{web_cloudflare_tunnel_process.returncode}. "
            f"Recent logs: "
            f"{web_cloudflare_logs[-10:]}"
        )

    VAULTIFY_WEB_PUBLIC_URL = (
        find_trycloudflare_url(
            web_cloudflare_logs
        )
    )

    if (
        VAULTIFY_WEB_PUBLIC_URL
        and web_tunnel_is_registered(
            web_cloudflare_logs
        )
    ):
        break

    time.sleep(0.25)


if not VAULTIFY_WEB_PUBLIC_URL:
    raise RuntimeError(
        "Cloudflare did not provide "
        "a public web URL."
    )


if not web_tunnel_is_registered(
    web_cloudflare_logs
):
    raise RuntimeError(
        "The web tunnel URL was created, "
        "but its connection was not registered."
    )


print(
    "\n✅ Vaultify web tunnel started."
)
print(
    f"🌍 Public web URL: "
    f"{VAULTIFY_WEB_PUBLIC_URL}"
)


public_health_succeeded = False
last_public_health_error = None


for attempt_number in range(
    1,
    WEB_PUBLIC_TEST_ATTEMPTS + 1,
):
    try:
        public_health_response = requests.get(
            (
                VAULTIFY_WEB_PUBLIC_URL
                + "/health"
            ),
            timeout=15,
        )

        if (
            public_health_response.status_code
            == 200
            and public_health_response.json().get(
                "status"
            )
            == "ok"
        ):
            public_health_succeeded = True
            break

        last_public_health_error = (
            RuntimeError(
                "Unexpected public health response: "
                f"{public_health_response.status_code}"
            )
        )

    except Exception as error:
        last_public_health_error = error

    print(
        f"⏳ Public web health attempt "
        f"{attempt_number}/"
        f"{WEB_PUBLIC_TEST_ATTEMPTS}..."
    )

    if (
        attempt_number
        < WEB_PUBLIC_TEST_ATTEMPTS
    ):
        time.sleep(
            WEB_PUBLIC_TEST_RETRY_SECONDS
        )


if public_health_succeeded:
    print(
        "✅ Public Vaultify health check passed."
    )

else:
    print(
        "⚠️ Colab could not complete the public "
        "health check yet."
    )
    print(
        f"   Last error: "
        f"{last_public_health_error}"
    )
    print(
        "   The browser URL may still become "
        "reachable after a short DNS delay."
    )


print(
    "\n🚀 Vaultify frontend is live:"
)
print(
    f"   Home: "
    f"{VAULTIFY_WEB_PUBLIC_URL}"
)
print(
    f"   Register: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/register"
)
print(
    f"   Login: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/login"
)
print(
    f"   Dashboard: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/dashboard"
)
print(
    f"   Documents: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/documents"
)

print(
    "\n✅ The existing MCP server "
    "and MCP tunnel were preserved."
)
print(
    "⚠️ Do not share this temporary "
    "unauthenticated registration URL publicly."
)
print(
    "⚠️ For now, inspect the design and "
    "test registration/login only."
)
print(
    "⏳ Wait before uploading a large PDF; "
    "background ingestion comes next."
)

INFO:werkzeug:127.0.0.1 - - [10/Jul/2026 17:27:15] "GET /health HTTP/1.1" 200 -


✅ Vaultify web server started locally.
🌐 Local web URL: http://127.0.0.1:5000
🩺 Local health: {'phase': 3, 'service': 'Vaultify', 'status': 'ok'}

🌐 Web Cloudflare origin: http://127.0.0.1:5000
[web-cloudflared] 2026-07-10T17:27:15Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[web-cloudflared] 2026-07-10T17:27:15Z INF Requesting new quick Tunnel on trycloudflare.com...
[web-cloudflared] 2026-07-10T17:27:19Z INF +-----------------------------------------------

INFO:werkzeug:127.0.0.1 - - [10/Jul/2026 17:27:50] "GET /health HTTP/1.1" 200 -


✅ Public Vaultify health check passed.

🚀 Vaultify frontend is live:
   Home: https://interests-developers-assembly-teams.trycloudflare.com
   Register: https://interests-developers-assembly-teams.trycloudflare.com/register
   Login: https://interests-developers-assembly-teams.trycloudflare.com/login
   Dashboard: https://interests-developers-assembly-teams.trycloudflare.com/dashboard
   Documents: https://interests-developers-assembly-teams.trycloudflare.com/documents

✅ The existing MCP server and MCP tunnel were preserved.
⚠️ Do not share this temporary unauthenticated registration URL publicly.
⚠️ For now, inspect the design and test registration/login only.
⏳ Wait before uploading a large PDF; background ingestion comes next.
